# Module 06 · Spatial validation

Are senescent cells spatially clustered, and is that clustering anything more
than tissue architecture?

Everything upstream treated cells as exchangeable. Spatial transcriptomics adds
coordinates, which makes one new question askable: whether senescence forms
**foci** — neighbourhoods of elevated score — rather than being scattered at
random through the section.

**The confound that shapes the whole module.** Brain tissue is not
compositionally uniform. White matter is oligodendrocyte-dense, grey matter is
neuron-dense, and the two form large contiguous territories. Any score that
differs between cell types will therefore look spatially autocorrelated,
because the *cell types* are spatially autocorrelated. A positive Moran's I on
the raw senescence score is the expected result whether or not senescence
clusters for its own reasons.

Section 07 is the section that addresses this: regress the score on local
cell-type composition, then re-run Moran's I on the **residual**. Autocorrelation
that survives that is not explained by architecture. Everything after section 07
should be read against the residual result, not the raw one.

| Section | | |
|---|---|---|
| 01-02 | setup, samples, spatial graphs | |
| 03-06 | univariate Moran's I per sample, DL meta, forest, scatter | raw score |
| 07-08 | composition regression, residual Moran's I | **the control** |
| 09-10 | Getis-Ord Gi\* hot vs cold spots | local |
| 11 | LISA — local Moran's I, HH vs NS | local |
| 12-14 | summary bars, pathways in hot spots, SnC-spot composition | |
| 15-17 | proportion · burden · susceptibility in space | ties to module 07 |
| 18 | bivariate Moran's I — cross-variable spatial coupling | |

**Cohorts.** Harari (`spatial_an1792`) NND controls and Morabito
(`morabito_dsad`) controls, pooled by random-effects meta-analysis rather than
concatenated, for the same reason as module 08 — the platforms and handling
differ.

**A limit worth stating before the results.** Spot-level spatial data does not
resolve microglia at usable numbers in these sections. Any claim specific to
microglial senescence cannot be tested here, whatever the pooled Moran's I
says.

---
## 01 · Config and setup

**Why.** Paths from the environment, sample registry, palettes, and the control
filter. `_env()` is the only line not in the source notebooks.

**Set before running:** the sample registry and which samples count as controls.

In [ ]:
# -----------------------------------------------------------------------------
# Path resolution - the only addition to the source notebooks
# -----------------------------------------------------------------------------
#   SENESCENCE_DATA : analysis root
#   SENESCENCE_REF  : reference root (published panels, read-only)
# See .env.example.
import os
from pathlib import Path


def _env(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(
            f"{name} is not set. Copy .env.example to .env, edit the paths, "
            f"and source it before starting the kernel.")
    return v.rstrip("/")


SEN_DATA        = _env("SENESCENCE_DATA")
SEN_REF         = _env("SENESCENCE_REF")
SEN_REF_MARKERS = SEN_REF + "/markers"

print(f"  SENESCENCE_DATA -> {SEN_DATA}")
print(f"  SENESCENCE_REF  -> {SEN_REF}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — CONFIG, PATHS, LOAD, OBS-PRESENCE CHECK
# ═══════════════════════════════════════════════════════════════════════════════
# Module : 04_validation   |   Dataset: morabito_dsad_control   |   Scope: controls only
# ═══════════════════════════════════════════════════════════════════════════════
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

import squidpy as sq
from scipy import sparse
from scipy.stats import norm, mannwhitneyu, fisher_exact, spearmanr
from scipy.sparse.csgraph import connected_components
from scipy.spatial import KDTree
from statsmodels.stats.multitest import multipletests
from esda.moran import Moran_BV
from libpysal.weights import W as LibW

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE    = Path(f'{SEN_DATA}/spatial/')
MODULE  = '04_validation'
DATASET = 'morabito_dsad_control'

FIGURES_DIR = BASE / MODULE / DATASET / 'figures'   # figures only
RESULTS_DIR = BASE / MODULE / DATASET / 'results'    # CSVs
DATA_DIR    = BASE / MODULE / DATASET / 'data'       # .h5ad etc
for d in (FIGURES_DIR, RESULTS_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

CHECKPOINT = Path(f'{SEN_DATA}/data/10_spatial/morabito_dsad_senepy_labeled.h5ad')

print("="*64); print("CELL 1 — CONFIG & LOAD"); print("="*64)
print(f"  Module : {MODULE}")
print(f"  Dataset: {DATASET}")
print(f"  Output : {BASE/MODULE/DATASET}  (figures/ results/ data/)")
print(f"  Input  : {CHECKPOINT}")

# ── Load ──────────────────────────────────────────────────────────────────────
adata = sc.read_h5ad(CHECKPOINT)
print(f"\n  Loaded: {adata.n_obs:,} spots × {adata.n_vars:,} genes")
print(f"  Layers: {list(adata.layers.keys())}")
print(f"  obsm  : {list(adata.obsm.keys())}")

# ── Scope ─────────────────────────────────────────────────────────────────────
sid_col   = 'sample_id' if 'sample_id' in adata.obs.columns else 'Sample'
group_col = next((c for c in ['group','Diagnosis','condition'] if c in adata.obs.columns), None)
print(f"\n  Sample column : '{sid_col}' → {adata.obs[sid_col].nunique()} samples")
if group_col:
    print(f"  Group column  : '{group_col}' → {dict(adata.obs[group_col].value_counts())}")

# ── Obs-presence check ────────────────────────────────────────────────────────
print(f"\n  {'─'*58}")
print("  OBS-PRESENCE CHECK")
print(f"  {'─'*58}")
needed = {
    'sample_id / Sample': sid_col, 'annotation (layers)': 'annotation',
    'sen_score': 'sen_score', 'is_senescent': 'is_senescent',
    'gi_star_z (Gi*)': 'gi_star_z', 'lisa_quad (LISA)': 'lisa_quad',
    'compartment': 'compartment',
}
print(f"  {'column':<22s} {'present':>8s} {'populated':>12s}   note")
print(f"  {'─'*58}")
for label, col in needed.items():
    present = col in adata.obs.columns
    if present:
        nn = adata.obs[col].notna().sum()
        pop, note = f"{nn:,}/{adata.n_obs:,}", ('' if nn == adata.n_obs else 'PARTIAL')
    else:
        pop = '—'; note = 'compute later' if col in ('gi_star_z','lisa_quad','compartment') else 'MISSING'
    print(f"  {label:<22s} {'✓' if present else '✗':>8s} {pop:>12s}   {note}")

score_cols = [c for c in adata.obs.columns if c.startswith('Score_')]
c2l_cols   = [c for c in adata.obs.columns if c.startswith('c2l_')]
print(f"\n  Score_* modules : {len(score_cols)}  {score_cols or '(none — pathways analysis blocked)'}")
print(f"  c2l_* deconv    : {len(c2l_cols)}  {[c.replace('c2l_','') for c in c2l_cols] or '(none — residual Moran blocked)'}")
if 'annotation' in adata.obs.columns:
    print(f"\n  annotation values:")
    print(adata.obs['annotation'].value_counts(dropna=False).to_string())

---
## 02 · Control filter and compartment assignment

**Why.** Two things happen here, and both matter for how every later section
reads.

**Controls only.** Spatial autocorrelation of senescence is being established
as a baseline property of tissue, not as a disease effect. Mixing cases in would
make a positive result ambiguous between the two.

**WM / GM compartment.** Assigning each spot to white or grey matter is what
makes the composition confound testable rather than merely acknowledged —
sections 09-11 report hot-versus-cold contrasts *within* compartment as well as
pooled, and the two can disagree.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — CONTROL FILTER + WM/GM COMPARTMENT LABEL + MARKER VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: Cell 1 (adata loaded). Scope = Morabito controls only.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 2 — CONTROL FILTER + WM/GM LABEL + VALIDATION"); print("="*64)

# ── BEAT 1: prep — filter to controls, display the cohort ────────────────────
ctrl = adata[adata.obs['group'] == 'Control'].copy()
print(f"\n  Filtered: {adata.n_obs:,} → {ctrl.n_obs:,} spots  (group == 'Control')")
print(f"  Control samples: {ctrl.obs['sample_id'].nunique()}")

samp_tbl = (ctrl.obs.groupby('sample_id', observed=True)
            .agg(n_spots=('sen_score','size'),
                 sex=('sex','first') if 'sex' in ctrl.obs else ('sample_id','first'),
                 age=('age','first') if 'age' in ctrl.obs else ('sample_id','first'))
            .sort_values('n_spots', ascending=False))
print(f"\n  Per-sample spot counts (controls):")
print(samp_tbl.to_string())

print(f"\n  annotation after control filter:")
print(ctrl.obs['annotation'].value_counts(dropna=False).to_string())

# ── BEAT 2: method ───────────────────────────────────────────────────────────
GM_LAYERS = ['L1','L2/3','L3/4','L3/4/5','L5/6','L6b']
WM_LAYERS = ['WM1','WM2','WM3']
WM_MARKERS = ['MBP','PLP1','MOBP']
OLIGO_COL  = 'c2l_Oligodendrocyte'

print(f"\n  {'─'*58}\n  METHOD\n  {'─'*58}")
print(f"  compartment := 'GM' if annotation ∈ {{L1..L6b}} else 'WM' if ∈ {{WM1..WM3}}")
print(f"  Validation: WM spots must show HIGHER myelin/oligo markers than GM,")
print(f"              in EVERY sample, else the layer label is untrustworthy.")
print(f"  Markers: {WM_MARKERS} (lognorm)  +  oligodendrocyte deconvolution ({OLIGO_COL})")

# ── BEAT 3: compute ──────────────────────────────────────────────────────────
def to_comp(s):
    if s in GM_LAYERS: return 'GM'
    if s in WM_LAYERS: return 'WM'
    return 'Unknown'
ctrl.obs['compartment'] = ctrl.obs['annotation'].map(to_comp).astype('category')

n_unknown = int((ctrl.obs['compartment']=='Unknown').sum())
markers_present = [g for g in WM_MARKERS if g in ctrl.var_names]
markers_missing = [g for g in WM_MARKERS if g not in ctrl.var_names]
print(f"\n  Unknown/unmapped spots: {n_unknown}")
print(f"  Overall: GM={int((ctrl.obs['compartment']=='GM').sum()):,}  "
      f"WM={int((ctrl.obs['compartment']=='WM').sum()):,}")
if markers_missing:
    print(f"  ⚠ markers not in var_names (symbols may be Ensembl IDs): {markers_missing}")

def mean_expr(ad, gene, mask):
    if gene not in ad.var_names: return np.nan
    X = ad.layers['lognorm'] if 'lognorm' in ad.layers else ad.X
    col = X[:, ad.var_names.get_loc(gene)]
    col = np.asarray(col.todense()).ravel() if sparse.issparse(col) else np.asarray(col).ravel()
    return col[mask].mean()

# ── BEAT 4: results table ────────────────────────────────────────────────────
print(f"\n  {'─'*58}")
print("  VALIDATION TABLE (WM should exceed GM on every metric, every sample)")
print(f"  {'─'*58}")
hdr = f"  {'Sample':<28s} {'Comp':<4s} {'nSpots':>7s}"
for g in markers_present: hdr += f" {g:>7s}"
hdr += f" {'Oligo':>7s}"
print(hdr); print("  " + "─"*(len(hdr)-2))

val_rows = []
for sid in sorted(ctrl.obs['sample_id'].unique()):
    m_samp = (ctrl.obs['sample_id']==sid).values          # was 'sm' — renamed to avoid clobbering statsmodels.api
    rec = {'sample_id': sid}
    for comp in ['GM','WM']:
        m = m_samp & (ctrl.obs['compartment']==comp).values
        if m.sum()==0: continue
        row = f"  {sid:<28s} {comp:<4s} {int(m.sum()):>7d}"
        for g in markers_present:
            v = mean_expr(ctrl, g, m); row += f" {v:>7.3f}"; rec[f'{g}_{comp}']=v
        oligo = ctrl.obs.loc[m, OLIGO_COL].mean() if OLIGO_COL in ctrl.obs else np.nan
        row += f" {oligo:>7.3f}"; rec[f'Oligo_{comp}']=oligo
        print(row)
    val_rows.append(rec)
    print()

df_validation = pd.DataFrame(val_rows)
df_validation.to_csv(RESULTS_DIR / 'wmgm_marker_validation.csv', index=False)
print(f"  ✓ saved: {RESULTS_DIR / 'wmgm_marker_validation.csv'}")
print(f"\n  GATE: WM > GM on MBP/PLP1/MOBP AND oligo in every sample.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2b — DISPLAY LABELS: F-/M-age.N
# ═══════════════════════════════════════════════════════════════════════════════
from collections import Counter
print("="*64); print("CELL 2b — DISPLAY LABELS (F-/M-age.N)"); print("="*64)

raw = {}
for sid in sorted(ctrl.obs['sample_id'].unique()):
    sub = ctrl.obs[ctrl.obs['sample_id']==sid]
    sex = str(sub['sex'].iloc[0]) if 'sex' in sub else '?'
    age = sub['age'].iloc[0]      if 'age' in sub else '?'
    raw[sid] = f"{sex}-{age}"

counts = Counter(raw.values()); seen={}; SAMPLE_LABELS={}
for sid in sorted(raw):
    lab=raw[sid]
    if counts[lab]>1:
        seen[lab]=seen.get(lab,0)+1; SAMPLE_LABELS[sid]=f"{lab}.{seen[lab]}"
    else:
        SAMPLE_LABELS[sid]=lab
ctrl.obs['display_name'] = ctrl.obs['sample_id'].map(SAMPLE_LABELS).astype('category')

print(f"\n  {'sample_id':<28s} {'label':<10s} {'nSpots':>7s}")
print("  "+"─"*48)
for sid in sorted(SAMPLE_LABELS):
    print(f"  {sid:<28s} {SAMPLE_LABELS[sid]:<10s} {int((ctrl.obs['sample_id']==sid).sum()):>7d}")
nu=len(set(SAMPLE_LABELS.values()))
print(f"\n  {len(SAMPLE_LABELS)} samples → {nu} unique labels {'✓' if nu==len(SAMPLE_LABELS) else '✗'}")

---
## 03 · Spatial graphs, Gi\*, and LISA

**Why.** Builds the neighbour graph each statistic is computed against, and
computes the local statistics once so sections 09-11 all read the same
neighbourhood definition. A Moran's I is only meaningful relative to a stated
graph; changing k changes the answer.

**Computes.** Spatial neighbour graph per sample · Getis-Ord Gi\* per spot ·
LISA (local Moran's I) per spot, with its quadrant label (HH / LL / HL / LH).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — PER-SAMPLE SPATIAL GRAPHS + Gi* + LISA
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 3 — SPATIAL GRAPHS + Gi* + LISA"); print("="*64)

K_NEIGH, HOT_Z, COLD_Z = 6, 1.96, -1.96
sids = sorted(ctrl.obs['sample_id'].unique())

sample_adatas = {sid: ctrl[ctrl.obs['sample_id']==sid].copy() for sid in sids}
print(f"\n  {len(sample_adatas)} per-sample objects; k={K_NEIGH}; Gi*/LISA on sen_score.")

def compute_gi_star(values, W):
    n=len(values); x=np.asarray(values,float); S=x.std()
    if S==0: return np.full(n,np.nan)
    Wd = W.toarray() if sparse.issparse(W) else np.array(W); np.fill_diagonal(Wd,1)
    Wi=Wd.sum(1); Wi2=(Wd**2).sum(1)
    denom=S*np.sqrt((n*Wi2-Wi**2)/(n-1)); denom[denom==0]=np.nan
    return (Wd@x - x.mean()*Wi)/denom

def compute_lisa(values, W, n_perms=999, seed=42):
    rng=np.random.default_rng(seed); x=np.asarray(values,float); n=len(x)
    z=x-x.mean(); var=(z@z)/n
    if var==0: return np.zeros(n), np.zeros(n), np.ones(n), np.full(n,'ns',object)
    if sparse.issparse(W):
        rs=np.array(W.sum(1)).ravel(); rs[rs==0]=1; Wn=W.multiply(1/rs[:,None]); Wz=np.array(Wn@z).ravel()
    else:
        rs=W.sum(1); rs[rs==0]=1; Wn=W/rs[:,None]; Wz=Wn@z
    Ii=z*Wz/var; cnt=np.zeros(n)
    for _ in range(n_perms):
        zp=rng.permutation(z); Wzp=np.array(Wn@zp).ravel() if sparse.issparse(Wn) else Wn@zp
        cnt += (np.abs(z*Wzp/var) >= np.abs(Ii))
    p=(cnt+1)/(n_perms+1); q=np.full(n,'ns',object); sig=p<0.05
    q[(z>0)&(Wz>0)&sig]='HH'; q[(z<0)&(Wz<0)&sig]='LL'
    q[(z>0)&(Wz<0)&sig]='HL'; q[(z<0)&(Wz>0)&sig]='LH'
    return Ii, Wz, p, q

rows=[]
for sid in sids:
    ad=sample_adatas[sid]
    sq.gr.spatial_neighbors(ad, n_neighs=K_NEIGH, coord_type='generic')
    W=ad.obsp['spatial_connectivities']
    s=ad.obs['sen_score'].astype(float).values
    ad.obs['gi_star_z']=compute_gi_star(s,W)
    Ii,Wz,p,q=compute_lisa(s,W)
    ad.obs['lisa_I'],ad.obs['lisa_Wz'],ad.obs['lisa_p'],ad.obs['lisa_quad']=Ii,Wz,p,q
    z=ad.obs['gi_star_z'].values
    rows.append({'label':SAMPLE_LABELS[sid],'n':ad.n_obs,'avg_k':float(np.array(W.sum(1)).ravel().mean()),
                 'n_hot':int((z>HOT_Z).sum()),'n_cold':int((z<COLD_Z).sum()),
                 'n_HH':int((q=='HH').sum()),'n_LL':int((q=='LL').sum())})

df_graph=pd.DataFrame(rows).sort_values('n',ascending=False)
print(f"\n  {'Sample':<8s} {'nSpot':>6s} {'avgK':>5s} {'hot':>5s} {'cold':>5s} {'HH':>5s} {'LL':>5s}")
print("  "+"─"*45)
for _,r in df_graph.iterrows():
    print(f"  {r['label']:<8s} {r['n']:>6d} {r['avg_k']:>5.1f} {r['n_hot']:>5d} {r['n_cold']:>5d} {r['n_HH']:>5d} {r['n_LL']:>5d}")
df_graph.to_csv(RESULTS_DIR/'graph_gistar_lisa_summary.csv',index=False)
print(f"\n  ✓ saved. avgK should be ≈6; hot/HH a small minority per section.")

---
## 04 · Univariate Moran's I — all control samples

**Why.** The global statistic, one value per variable per sample. Positive
means like values sit near like values.

**Test.** Moran's I with 999 permutations, so the null is the observed values
shuffled over the same graph — which holds the graph and the marginal
distribution fixed and varies only the spatial arrangement.

**Per sample, not pooled.** Concatenating sections would create artificial
neighbour relationships across tissue boundaries. Each section is its own
graph; pooling happens at the estimate level in section 05.

In [ ]:
#\!/usr/bin/env python3
# =============================================================================
# MORAN'S I — ALL CONTROL SAMPLES + CROSS-DATASET META-ANALYSIS
# =============================================================================
# Env: omicverse (needs: scanpy, squidpy, numpy, pandas, scipy, matplotlib)
#
# Loads the senepy-labeled checkpoints from BOTH datasets, subsets to ALL
# control samples, runs per-sample Moran's I with 999 permutations, then
# performs a random-effects meta-analysis (DerSimonian–Laird) across all
# control samples, with Fisher's combined p-values as a secondary check.
#
# Control samples:
#   Harari  (spatial_an1792):  4 NND controls  (NND-2, NND-3, NND-5, NND-6)
#   Morabito (morabito_dsad): 15 Control samples (post-QC)
#   Total: 19 control samples
# =============================================================================

import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse as sp_sparse
from scipy.stats import norm, combine_pvalues
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR    = Path(SEN_DATA)
DATA_DIR    = BASE_DIR / 'data' / '10_spatial'
RESULTS_DIR = BASE_DIR / 'results' / '10_spatial' / 'meta_controls'
FIGURES_DIR = BASE_DIR / 'figures' / '10_spatial' / 'meta_controls'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint files (senepy-labeled, have is_senescent + sen_score + module scores)
HARARI_CHECKPOINT   = DATA_DIR / 'spatial_an1792_senepy_labeled.h5ad'
MORABITO_CHECKPOINT = DATA_DIR / 'morabito_dsad_senepy_labeled.h5ad'

# Score columns to test
SCORE_COLS = [
    'is_senescent',
    'sen_score',
    'Score_p53_Targets',
    'Score_CellCycleArrest',
    'Score_SASP',
    'Score_AntiApoptosis',
    'Score_DDR',
    'Score_CellSurfaceMarkers',
    'Score_LysosomalContent',
    'Score_SD_TMC',
    'Score_SenMayo',
    'Score_Fridman_Up',
]

N_PERMS = 999   # permutations for Moran's I
K_NEIGH = 6     # Visium hexagonal grid neighbors

print("=" * 70)
print("MORAN'S I — ALL CONTROL SAMPLES + META-ANALYSIS")
print("=" * 70)

In [ ]:
print("\n── Loading datasets ──\n")

# --- Harari ---
print(f"  Loading Harari: {HARARI_CHECKPOINT}")
adata_harari = sc.read_h5ad(HARARI_CHECKPOINT)
print(f"    {adata_harari.n_obs:,} spots × {adata_harari.n_vars:,} genes")

# Harari controls are all NND (group == 'Control' after relabeling in notebook)
harari_controls = sorted(adata_harari.obs['gsm_id'].unique())
print(f"    Control samples: {len(harari_controls)}")

# Harari sample metadata
HARARI_META = {
    'GSM8184318': {'name': 'NND-1', 'age': 82, 'sex': 'F'},
    'GSM8184319': {'name': 'NND-2', 'age': 75, 'sex': 'M'},
    'GSM8184320': {'name': 'NND-3', 'age': 63, 'sex': 'M'},
    'GSM8184321': {'name': 'NND-4', 'age': 79, 'sex': 'M'},
    'GSM8184322': {'name': 'NND-5', 'age': 68, 'sex': 'M'},
    'GSM8184323': {'name': 'NND-6', 'age': 82, 'sex': 'F'},
}
for gsm in harari_controls:
    meta = HARARI_META.get(gsm, {})
    mask = adata_harari.obs['gsm_id'] == gsm
    n = mask.sum()
    n_snc = (adata_harari.obs.loc[mask, 'is_senescent'] == 1).sum()
    print(f"      {gsm} ({meta.get('name','?')}): {n:,} spots, "
          f"SnC={n_snc} ({n_snc/n*100:.1f}%)")

# --- Morabito ---
print(f"\n  Loading Morabito: {MORABITO_CHECKPOINT}")
adata_morabito = sc.read_h5ad(MORABITO_CHECKPOINT)
print(f"    {adata_morabito.n_obs:,} spots × {adata_morabito.n_vars:,} genes")

# Filter to Control group only
control_mask = adata_morabito.obs['group'] == 'Control'
adata_morabito_ctrl = adata_morabito[control_mask].copy()
morabito_controls = sorted(adata_morabito_ctrl.obs['sample_id'].unique())
print(f"    Control samples: {len(morabito_controls)}")

for sid in morabito_controls:
    mask = adata_morabito_ctrl.obs['sample_id'] == sid
    n = mask.sum()
    n_snc = (adata_morabito_ctrl.obs.loc[mask, 'is_senescent'] == 1).sum()
    sex = adata_morabito_ctrl.obs.loc[mask, 'sex'].iloc[0]
    age = adata_morabito_ctrl.obs.loc[mask, 'age'].iloc[0]
    print(f"      {sid}: {n:,} spots, SnC={n_snc} ({n_snc/n*100:.1f}%), "
          f"{sex}, age={age}")

total_controls = len(harari_controls) + len(morabito_controls)
print(f"\n  Total control samples: {total_controls}")

# Free full Morabito object
del adata_morabito

In [ ]:
print("\n" + "=" * 70)
print("BUILDING SPATIAL GRAPHS & RUNNING MORAN'S I")
print("=" * 70)

all_morans = []

def run_morans_for_sample(ad, sample_name, dataset, sample_id):
    """Build spatial graph and compute Moran's I for one sample."""
    
    # Build k=6 spatial neighbor graph
    sq.gr.spatial_neighbors(ad, n_neighs=K_NEIGH, coord_type='generic')
    
    n_spots = ad.n_obs
    n_snc = (ad.obs['is_senescent'] == 1).sum()
    
    # Prepare numeric senescence column
    ad.obs['is_senescent_num'] = ad.obs['is_senescent'].astype(float)
    
    # Identify available score columns
    test_cols = ['is_senescent_num'] + [c for c in SCORE_COLS[1:] if c in ad.obs.columns]
    
    # Run squidpy Moran's I
    sq.gr.spatial_autocorr(ad, mode='moran', genes=test_cols, attr='obs', n_perms=N_PERMS)
    
    print(f"\n  {sample_name} ({dataset}, n={n_spots}, SnC={n_snc}):")
    print(f"  {'Variable':<26s} {'I':>8s} {'p':>8s} {'':>4s}")
    print(f"  {'─'*50}")
    
    results = []
    for col in test_cols:
        display_name = col.replace('is_senescent_num', 'SnC (binary)')
        display_name = display_name.replace('sen_score', 'Senescence Score')
        display_name = display_name.replace('Score_', '')
        
        I_val = ad.uns['moranI'].loc[col, 'I']
        pval  = ad.uns['moranI'].loc[col, 'pval_sim']
        
        # Variance of Moran's I under permutation null
        # Use expected I and simulated variance for meta-analysis
        E_I = -1.0 / (n_spots - 1)
        
        # Estimate SE from z-score: z = (I - E(I)) / SE
        # With permutation p-value, we can back-calculate z
        # But more robust: use the analytic variance from squidpy if available
        # Otherwise estimate from the permutation null
        if 'pval_sim_fdr' in ad.uns['moranI'].columns:
            # Use z-score approach: z = norm.isf(pval/2) * sign(I - E_I)
            pass
        
        # For meta-analysis, compute z-score from p-value
        if pval < 1e-10:
            pval_adj = 1e-10  # floor to avoid infinity
        else:
            pval_adj = pval
        
        if pval_adj >= 1.0:
            z_score = 0.0
        else:
            z_score = norm.isf(pval_adj / 2)  # two-sided
            if I_val < E_I:
                z_score = -z_score
        
        se_I = (I_val - E_I) / z_score if abs(z_score) > 0 else np.nan
        
        sig = ''
        if pval < 0.001: sig = '***'
        elif pval < 0.01: sig = '**'
        elif pval < 0.05: sig = '*'
        
        interp = 'CLUSTERED' if (I_val > 0 and pval < 0.05) else (
            'dispersed' if (I_val < 0 and pval < 0.05) else 'random')
        
        print(f"  {display_name:<26s} {I_val:>+8.4f} {pval:>8.4f} {sig:>4s}")
        
        results.append({
            'dataset': dataset,
            'sample_id': sample_id,
            'sample_name': sample_name,
            'n_spots': n_spots,
            'n_snc': n_snc,
            'pct_snc': n_snc / n_spots * 100,
            'variable': display_name,
            'variable_raw': col,
            'morans_I': I_val,
            'E_I': E_I,
            'SE_I': se_I,
            'z_score': z_score,
            'p_value': pval,
            'significant': sig,
            'interpretation': interp,
        })
    
    return results

# --- Process Harari controls ---
print("\n── Harari Controls ──")
for gsm in harari_controls:
    mask = (adata_harari.obs['gsm_id'] == gsm).values
    ad = adata_harari[mask].copy()
    
    # Ensure spatial coords are clean
    valid = ~np.isnan(ad.obsm['spatial']).any(axis=1)
    if valid.sum() < ad.n_obs:
        ad = ad[valid].copy()
    
    meta = HARARI_META.get(gsm, {'name': gsm})
    results = run_morans_for_sample(ad, meta['name'], 'Harari', gsm)
    all_morans.extend(results)
    del ad

# --- Process Morabito controls ---
print("\n── Morabito Controls ──")
for sid in morabito_controls:
    mask = (adata_morabito_ctrl.obs['sample_id'] == sid).values
    ad = adata_morabito_ctrl[mask].copy()
    
    valid = ~np.isnan(ad.obsm['spatial']).any(axis=1)
    if valid.sum() < ad.n_obs:
        ad = ad[valid].copy()
    
    sex = ad.obs['sex'].iloc[0]
    age = int(ad.obs['age'].iloc[0])
    short_name = f"{sex}-{age}"
    # Disambiguate duplicate sex-age combos
    existing = [r['sample_name'] for r in all_morans if r['dataset'] == 'Morabito']
    if short_name in existing:
        count = sum(1 for n in existing if n.startswith(short_name))
        short_name = f"{short_name}.{count+1}"
    
    results = run_morans_for_sample(ad, short_name, 'Morabito', sid)
    all_morans.extend(results)
    del ad

df_all = pd.DataFrame(all_morans)
df_all.to_csv(RESULTS_DIR / 'morans_I_all_controls.csv', index=False)
print(f"\n✓ Per-sample results saved: {RESULTS_DIR / 'morans_I_all_controls.csv'}")
print(f"  {len(df_all)} rows ({df_all['sample_name'].nunique()} samples × "
      f"{df_all['variable'].nunique()} variables)")

---
## 05 · Random-effects meta-analysis

**Why.** Same reasoning as module 08 — sections differ in platform, handling
and donor, so a fixed-effect pool would understate the uncertainty.

**Test.** DerSimonian-Laird across all control samples, with Fisher's combined
p as a secondary check. Then stratified by dataset, which is the replication
question: does each cohort show it on its own?

**Read the stratified result before the pooled one.** A pooled estimate driven
by one cohort is not replication.

In [ ]:
print("\n" + "=" * 70)
print("META-ANALYSIS: RANDOM-EFFECTS (DerSimonian–Laird)")
print("=" * 70)
print()
print("Method: For each variable, pool Moran's I across all control samples")
print("  1) DerSimonian–Laird random-effects model (accounts for heterogeneity)")
print("  2) Fisher's combined p-value (secondary confirmation)")
print("  3) Stouffer's weighted z-method (tertiary)")
print()

def dersimonian_laird(effects, variances):
    """
    DerSimonian–Laird random-effects meta-analysis.
    
    Parameters
    ----------
    effects : array-like
        Per-study effect sizes (Moran's I values)
    variances : array-like
        Per-study variance estimates (SE^2)
    
    Returns
    -------
    dict with pooled_effect, pooled_se, z, p_value, tau2, I2, Q, Q_p
    """
    effects = np.asarray(effects, dtype=float)
    variances = np.asarray(variances, dtype=float)
    
    # Drop any NaN
    valid = ~(np.isnan(effects) | np.isnan(variances) | (variances <= 0))
    effects = effects[valid]
    variances = variances[valid]
    k = len(effects)
    
    if k < 2:
        return {
            'pooled_I': effects[0] if k == 1 else np.nan,
            'pooled_SE': np.sqrt(variances[0]) if k == 1 else np.nan,
            'z': np.nan, 'p_value': np.nan,
            'tau2': 0.0, 'I2': 0.0, 'Q': 0.0, 'Q_p': np.nan, 'k': k,
        }
    
    # Fixed-effect weights
    w = 1.0 / variances
    
    # Cochran's Q
    theta_fe = np.sum(w * effects) / np.sum(w)
    Q = np.sum(w * (effects - theta_fe) ** 2)
    Q_df = k - 1
    Q_p = 1.0 - norm.cdf(Q, loc=Q_df, scale=np.sqrt(2 * Q_df)) if Q_df > 0 else 1.0
    # More accurate: use chi2
    from scipy.stats import chi2
    Q_p = 1.0 - chi2.cdf(Q, df=Q_df)
    
    # Between-study variance (tau²)
    C = np.sum(w) - np.sum(w ** 2) / np.sum(w)
    tau2 = max(0, (Q - Q_df) / C)
    
    # I² statistic
    I2 = max(0, (Q - Q_df) / Q * 100) if Q > 0 else 0.0
    
    # Random-effects weights
    w_re = 1.0 / (variances + tau2)
    
    # Pooled estimate
    pooled = np.sum(w_re * effects) / np.sum(w_re)
    pooled_var = 1.0 / np.sum(w_re)
    pooled_se = np.sqrt(pooled_var)
    
    z = pooled / pooled_se
    p = 2 * norm.sf(abs(z))
    
    return {
        'pooled_I': pooled,
        'pooled_SE': pooled_se,
        'z': z,
        'p_value': p,
        'tau2': tau2,
        'I2': I2,
        'Q': Q,
        'Q_p': Q_p,
        'k': k,
    }

meta_results = []
variables = df_all['variable'].unique()

print(f"{'Variable':<26s} {'k':>3s} {'Pooled I':>10s} {'SE':>8s} {'z':>8s} "
      f"{'p (RE)':>10s} {'p (Fisher)':>10s} {'I²':>6s} {'τ²':>8s}")
print("─" * 100)

for var in variables:
    sub = df_all[df_all['variable'] == var].copy()
    
    effects = sub['morans_I'].values
    se_vals = sub['SE_I'].values
    variances = se_vals ** 2
    pvals = sub['p_value'].values
    
    # DerSimonian–Laird
    dl = dersimonian_laird(effects, variances)
    
    # Fisher's method
    # Floor p-values at 1/(N_PERMS+1) for permutation tests
    pvals_floored = np.clip(pvals, 1.0 / (N_PERMS + 1), 1.0)
    fisher_stat, fisher_p = combine_pvalues(pvals_floored, method='fisher')
    
    # Stouffer's weighted z (weight by sqrt(n_spots))
    n_spots = sub['n_spots'].values
    z_scores = sub['z_score'].values
    weights = np.sqrt(n_spots)
    valid_z = ~np.isnan(z_scores)
    if valid_z.sum() > 0:
        stouffer_z = np.sum(weights[valid_z] * z_scores[valid_z]) / np.sqrt(np.sum(weights[valid_z] ** 2))
        stouffer_p = 2 * norm.sf(abs(stouffer_z))
    else:
        stouffer_z, stouffer_p = np.nan, np.nan
    
    sig = ''
    if dl['p_value'] < 0.001: sig = '***'
    elif dl['p_value'] < 0.01: sig = '**'
    elif dl['p_value'] < 0.05: sig = '*'
    
    print(f"{var:<26s} {dl['k']:>3d} {dl['pooled_I']:>+10.4f} {dl['pooled_SE']:>8.4f} "
          f"{dl['z']:>8.2f} {dl['p_value']:>10.2e} {fisher_p:>10.2e} "
          f"{dl['I2']:>5.1f}% {dl['tau2']:>8.5f} {sig}")
    
    meta_results.append({
        'variable': var,
        'k_studies': dl['k'],
        'pooled_I': dl['pooled_I'],
        'pooled_SE': dl['pooled_SE'],
        'z_RE': dl['z'],
        'p_RE': dl['p_value'],
        'tau2': dl['tau2'],
        'I2_pct': dl['I2'],
        'Q': dl['Q'],
        'Q_p': dl['Q_p'],
        'fisher_stat': fisher_stat,
        'fisher_p': fisher_p,
        'stouffer_z': stouffer_z,
        'stouffer_p': stouffer_p,
        'mean_I': effects.mean(),
        'median_I': np.median(effects),
        'sd_I': effects.std(),
        'min_I': effects.min(),
        'max_I': effects.max(),
        'n_sig_samples': (pvals < 0.05).sum(),
        'pct_sig_samples': (pvals < 0.05).sum() / len(pvals) * 100,
    })

df_meta = pd.DataFrame(meta_results)
df_meta.to_csv(RESULTS_DIR / 'meta_analysis_morans_I.csv', index=False)
print(f"\n✓ Meta-analysis saved: {RESULTS_DIR / 'meta_analysis_morans_I.csv'}")

In [ ]:
print("\n" + "=" * 70)
print("DATASET-STRATIFIED META-ANALYSIS")
print("=" * 70)

for dataset in ['Harari', 'Morabito']:
    sub_ds = df_all[df_all['dataset'] == dataset]
    n_samples = sub_ds['sample_name'].nunique()
    print(f"\n── {dataset} ({n_samples} controls) ──")
    print(f"{'Variable':<26s} {'k':>3s} {'Pooled I':>10s} {'p (RE)':>10s} {'I²':>6s}")
    print("─" * 65)
    
    for var in variables:
        sub = sub_ds[sub_ds['variable'] == var]
        if len(sub) < 2:
            continue
        
        effects = sub['morans_I'].values
        variances = sub['SE_I'].values ** 2
        dl = dersimonian_laird(effects, variances)
        
        sig = ''
        if dl['p_value'] < 0.001: sig = '***'
        elif dl['p_value'] < 0.01: sig = '**'
        elif dl['p_value'] < 0.05: sig = '*'
        
        print(f"{var:<26s} {dl['k']:>3d} {dl['pooled_I']:>+10.4f} "
              f"{dl['p_value']:>10.2e} {dl['I2']:>5.1f}% {sig}")

---
## 06 · Forest, heatmap, summary, and Moran scatter

**Why.** Four views of the same numbers. The forest shows per-sample intervals
against the pooled estimate; the heatmap shows all samples × all variables at
once, which is where a variable that behaves inconsistently becomes visible;
the summary table is what gets reported.

**Moran scatter.** Value against spatially-lagged value, per sample. The slope
*is* Moran's I — so the scatter shows whether the statistic reflects a general
trend or is being carried by a handful of extreme spots.

In [ ]:
print("\n" + "=" * 70)
print("GENERATING FOREST PLOTS (TABLE STYLE)")
print("=" * 70)

from matplotlib.patches import Polygon
from matplotlib.lines import Line2D

print("\nVariables in df_all:", sorted(df_all['variable'].unique()))
print("Variables in df_meta:", sorted(df_meta['variable'].unique()))

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 7,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'pdf.fonttype': 42,
})

# ── Config ────────────────────────────────────────────────────────────────
FONT_HEADER = 9
FONT_CELL   = 8
FONT_DATA   = 7
FONT_SMALL  = 6.5

DS_COLORS = {'Harari': '#1f77b4', 'Morabito': '#ff7f0e'}

# Column positions (axes fraction, fig width = 11 inches)
x_pos = {
    'sample':       0.02,
    'ds_tag':       0.155,
    'forest_left':  0.18,
    'forest_right': 0.52,
    'i_val':        0.57,
    'ci':           0.68,
    'pval':         0.82,
    'stars':        0.92,
}

# ── Variable selection ────────────────────────────────────────────────────
key_vars_wanted = ['Senescence Score', 'SnC (binary)', 'SASP', 'SenMayo',
                   'Fridman_Up', 'p53_Targets', 'LysosomalContent', 'DDR']

available = set(df_all['variable'].unique()) & set(df_meta['variable'].unique())
key_vars = [v for v in key_vars_wanted if v in available]

missing = [v for v in key_vars_wanted if v not in available]
if missing:
    print(f"\n⚠ Missing: {missing}")
    print(f"  Available: {sorted(available)}")
    for v in sorted(available):
        if v not in key_vars:
            key_vars.append(v)
    print(f"  Using: {key_vars}")

n_vars = len(key_vars)

if n_vars == 0:
    print("ERROR: No matching variables found.")
else:
    # ── Compute figure height from actual sample count ────────────────
    # Count max samples in any variable
    max_samples = max(
        len(df_all[df_all['variable'] == var]) for var in key_vars
    )
    # Each row needs ~0.2 inches for readable text at 7-8pt
    INCHES_PER_ROW = 0.22
    # Each panel: samples + dataset gap + pooled row + header + x-axis + padding
    n_datasets = df_all['dataset'].nunique()
    panel_height = (max_samples + n_datasets + 3) * INCHES_PER_ROW + 0.8

    fig_width = 11
    fig_height = panel_height * n_vars + 0.5  # 0.5 for suptitle + legend

    print(f"\n  {max_samples} samples, {n_vars} variables")
    print(f"  Panel height: {panel_height:.1f} in, Total: {fig_width}×{fig_height:.1f} in")

    fig, axes = plt.subplots(n_vars, 1, figsize=(fig_width, fig_height),
                             squeeze=False)

    for ax_idx, var in enumerate(key_vars):
        ax = axes[ax_idx, 0]

        sub = df_all[df_all['variable'] == var].sort_values(['dataset', 'sample_name'])
        meta_match = df_meta[df_meta['variable'] == var]

        if len(meta_match) == 0 or len(sub) == 0:
            ax.text(0.5, 0.5, f'"{var}" — no data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=9, color='#AAA')
            ax.axis('off')
            continue

        meta_row = meta_match.iloc[0]
        n_studies = len(sub)

        # ── Y-positions: integer spacing, gaps between groups ─────────
        # Use simple integer y so 1 unit = 1 row
        y_positions = []
        current_y = 0
        prev_dataset = None

        for _, row in sub.iterrows():
            if prev_dataset is not None and row['dataset'] != prev_dataset:
                current_y += 0.6  # dataset gap (fraction of a row)
            y_positions.append(current_y)
            current_y += 1.0     # one full row per sample
            prev_dataset = row['dataset']

        y_positions = np.array(y_positions)
        pooled_y = y_positions[-1] + 1.5  # gap before pooled

        y_first = y_positions[0]
        y_headers = pooled_y + 1.2
        y_header_line = pooled_y + 0.85

        forest_mid = (x_pos['forest_left'] + x_pos['forest_right']) / 2

        # ── Forest scaling ────────────────────────────────────────────
        all_vals = []
        for _, row in sub.iterrows():
            se = row['SE_I'] if not np.isnan(row['SE_I']) else 0
            all_vals.extend([row['morans_I'] - 1.96 * se,
                             row['morans_I'] + 1.96 * se])

        pooled_I  = meta_row['pooled_I']
        pooled_se = meta_row['pooled_SE']
        all_vals.extend([pooled_I - 1.96 * pooled_se,
                         pooled_I + 1.96 * pooled_se])

        val_min, val_max = min(all_vals), max(all_vals)
        val_range = val_max - val_min
        pad = max(val_range * 0.3, 0.02)
        axis_min, axis_max = val_min - pad, val_max + pad

        def val_to_x(val):
            norm = (np.clip(val, axis_min, axis_max) - axis_min) / (axis_max - axis_min)
            return x_pos['forest_left'] + norm * (x_pos['forest_right'] - x_pos['forest_left'])

        # ── Weight-proportional sizes ─────────────────────────────────
        ses = sub['SE_I'].values.copy()
        ses[np.isnan(ses) | (ses <= 0)] = np.nanmedian(ses[ses > 0])
        weights = 1.0 / (ses ** 2)
        w_min, w_max = weights.min(), weights.max()
        if w_max > w_min:
            diamond_widths = 0.005 + (weights - w_min) / (w_max - w_min) * 0.010
        else:
            diamond_widths = np.full(len(weights), 0.007)

        # ── Column headers ────────────────────────────────────────────
        ax.text(x_pos['sample'], y_headers, 'Sample',
                fontsize=FONT_HEADER, fontweight='bold', ha='left', va='bottom')
        ax.text(forest_mid, y_headers, "Moran's I (95% CI)",
                fontsize=FONT_HEADER, fontweight='bold', ha='center', va='bottom')
        ax.text(x_pos['i_val'], y_headers, 'I',
                fontsize=FONT_HEADER, fontweight='bold', ha='center', va='bottom')
        ax.text(x_pos['ci'], y_headers, '[95% CI]',
                fontsize=FONT_HEADER, fontweight='bold', ha='center', va='bottom')
        ax.text(x_pos['pval'], y_headers, 'p-value',
                fontsize=FONT_HEADER, fontweight='bold', ha='center', va='bottom')

        ax.plot([0.0, 0.99], [y_header_line, y_header_line],
                'k-', lw=0.8, clip_on=False)

        # ── Null line (I = 0) ─────────────────────────────────────────
        x_zero = val_to_x(0)
        if x_pos['forest_left'] <= x_zero <= x_pos['forest_right']:
            ax.axvline(x_zero, color='#888', linestyle='--', lw=0.6,
                       ymin=0.02, ymax=0.90, alpha=0.4)

        # ── Per-sample rows ───────────────────────────────────────────
        for i, (_, row) in enumerate(sub.iterrows()):
            y  = y_positions[i]
            se = row['SE_I'] if not np.isnan(row['SE_I']) else 0
            ci_lo = row['morans_I'] - 1.96 * se
            ci_hi = row['morans_I'] + 1.96 * se
            color = DS_COLORS.get(row['dataset'], '#888')
            p = row['p_value']
            is_sig = p < 0.05

            # Row shading
            if i % 2 == 0:
                ax.axhspan(y - 0.42, y + 0.42, color='#f7f7f7', zorder=0)

            text_color = '#222' if is_sig else '#999'
            fw = 'bold' if is_sig else 'normal'

            # Sample name
            ax.text(x_pos['sample'], y, row['sample_name'],
                    fontsize=FONT_CELL, ha='left', va='center',
                    color=text_color, fontweight=fw)

            # Dataset tag
            ds_tag = '(H)' if row['dataset'] == 'Harari' else '(M)'
            ax.text(x_pos['ds_tag'], y, ds_tag,
                    fontsize=FONT_SMALL, ha='right', va='center',
                    color=color, fontweight='bold')

            # Forest: CI line
            x_l = val_to_x(ci_lo)
            x_h = val_to_x(ci_hi)
            x_b = val_to_x(row['morans_I'])

            ax.plot([x_l, x_h], [y, y], color=color,
                    lw=1.8, alpha=0.7 if is_sig else 0.25,
                    solid_capstyle='round', zorder=2)

            # Forest: diamond
            dw = diamond_widths[i]
            dh = 0.22
            ax.add_patch(Polygon(
                [[x_b - dw, y], [x_b, y - dh], [x_b + dw, y], [x_b, y + dh]],
                fc=color, ec='black', lw=0.4, zorder=5,
                alpha=0.9 if is_sig else 0.3))

            # I value
            ax.text(x_pos['i_val'], y, f"{row['morans_I']:+.4f}",
                    fontsize=FONT_DATA, ha='center', va='center',
                    fontfamily='monospace', color=text_color, fontweight=fw)

            # [CI]
            ax.text(x_pos['ci'], y, f"[{ci_lo:+.3f}, {ci_hi:+.3f}]",
                    fontsize=FONT_DATA, ha='center', va='center',
                    fontfamily='monospace', color=text_color)

            # p-value
            p_txt = f"{p:.1e}" if p < 0.001 else f"{p:.3f}"
            ax.text(x_pos['pval'], y, p_txt,
                    fontsize=FONT_DATA, ha='center', va='center',
                    fontweight='bold' if is_sig else 'normal',
                    color='#222' if is_sig else '#AAA')

            # Stars
            stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
            ax.text(x_pos['stars'], y, stars,
                    fontsize=FONT_DATA + 1, ha='center', va='center',
                    fontweight='bold', color=color if is_sig else '#DDD')

        # ── Separator before pooled ───────────────────────────────────
        sep_y = (y_positions[-1] + pooled_y) / 2
        ax.plot([0.0, 0.99], [sep_y, sep_y], '-',
                color='#BBB', lw=0.5, clip_on=False)

        # ── Pooled diamond ────────────────────────────────────────────
        ci_lo_p = pooled_I - 1.96 * pooled_se
        ci_hi_p = pooled_I + 1.96 * pooled_se

        x_l_p = val_to_x(ci_lo_p)
        x_h_p = val_to_x(ci_hi_p)
        x_b_p = val_to_x(pooled_I)

        dh_p = 0.30
        diamond_x = [x_l_p, x_b_p, x_h_p, x_b_p]
        diamond_y = [pooled_y, pooled_y + dh_p, pooled_y, pooled_y - dh_p]
        ax.add_patch(Polygon(
            list(zip(diamond_x, diamond_y)),
            fc='#d62728', ec='black', lw=0.6, zorder=5, alpha=0.80))

        ax.text(x_pos['sample'], pooled_y,
                f"► Pooled (k={int(meta_row['k_studies'])})",
                fontsize=FONT_CELL, ha='left', va='center',
                fontweight='bold', color='#d62728')

        ax.text(x_pos['i_val'], pooled_y, f"{pooled_I:+.4f}",
                fontsize=FONT_DATA, ha='center', va='center',
                fontfamily='monospace', fontweight='bold', color='#d62728')

        ax.text(x_pos['ci'], pooled_y, f"[{ci_lo_p:+.4f}, {ci_hi_p:+.4f}]",
                fontsize=FONT_DATA, ha='center', va='center',
                fontfamily='monospace', fontweight='bold', color='#d62728')

        p_re = meta_row['p_RE']
        p_str_p = '<1e-10' if p_re < 1e-10 else (f"{p_re:.1e}" if p_re < 0.001 else f"{p_re:.3f}")
        ax.text(x_pos['pval'], pooled_y, p_str_p,
                fontsize=FONT_DATA, ha='center', va='center',
                fontweight='bold', color='#d62728')

        p_stars = '***' if p_re < 0.001 else ('**' if p_re < 0.01 else ('*' if p_re < 0.05 else ''))
        ax.text(x_pos['stars'], pooled_y, p_stars,
                fontsize=FONT_DATA + 1, ha='center', va='center',
                fontweight='bold', color='#d62728')

        # ── X-axis ticks ──────────────────────────────────────────────
        y_axis = y_first - 1.2
        ax.plot([x_pos['forest_left'], x_pos['forest_right']],
                [y_axis, y_axis], 'k-', lw=0.6)

        step = (axis_max - axis_min) / 4
        mag = 10 ** np.floor(np.log10(max(abs(step), 1e-10)))
        nice_step = np.ceil(step / mag) * mag
        ticks = []
        t = np.floor(axis_min / nice_step) * nice_step
        while t <= axis_max + nice_step * 0.1:
            if axis_min <= t <= axis_max:
                ticks.append(t)
            t += nice_step

        if axis_min <= 0 <= axis_max and not any(abs(t) < 1e-9 for t in ticks):
            ticks.append(0)
            ticks.sort()

        for t in ticks:
            xt = val_to_x(t)
            ax.plot([xt, xt], [y_axis, y_axis - 0.12], 'k-', lw=0.5)
            label = '0' if abs(t) < 1e-9 else f'{t:.2f}'
            ax.text(xt, y_axis - 0.18, label,
                    fontsize=FONT_SMALL, ha='center', va='top')

        ax.text(forest_mid, y_axis - 0.55, "Moran's I",
                fontsize=FONT_DATA, ha='center', va='top', color='#555')

        # ── Title ─────────────────────────────────────────────────────
        i2_str = f"I²={meta_row['I2_pct']:.0f}%"
        p_title = f"p={meta_row['p_RE']:.1e}" if meta_row['p_RE'] >= 1e-10 else "p<1e-10"
        sig_mark = '***' if meta_row['p_RE'] < 0.001 else ('**' if meta_row['p_RE'] < 0.01 else ('*' if meta_row['p_RE'] < 0.05 else ''))

        ax.text(0.01, y_headers + 0.50,
                f'{var}  ({p_title} {sig_mark},  {i2_str})',
                fontsize=9, fontweight='bold', ha='left', va='bottom', color='#222')

        # ── Finalize axes ─────────────────────────────────────────────
        ax.set_xlim(-0.01, 1.01)
        ax.set_ylim(y_axis - 0.8, y_headers + 0.75)
        ax.axis('off')

    # ── Legend ─────────────────────────────────────────────────────────
    handles = [
        Line2D([0], [0], marker='D', color='w', markerfacecolor='#1f77b4',
               markersize=6, markeredgecolor='k', markeredgewidth=0.3,
               label='Harari (sig)'),
        Line2D([0], [0], marker='D', color='w', markerfacecolor='#ff7f0e',
               markersize=6, markeredgecolor='k', markeredgewidth=0.3,
               label='Morabito (sig)'),
        Line2D([0], [0], marker='D', color='w', markerfacecolor='white',
               markersize=6, markeredgecolor='#888', markeredgewidth=0.8,
               label='Not significant'),
        Line2D([0], [0], marker='D', color='w', markerfacecolor='#d62728',
               markersize=6, markeredgecolor='k', markeredgewidth=0.3,
               label='Pooled (DL)'),
        Line2D([0], [0], linestyle='--', color='#888', lw=0.6,
               label='I = 0 (null)'),
        Line2D([0], [0], color='none', marker='None',
               label='Diamond size ∝ weight'),
    ]

    fig.legend(handles=handles, loc='lower center', ncol=6, fontsize=7,
               bbox_to_anchor=(0.5, -0.01), frameon=True, fancybox=False,
               edgecolor='#CCC', handletextpad=0.3, columnspacing=0.8)

    fig.suptitle("Moran's I meta-analysis: spatial autocorrelation across all controls",
                 fontsize=11, fontweight='bold', y=1.0)

    plt.tight_layout(h_pad=2.0)
    plt.savefig(str(FIGURES_DIR / 'Fig_forest_morans_I.pdf'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(str(FIGURES_DIR / 'Fig_forest_morans_I.svg'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print("✓ Fig_forest_morans_I.pdf / .svg")

In [ ]:
print("\n── Moran's I Heatmap (all controls × all variables) ──")

pivot_I = df_all.pivot(index='sample_name', columns='variable', values='morans_I')
pivot_p = df_all.pivot(index='sample_name', columns='variable', values='p_value')
pivot_ds = df_all.groupby('sample_name')['dataset'].first()

# Order: Harari first, then Morabito
order_h = sorted([s for s, d in pivot_ds.items() if d == 'Harari'])
order_m = sorted([s for s, d in pivot_ds.items() if d == 'Morabito'])
sample_order = order_h + order_m

var_order = ['SnC (binary)', 'Senescence Score', 'SASP', 'SenMayo', 'Fridman_Up',
             'p53_Targets', 'CellCycleArrest', 'DDR', 'AntiApoptosis',
             'CellSurfaceMarkers', 'LysosomalContent', 'SD_TMC']
var_order = [v for v in var_order if v in pivot_I.columns]

pivot_I = pivot_I.loc[sample_order, var_order]
pivot_p = pivot_p.loc[sample_order, var_order]

fig, ax = plt.subplots(figsize=(6, 0.35 * len(sample_order) + 1.5))

vmax = max(abs(pivot_I.values.min()), abs(pivot_I.values.max()), 0.1)
im = ax.imshow(pivot_I.values, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)

for i in range(pivot_I.shape[0]):
    for j in range(pivot_I.shape[1]):
        val = pivot_I.values[i, j]
        pv = pivot_p.values[i, j]
        if np.isnan(val):
            continue
        sig = '***' if pv < 0.001 else ('**' if pv < 0.01 else ('*' if pv < 0.05 else ''))
        txt_color = 'white' if abs(val) > vmax * 0.5 else 'black'
        ax.text(j, i, f'{val:+.2f}', ha='center', va='center', fontsize=4, color=txt_color)
        if sig:
            ax.text(j, i + 0.35, sig, ha='center', va='center', fontsize=3.5,
                    color=txt_color, fontweight='bold')

# Dataset separator line
if len(order_h) > 0 and len(order_m) > 0:
    ax.axhline(len(order_h) - 0.5, color='black', linewidth=1.5)
    ax.text(-0.6, len(order_h) / 2 - 0.5, 'Harari', ha='right', va='center',
            fontsize=6, fontweight='bold', rotation=90)
    ax.text(-0.6, len(order_h) + len(order_m) / 2 - 0.5, 'Morabito',
            ha='right', va='center', fontsize=6, fontweight='bold', rotation=90)

ax.set_xticks(range(len(var_order)))
ax.set_xticklabels(var_order, fontsize=5.5, rotation=45, ha='right')
ax.set_yticks(range(len(sample_order)))
ax.set_yticklabels(sample_order, fontsize=5)
ax.set_title("Moran's I — All Control Samples", fontsize=9, fontweight='bold', pad=8)

cbar = plt.colorbar(im, ax=ax, shrink=0.4, aspect=20, pad=0.03)
cbar.ax.tick_params(labelsize=5)
cbar.set_label("Moran's I", fontsize=6)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'Fig_morans_I_heatmap_all_controls.pdf'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_morans_I_heatmap_all_controls.svg'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_morans_I_heatmap_all_controls.pdf / .svg")

In [ ]:
print("\n" + "=" * 70)
print("META-ANALYSIS SUMMARY")
print("=" * 70)
print()
print(f"{'Variable':<26s} {'Pooled I':>10s} {'95% CI':>20s} {'p (RE)':>12s} "
      f"{'p (Fisher)':>12s} {'I²':>7s} {'k sig/k':>8s}")
print("─" * 100)

for _, row in df_meta.iterrows():
    ci_lo = row['pooled_I'] - 1.96 * row['pooled_SE']
    ci_hi = row['pooled_I'] + 1.96 * row['pooled_SE']
    
    p_re_str = f"{row['p_RE']:.2e}" if row['p_RE'] >= 1e-10 else "<1e-10"
    p_fi_str = f"{row['fisher_p']:.2e}" if row['fisher_p'] >= 1e-10 else "<1e-10"
    
    sig = ''
    if row['p_RE'] < 0.001: sig = '***'
    elif row['p_RE'] < 0.01: sig = '**'
    elif row['p_RE'] < 0.05: sig = '*'
    
    print(f"{row['variable']:<26s} {row['pooled_I']:>+10.4f} "
          f"[{ci_lo:+.4f}, {ci_hi:+.4f}] {p_re_str:>12s} "
          f"{p_fi_str:>12s} {row['I2_pct']:>5.1f}% "
          f"{int(row['n_sig_samples']):>3d}/{int(row['k_studies']):>2d}  {sig}")

print(f"\nInterpretation:")
print(f"  Pooled I > 0 with p < 0.05 → senescence is spatially CLUSTERED in controls")
print(f"  I² > 50% → substantial heterogeneity across samples")
print(f"  Fisher's p confirms combined significance across all samples")

In [ ]:
print("\n── Generating Moran scatter plots for all controls ──")

# Re-build sample_adatas for scatter plot computation
# (This re-processes; can be skipped if memory is tight)

n_ctrl = total_controls
n_cols = min(5, n_ctrl)
n_rows = int(np.ceil(n_ctrl / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 2.8 * n_rows))
axes = np.atleast_2d(axes)

plot_idx = 0

def plot_moran_scatter(ax, ad, name, dataset, df_all):
    """Plot Moran scatter for sen_score."""
    score_col = 'sen_score' if 'sen_score' in ad.obs.columns else 'is_senescent_num'
    x = ad.obs[score_col].values.astype(float)
    if x.std() == 0:
        ax.text(0.5, 0.5, 'No variance', transform=ax.transAxes, ha='center')
        ax.set_title(f"{name}", fontsize=6)
        return
    
    x_std = (x - x.mean()) / x.std()
    
    W = ad.obsp['spatial_connectivities']
    if sp_sparse.issparse(W):
        row_sums = np.array(W.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1
        W_norm = W.multiply(1 / row_sums[:, np.newaxis])
        lag = np.array(W_norm @ x_std).flatten()
    else:
        row_sums = W.sum(axis=1)
        row_sums[row_sums == 0] = 1
        W_norm = W / row_sums[:, np.newaxis]
        lag = W_norm @ x_std
    
    is_snc = ad.obs['is_senescent'].values == 1
    
    ax.scatter(x_std[~is_snc], lag[~is_snc], c='#CCCCCC', s=1, alpha=0.3,
               edgecolors='none', rasterized=True)
    ax.scatter(x_std[is_snc], lag[is_snc], c='#D62728', s=4, alpha=0.7,
               edgecolors='none', rasterized=True, zorder=3)
    
    m, b = np.polyfit(x_std, lag, 1)
    x_line = np.linspace(x_std.min(), x_std.max(), 100)
    ax.plot(x_line, m * x_line + b, color='#333333', linewidth=0.8)
    ax.axhline(0, color='#999', linewidth=0.3)
    ax.axvline(0, color='#999', linewidth=0.3)
    
    # Get stats from df
    moran_row = df_all[(df_all['sample_name'] == name) & 
                       (df_all['variable'] == 'Senescence Score')]
    if len(moran_row) > 0:
        I_val = moran_row.iloc[0]['morans_I']
        pval = moran_row.iloc[0]['p_value']
        sig = moran_row.iloc[0]['significant']
        ax.text(0.04, 0.96, f"I={I_val:+.3f}\np={pval:.3f}{sig}",
                transform=ax.transAxes, fontsize=4.5, va='top', ha='left',
                family='monospace', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                          edgecolor='#CCC', linewidth=0.3, alpha=0.9))
    
    ds_tag = 'H' if dataset == 'Harari' else 'M'
    ax.set_title(f"{name} ({ds_tag})", fontsize=6, fontweight='bold', pad=2)
    ax.tick_params(labelsize=4)

# Process all controls for scatter plots
all_samples_for_plot = []

for gsm in harari_controls:
    mask = (adata_harari.obs['gsm_id'] == gsm).values
    ad = adata_harari[mask].copy()
    valid = ~np.isnan(ad.obsm['spatial']).any(axis=1)
    if valid.sum() < ad.n_obs:
        ad = ad[valid].copy()
    sq.gr.spatial_neighbors(ad, n_neighs=K_NEIGH, coord_type='generic')
    ad.obs['is_senescent_num'] = ad.obs['is_senescent'].astype(float)
    meta = HARARI_META.get(gsm, {'name': gsm})
    all_samples_for_plot.append((ad, meta['name'], 'Harari'))

for sid in morabito_controls:
    mask = (adata_morabito_ctrl.obs['sample_id'] == sid).values
    ad = adata_morabito_ctrl[mask].copy()
    valid = ~np.isnan(ad.obsm['spatial']).any(axis=1)
    if valid.sum() < ad.n_obs:
        ad = ad[valid].copy()
    sq.gr.spatial_neighbors(ad, n_neighs=K_NEIGH, coord_type='generic')
    ad.obs['is_senescent_num'] = ad.obs['is_senescent'].astype(float)
    # Find matching name from df_all
    matching = df_all[(df_all['dataset'] == 'Morabito') & (df_all['sample_id'] == sid)]
    if len(matching) > 0:
        name = matching.iloc[0]['sample_name']
    else:
        name = sid
    all_samples_for_plot.append((ad, name, 'Morabito'))

for idx, (ad, name, dataset) in enumerate(all_samples_for_plot):
    r, c = divmod(idx, n_cols)
    plot_moran_scatter(axes[r, c], ad, name, dataset, df_all)

# Turn off unused axes
for idx in range(len(all_samples_for_plot), n_rows * n_cols):
    r, c = divmod(idx, n_cols)
    axes[r, c].axis('off')

fig.suptitle("Moran Scatter — Senescence Score (All Controls)",
             fontsize=10, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'Fig_moran_scatter_all_controls.pdf'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_moran_scatter_all_controls.svg'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_moran_scatter_all_controls.pdf / .svg")

# ─────────────────────────────────────────────────────────────────────────────
# DONE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("ALL DONE")
print("=" * 70)
print(f"\nOutputs:")
print(f"  {RESULTS_DIR / 'morans_I_all_controls.csv'}")
print(f"  {RESULTS_DIR / 'meta_analysis_morans_I.csv'}")
print(f"  {FIGURES_DIR / 'Fig_forest_morans_I.pdf'}")
print(f"  {FIGURES_DIR / 'Fig_morans_I_heatmap_all_controls.pdf'}")
print(f"  {FIGURES_DIR / 'Fig_moran_scatter_all_controls.pdf'}")

---
## 06b · Restore output directories

**Why.** Section 04 lifts the Moran's-I notebook's own config, which reassigns
`DATA_DIR`, `RESULTS_DIR` and `FIGURES_DIR` to its meta-analysis output tree.
Left as-is, every figure and table from section 07 onward would be written
under `results/10_spatial/meta_controls/` instead of the spatial-validation
directories set in section 01.

This cell restores them. It is the one addition to the merged code — the three
source notebooks each ran standalone, so the collision only exists once they
share a kernel.

In [ ]:
# -----------------------------------------------------------------------------
# Restore the section-01 output directories.
# Section 04 (lifted from the Moran's I notebook) reassigned these to its own
# meta-analysis tree; sections 07+ expect the spatial-validation tree.
# -----------------------------------------------------------------------------
FIGURES_DIR = BASE / MODULE / DATASET / "figures"
RESULTS_DIR = BASE / MODULE / DATASET / "results"
DATA_DIR    = BASE / MODULE / DATASET / "data"
for _d in (FIGURES_DIR, RESULTS_DIR, DATA_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print(f"  figures -> {FIGURES_DIR}")
print(f"  results -> {RESULTS_DIR}")
print(f"  data    -> {DATA_DIR}")

---
## 07 · Composition regression

**Why.** This is the control the module exists for.

Cell types are spatially organised. If senescence scores differ by cell type —
and module 07 established that they do — then the score inherits the spatial
structure of the composition. Moran's I on the raw score cannot distinguish
"senescence clusters" from "oligodendrocytes cluster, and oligodendrocytes score
higher".

**Test.** Regress the spot senescence score on local cell-type proportions. The
residual is the part of the score not predicted by what is there.

**Formula.** `sen_score ~ prop_celltype_1 + … + prop_celltype_k`

**Display.** R² of the composition fit — how much of the score composition
explains — plus score-versus-proportion scatters per cell type.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4a — COMPOSITION REGRESSION (prep for residual Moran's I)
# ═══════════════════════════════════════════════════════════════════════════════

import statsmodels.api as sm

print("="*64); print("CELL 4a — COMPOSITION REGRESSION"); print("="*64)

c2l_cols = [c for c in ctrl.obs.columns if c.startswith('c2l_')]
print(f"\n  Adjusting sen_score for {len(c2l_cols)} cell types: {[c.replace('c2l_','') for c in c2l_cols]}")
print(f"  Model (per sample, global OLS):  sen_score = β0 + Σ βj·c2l_j + residual")

fit_rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]
    y=ad.obs['sen_score'].astype(float).values
    X=ad.obs[c2l_cols].astype(float).values
    model=sm.OLS(y, sm.add_constant(X, has_constant='add')).fit()
    ad.obs['sen_resid']=model.resid
    betas=model.params[1:]; std_betas=betas*X.std(0); ti=np.argmax(np.abs(std_betas))
    fit_rows.append({'label':SAMPLE_LABELS[sid],'n':ad.n_obs,'R2':model.rsquared,
                     'top_celltype':c2l_cols[ti].replace('c2l_',''),
                     'oligo_beta':betas[c2l_cols.index('c2l_Oligodendrocyte')] if 'c2l_Oligodendrocyte' in c2l_cols else np.nan})
df_fit=pd.DataFrame(fit_rows).sort_values('R2',ascending=False)

print(f"\n  {'Sample':<8s} {'n':>5s} {'R²':>6s} {'top driver':>16s} {'oligoβ':>8s}")
print("  "+"─"*46)
for _,r in df_fit.iterrows():
    print(f"  {r['label']:<8s} {r['n']:>5d} {r['R2']:>6.3f} {r['top_celltype']:>16s} {r['oligo_beta']:>+8.3f}")
print(f"\n  Mean R² = {df_fit['R2'].mean():.3f}  (fraction of sen_score variance explained by composition)")
df_fit.to_csv(RESULTS_DIR/'composition_regression_fit.csv',index=False)
print(f"  ✓ saved. Low R² + Oligodendrocyte as top driver expected.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4a-scatter — sen_score vs oligodendrocyte / excitatory proportion
# ═══════════════════════════════════════════════════════════════════════════════
from scipy.stats import spearmanr
print("="*64); print("CELL 4a-scatter — sen_score vs composition"); print("="*64)

pairs = [('c2l_Oligodendrocyte','Oligodendrocyte'), ('c2l_Excitatory','Excitatory')]
sen = ctrl.obs['sen_score'].astype(float).values

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
hb=None
for ax,(col,name) in zip(axes, pairs):
    x = ctrl.obs[col].astype(float).values
    rho,p = spearmanr(x, sen)
    hb = ax.hexbin(x, sen, gridsize=40, cmap='magma', bins='log', mincnt=1)
    ax.set_xlabel(f"{name} proportion", fontsize=8)
    ax.set_ylabel("sen_score", fontsize=8)
    ax.set_title(f"{name}\nSpearman ρ = {rho:+.2f}  (p={p:.1e})", fontsize=8.5, fontweight='bold')
    ax.tick_params(labelsize=7)
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
    print(f"  sen_score vs {name:<16s}: Spearman ρ = {rho:+.3f}  (p={p:.1e}, n={len(x):,})")
fig.colorbar(hb, ax=axes, label='log10 spots', shrink=0.8)
fig.suptitle("Senescence score vs cell-type proportion (all control spots)",
             fontsize=9.5, fontweight='bold', y=1.04)
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_senscore_vs_composition.{fmt}', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print(f"\n  ✓ Fig_senscore_vs_composition.pdf / .svg")
print(f"  Note: pooled across spots (descriptive). Spatial test = Cell 4b.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4a-scatter — sen_score vs ALL cell-type proportions
# ═══════════════════════════════════════════════════════════════════════════════
from scipy.stats import spearmanr
print("="*64); print("CELL 4a-scatter — sen_score vs ALL cell types"); print("="*64)
c2l_cols  = [c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes = [c.replace('c2l_','') for c in c2l_cols]
sen = ctrl.obs['sen_score'].astype(float).values

# ── values table first ───────────────────────────────────────────────────────
stats=[]
for col,name in zip(c2l_cols,celltypes):
    x=ctrl.obs[col].astype(float).values
    rho,p=spearmanr(x,sen)
    stats.append({'celltype':name,'rho':rho,'p':p})
df_rho=pd.DataFrame(stats).sort_values('rho',ascending=False).reset_index(drop=True)
print(f"\n  Spearman ρ: sen_score vs cell-type proportion (n={len(sen):,} spots)")
print(f"  {'CellType':<16s} {'ρ':>8s} {'p':>10s} {'direction':>12s}")
print("  "+"─"*48)
for _,r in df_rho.iterrows():
    d = 'co-located ↑' if r['rho']>0.05 else ('anti ↓' if r['rho']<-0.05 else 'weak/none')
    print(f"  {r['celltype']:<16s} {r['rho']:>+8.3f} {r['p']:>10.1e} {d:>12s}")
df_rho.to_csv(RESULTS_DIR/'senscore_vs_composition_spearman.csv',index=False)

# ── hexbin grid ──────────────────────────────────────────────────────────────
order = df_rho['celltype'].tolist()
ncol = 4; nrow = int(np.ceil(len(order)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.4*nrow))   # wider + taller per row
axes = axes.ravel()
hb=None
for ax, name in zip(axes, order):
    col = f'c2l_{name}'
    x = ctrl.obs[col].astype(float).values
    rho = df_rho.loc[df_rho['celltype']==name,'rho'].iloc[0]
    p   = df_rho.loc[df_rho['celltype']==name,'p'].iloc[0]
    hb = ax.hexbin(x, sen, gridsize=35, cmap='magma', bins='log', mincnt=1)
    ax.set_xlabel(f"{name} proportion", fontsize=8)
    ax.set_ylabel("sen_score", fontsize=8)
    tcolor = '#C0392B' if rho>0 else ('#2b5d8c' if rho<0 else '#555')
    ax.set_title(f"{name}\nρ = {rho:+.2f}  (p={p:.0e})", fontsize=8.5, fontweight='bold', color=tcolor, pad=8)
    ax.tick_params(labelsize=7)
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
for ax in axes[len(order):]: ax.set_visible(False)

fig.suptitle("Senescence score vs cell-type proportion — all 7 types (control spots)",
             fontsize=11, fontweight='bold')
# layout: reserve space, THEN add colorbar in the leftover slot so it doesn't crowd panels
fig.tight_layout(rect=[0, 0, 0.92, 0.96], h_pad=2.5, w_pad=1.8)
cax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
fig.colorbar(hb, cax=cax, label='log10 spots')

for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_senscore_vs_composition_all.{fmt}', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print(f"\n  ✓ Fig_senscore_vs_composition_all.pdf / .svg")
print(f"  Note: pooled, descriptive. Red title = positive ρ, blue = negative. Spatial test = 4b.")

---
## 08 · Residual Moran's I

**Why.** Moran's I recomputed on the residual from section 07, on the same
graph, with the same permutation null.

**This is the number to quote.** Autocorrelation that survives composition
adjustment is spatial structure in senescence itself. If the residual I
collapses to zero, the raw signal was architecture and the rest of the module
describes tissue anatomy rather than senescence biology.

**Display.** Residual I per sample, meta-pooled, next to the raw I so the drop
is legible.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4b — RESIDUAL MORAN'S I (raw vs composition-adjusted)
# ═══════════════════════════════════════════════════════════════════════════════
from scipy.stats import wilcoxon
print("="*64); print("CELL 4b — RESIDUAL MORAN'S I"); print("="*64)
N_PERM=999

def morans_I(values, W):
    z=np.asarray(values,float)-np.mean(values); N=len(z); S0=W.sum()
    Wz=np.array(W@z).ravel() if sparse.issparse(W) else W@z; zz=z@z
    return np.nan if zz==0 else (N/S0)*(z@Wz)/zz
def morans_p(values, W, obs_I, n=N_PERM, seed=42):
    rng=np.random.default_rng(seed); v=np.asarray(values,float)
    null=np.array([morans_I(rng.permutation(v),W) for _ in range(n)])
    return (np.sum(null>=obs_I)+1)/(n+1)

rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; W=ad.obsp['spatial_connectivities']
    raw=ad.obs['sen_score'].astype(float).values; res=ad.obs['sen_resid'].astype(float).values
    Ir,Ie=morans_I(raw,W),morans_I(res,W)
    rows.append({'label':SAMPLE_LABELS[sid],'I_raw':Ir,'p_raw':morans_p(raw,W,Ir),
                 'I_res':Ie,'p_res':morans_p(res,W,Ie),'delta':Ir-Ie,
                 'pct_drop':100*(Ir-Ie)/Ir if Ir else np.nan})
df_mi=pd.DataFrame(rows).sort_values('I_raw',ascending=False)

print(f"\n  {'Sample':<8s} {'I_raw':>7s} {'p_raw':>7s} {'I_res':>7s} {'p_res':>7s} {'%drop':>6s} {'survives':>9s}")
print("  "+"─"*56)
for _,r in df_mi.iterrows():
    surv=(r['p_res']<0.05)and(r['I_res']>0)
    print(f"  {r['label']:<8s} {r['I_raw']:>+7.3f} {r['p_raw']:>7.1e} {r['I_res']:>+7.3f} "
          f"{r['p_res']:>7.1e} {r['pct_drop']:>5.0f}% {'✓' if surv else '✗':>9s}")
mr,me=df_mi['I_raw'].mean(),df_mi['I_res'].mean()
nsv=int(((df_mi['p_res']<0.05)&(df_mi['I_res']>0)).sum())
try: _,wp=wilcoxon(df_mi['I_raw'],df_mi['I_res'])
except Exception: wp=np.nan
print(f"\n  Mean I {mr:+.3f} → {me:+.3f}  (Δ {mr-me:+.3f}, {100*(mr-me)/mr:.0f}%)")
print(f"  Residual clustering significant: {nsv}/{len(df_mi)} samples")
print(f"  Paired Wilcoxon (raw vs residual): p={wp:.4f}")
df_mi.to_csv(RESULTS_DIR/'residual_morans_I.csv',index=False)

fig,ax=plt.subplots(figsize=(5,4))
for _,r in df_mi.iterrows():
    c='#2E7D32' if (r['p_res']<0.05 and r['I_res']>0) else '#C0392B'
    ax.plot([0,1],[r['I_raw'],r['I_res']],'-o',color=c,alpha=0.55,ms=4,lw=1)
ax.plot([0,1],[mr,me],'-D',color='black',lw=2.4,ms=7,zorder=5)
ax.axhline(0,color='#999',ls='--',lw=0.6)
ax.set_xticks([0,1]); ax.set_xticklabels(['Raw','Residual\n(composition\nremoved)'],fontsize=8)
ax.set_xlim(-0.15,1.15); ax.set_ylabel("Moran's I",fontsize=9)
ax.set_title(f"Clustering survives composition adjustment\nmean I {mr:.2f}→{me:.2f}; {nsv}/{len(df_mi)} significant",
             fontsize=9,fontweight='bold')
for sp in ['top','right']: ax.spines[sp].set_visible(False)
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_residual_morans_I.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_residual_morans_I.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4c.1 — PERMUTATION NULL (raw + residual): DATA PREP + DISPLAY
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: sample_adatas (spatial_connectivities, sen_score, sen_resid), SAMPLE_LABELS.
# Assemble per-sample graph + raw score + residual; confirm readiness. NO compute yet.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 4c.1 — PERMUTATION NULL PREP (raw + residual)"); print("="*64)

N_SHUFFLE = 100
SEED = 0

print(f"\n  Plan: for EACH sample, shuffle the values across spots {N_SHUFFLE}× and recompute")
print(f"        Moran's I — separately for RAW sen_score and for RESIDUAL (sen_resid).")
print(f"        Two nulls (raw-null, residual-null). Each real I compared to its OWN null.")
print(f"        Tests: (raw ≫ null) clustering is real; (residual ≫ null) the composition-")
print(f"        adjusted clustering is also real, not a regression artifact.\n")

prep=[]
for sid in sorted(sample_adatas):
    ad = sample_adatas[sid]
    has_W = 'spatial_connectivities' in ad.obsp
    has_resid = 'sen_resid' in ad.obs.columns
    W = ad.obsp['spatial_connectivities'] if has_W else None
    raw = ad.obs['sen_score'].astype(float).values
    res = ad.obs['sen_resid'].astype(float).values if has_resid else None
    prep.append({
        'sid':sid, 'label':SAMPLE_LABELS[sid], 'n_spots':ad.n_obs,
        'has_graph':has_W, 'has_resid':has_resid,
        'avg_k': float(np.array(W.sum(1)).ravel().mean()) if has_W else np.nan,
        'raw_var': float(np.var(raw)),
        'res_var': float(np.var(res)) if res is not None else np.nan,
    })
df_prep = pd.DataFrame(prep)

print(f"  {'Sample':<8s} {'nSpots':>7s} {'graph':>6s} {'resid':>6s} {'avgK':>6s} {'raw_var':>9s} {'res_var':>9s}")
print("  "+"─"*56)
for _,r in df_prep.iterrows():
    print(f"  {r['label']:<8s} {r['n_spots']:>7d} {'✓' if r['has_graph'] else '✗':>6s} "
          f"{'✓' if r['has_resid'] else '✗':>6s} {r['avg_k']:>6.1f} "
          f"{r['raw_var']:>9.4f} {r['res_var']:>9.4f}")

# ── readiness ────────────────────────────────────────────────────────────────
n_no_graph = int((~df_prep['has_graph']).sum())
n_no_resid = int((~df_prep['has_resid']).sum())
n_zero_raw = int((df_prep['raw_var']==0).sum())
n_zero_res = int((df_prep['res_var'].fillna(0)==0).sum())
print(f"\n  Readiness:")
print(f"    graph present    : {int(df_prep['has_graph'].sum())}/{len(df_prep)}"
      f"{'  ⚠ MISSING' if n_no_graph else ' ✓'}")
print(f"    residual present : {int(df_prep['has_resid'].sum())}/{len(df_prep)}"
      f"{'  ⚠ run Cell 4a first' if n_no_resid else ' ✓'}")
print(f"    raw_var > 0      : {len(df_prep)-n_zero_raw}/{len(df_prep)}{'  ⚠' if n_zero_raw else ' ✓'}")
print(f"    res_var > 0      : {len(df_prep)-n_zero_res}/{len(df_prep)}{'  ⚠' if n_zero_res else ' ✓'}")
print(f"\n  → if all ✓: 4c.2 computes BOTH nulls per sample; 4c.3 plots raw|residual|null forest.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4c.2 — PERMUTATION NULL (raw + residual): COMPUTE
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: 4c.1 (N_SHUFFLE, SEED), sample_adatas (sen_score, sen_resid), SAMPLE_LABELS.
# Real I and shuffled null for BOTH raw and residual. Each real vs its OWN null. NO plot.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 4c.2 — PERMUTATION NULL COMPUTE (raw + residual)"); print("="*64)

def morans_I(values, W):
    z=np.asarray(values,float)-np.mean(values); N=len(z); S0=W.sum()
    Wz=np.array(W@z).ravel() if sparse.issparse(W) else W@z; zz=z@z
    return np.nan if zz==0 else (N/S0)*(z@Wz)/zz

def null_stats(values, W, rng, n):
    real = morans_I(values, W)
    null = np.array([morans_I(rng.permutation(values), W) for _ in range(n)])
    z = (real-null.mean())/null.std(ddof=1) if null.std()>0 else np.nan
    p = (np.sum(null >= real)+1)/(n+1)
    return real, null.mean(), null.std(ddof=1), np.percentile(null,2.5), np.percentile(null,97.5), z, p

rng = np.random.default_rng(SEED)
rows=[]
print(f"\n  Computing raw-null and residual-null ({N_SHUFFLE} shuffles each)...")
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; W=ad.obsp['spatial_connectivities']
    raw=ad.obs['sen_score'].astype(float).values
    res=ad.obs['sen_resid'].astype(float).values
    rI,rNm,rNs,rLo,rHi,rZ,rP = null_stats(raw, W, rng, N_SHUFFLE)
    eI,eNm,eNs,eLo,eHi,eZ,eP = null_stats(res, W, rng, N_SHUFFLE)
    rows.append({'label':SAMPLE_LABELS[sid],
                 'raw_I':rI,'raw_null':rNm,'raw_lo':rLo,'raw_hi':rHi,'raw_z':rZ,'raw_p':rP,
                 'res_I':eI,'res_null':eNm,'res_lo':eLo,'res_hi':eHi,'res_z':eZ,'res_p':eP})
df_perm=pd.DataFrame(rows).sort_values('raw_I',ascending=False).reset_index(drop=True)

print(f"\n  {'Sample':<8s} | {'raw_I':>7s} {'raw_null':>9s} {'raw_z':>6s} | {'res_I':>7s} {'res_null':>9s} {'res_z':>6s}")
print("  "+"─"*64)
for _,r in df_perm.iterrows():
    print(f"  {r['label']:<8s} | {r['raw_I']:>+7.3f} {r['raw_null']:>+9.4f} {r['raw_z']:>6.1f} | "
          f"{r['res_I']:>+7.3f} {r['res_null']:>+9.4f} {r['res_z']:>6.1f}")

print(f"\n  RAW:      mean I {df_perm['raw_I'].mean():+.3f}, mean null {df_perm['raw_null'].mean():+.5f}, "
      f"mean z {df_perm['raw_z'].mean():.0f}σ, above-null95 {int((df_perm['raw_I']>df_perm['raw_hi']).sum())}/15")
print(f"  RESIDUAL: mean I {df_perm['res_I'].mean():+.3f}, mean null {df_perm['res_null'].mean():+.5f}, "
      f"mean z {df_perm['res_z'].mean():.0f}σ, above-null95 {int((df_perm['res_I']>df_perm['res_hi']).sum())}/15")
df_perm.to_csv(RESULTS_DIR/'morans_permutation_raw_residual.csv',index=False)
print(f"\n  ✓ saved. → 4c.3 plots raw | residual | null forest.")
print(f"  Read: raw≫null (clustering real); res≫null (adjusted clustering also real, not regression noise).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4c.3 — FOREST: raw vs residual vs null Moran's I per sample
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: df_perm (4c.2). Raw I, residual I, and shuffled null band per sample.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 4c.3 — FOREST: raw | residual | null"); print("="*64)

RAW_C, RES_C, NULL_C = '#6a51a3', '#C0392B', '#9e9e9e'
d = df_perm.copy()   # sorted by raw_I desc
n_rows=len(d); y=np.arange(n_rows)[::-1]

fig,ax=plt.subplots(figsize=(9, 0.46*n_rows+1.4))
trans=ax.get_yaxis_transform()
COLX={'raw':1.04,'res':1.18,'rz':1.30}
yhead=n_rows-0.1
for key,lab in [('raw',"raw I"),('res',"res I"),('rz',"res z")]:
    ax.text(COLX[key],yhead,lab,transform=trans,fontsize=8,fontweight='bold',ha='center',va='bottom')

for i,(_,r) in enumerate(d.iterrows()):
    yi=y[i]
    if i%2==0: ax.axhspan(yi-0.5,yi+0.5,color='#f7f7f7',zorder=0)
    # null band (use raw null CI — both nulls ≈0, essentially the same floor)
    ax.plot([r['raw_lo'],r['raw_hi']],[yi,yi],color=NULL_C,lw=3,alpha=0.45,solid_capstyle='round',zorder=2)
    ax.plot(r['raw_null'],yi,'o',color=NULL_C,ms=3.5,zorder=3)
    # connectors null→res→raw
    ax.plot([r['raw_null'],r['raw_I']],[yi,yi],color='#e2e2e2',lw=0.8,zorder=1)
    ax.plot(r['res_I'],yi,'D',color=RES_C,ms=6,mec='black',mew=0.3,zorder=4)
    ax.plot(r['raw_I'],yi,'o',color=RAW_C,ms=6.5,mec='black',mew=0.3,zorder=4)
    ax.text(-0.02,yi,r['label'],transform=trans,fontsize=8,ha='right',va='center',fontweight='bold')
    ax.text(COLX['raw'],yi,f"{r['raw_I']:.3f}",transform=trans,fontsize=7,ha='center',va='center',color=RAW_C,fontweight='bold')
    ax.text(COLX['res'],yi,f"{r['res_I']:.3f}",transform=trans,fontsize=7,ha='center',va='center',color=RES_C,fontweight='bold')
    ax.text(COLX['rz'],yi,f"{r['res_z']:.0f}σ",transform=trans,fontsize=7,ha='center',va='center')

ax.axvline(0,color='#999',lw=0.6,ls='--')
ax.set_ylim(-0.7,n_rows+0.2); ax.set_yticks([])
ax.set_xlim(-0.04, d['raw_I'].max()*1.12)
ax.set_xlabel("Moran's I",fontsize=9)
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
ax.set_title("Clustering is real and survives composition adjustment\n"
             f"raw {d['raw_I'].mean():.2f} (z={d['raw_z'].mean():.0f}σ) → residual {d['res_I'].mean():.2f} "
             f"(z={d['res_z'].mean():.0f}σ); both 15/15 above null",
             fontsize=10,fontweight='bold',loc='left',pad=10)
ax.legend([plt.Line2D([0],[0],marker='o',color='w',markerfacecolor=RAW_C,mec='black',ms=7),
           plt.Line2D([0],[0],marker='D',color='w',markerfacecolor=RES_C,mec='black',ms=7),
           plt.Line2D([0],[0],marker='o',color='w',markerfacecolor=NULL_C,ms=6)],
          ['Raw sen_score','Residual (composition removed)','Shuffled null (95% CI)'],
          fontsize=8,frameon=False,loc='lower right')
plt.subplots_adjust(right=0.64)
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_morans_raw_residual_null.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_morans_raw_residual_null.pdf / .svg")

---
## 09 · Getis-Ord Gi\* — hot versus cold spots

**Why.** Moran's I is one number for a whole section; it says clustering exists
but not where. Gi\* is local — it flags each spot as sitting in a
significantly high or low neighbourhood.

That turns the question into a comparison: what is *in* the hot spots that is
not in the cold ones?

**Display.** Hot/cold assignment, a calculation trace across all cell types,
fractions by compartment and by dominant cell type, and the composition of hot
versus cold as stacked bars.

**Read the by-compartment split.** If hot spots are mostly white matter, the
composition difference is the compartment difference.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.1 — Gi* HOT vs COLD: DATA PREP + DISPLAY
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.1 — Gi* HOT vs COLD: DATA PREP"); print("="*64)

HOT_Z, COLD_Z = 1.96, -1.96
c2l_cols  = [c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes = [c.replace('c2l_','') for c in c2l_cols]

# (a) per-sample hot/cold counts
count_rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
    count_rows.append({'label':SAMPLE_LABELS[sid],'n_total':ad.n_obs,
                       'n_hot':int((z>HOT_Z).sum()),'n_cold':int((z<COLD_Z).sum())})
df_counts=pd.DataFrame(count_rows).sort_values('n_total',ascending=False)
print("\n  (a) Hot/cold counts per sample:")
print(f"  {'Sample':<8s} {'nTotal':>7s} {'nHot':>6s} {'nCold':>6s}  {'gate(≥15)':>10s}")
print("  "+"─"*44)
for _,r in df_counts.iterrows():
    ok=(r['n_hot']>=15) and (r['n_cold']>=15)
    print(f"  {r['label']:<8s} {r['n_total']:>7d} {r['n_hot']:>6d} {r['n_cold']:>6d}  {'✓' if ok else '✗ thin':>10s}")

# (b) composition table (gate-passing samples)
comp_rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
    hot,cold=z>HOT_Z,z<COLD_Z
    if hot.sum()<15 or cold.sum()<15: continue
    for ct,col in zip(celltypes,c2l_cols):
        comp_rows.append({'label':SAMPLE_LABELS[sid],'celltype':ct,
                          'hot_mean':ad.obs.loc[hot,col].mean(),
                          'cold_mean':ad.obs.loc[cold,col].mean()})
df_hotcold_comp=pd.DataFrame(comp_rows)
print(f"\n  (b) Composition table: {df_hotcold_comp['label'].nunique()} samples × {len(celltypes)} types")

prev=df_hotcold_comp.groupby('celltype')[['hot_mean','cold_mean']].mean()
prev['diff']=prev['hot_mean']-prev['cold_mean']; prev=prev.sort_values('diff',ascending=False)
print(f"\n  Preview — mean proportion (hot vs cold) per cell type:")
print(f"  {'CellType':<16s} {'hot':>7s} {'cold':>7s} {'diff':>8s}")
print("  "+"─"*42)
for ct,r in prev.iterrows():
    print(f"  {ct:<16s} {r['hot_mean']:>7.3f} {r['cold_mean']:>7.3f} {r['diff']:>+8.3f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.1c — CALCULATION TRACE: all cell types, hot vs cold
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.1c — TRACE: all cell types, hot vs cold"); print("="*64)

gate_sids=[sid for sid in sorted(sample_adatas)
           if (sample_adatas[sid].obs['gi_star_z'].astype(float).values>HOT_Z).sum()>=15
           and (sample_adatas[sid].obs['gi_star_z'].astype(float).values<COLD_Z).sum()>=15]
print(f"\n  Gate-passing: {len(gate_sids)}/15  (excluded: "
      f"{[SAMPLE_LABELS[s] for s in sorted(sample_adatas) if s not in gate_sids]})")

trace={}
for col,ct in zip(c2l_cols,celltypes):
    diffs=[]
    for sid in gate_sids:
        ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
        v=ad.obs[col].astype(float).values
        diffs.append(v[z>HOT_Z].mean()-v[z<COLD_Z].mean())
    trace[ct]=np.array(diffs)

print(f"\n  {'CellType':<16s} {'meanΔ':>8s} {'sdΔ':>7s} {'dir(hot>cold)':>14s} {'consistent?':>12s}")
print("  "+"─"*62)
summ=[]
for ct in sorted(celltypes,key=lambda c:-trace[c].mean()):
    d=trace[ct]; npos=int((d>0).sum())
    cons='✓ all' if (npos==len(d) or npos==0) else ('mostly' if max(npos,len(d)-npos)>=len(d)*0.8 else 'mixed')
    summ.append({'celltype':ct,'mean_diff':d.mean(),'sd':d.std(ddof=1),'npos':npos,'n':len(d)})
    print(f"  {ct:<16s} {d.mean()*100:>+7.1f} {d.std(ddof=1)*100:>7.1f} {npos:>6d}/{len(d)}{'':>4s} {cons:>12s}")
pd.DataFrame(summ).to_csv(RESULTS_DIR/'hotcold_composition_trace.csv',index=False)
print(f"\n  ✓ saved. '✓ all' = every sample agrees on direction (strongest signal).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.2 — Gi* FRACTIONS by COMPARTMENT (both denominators)
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.2 — Gi* FRACTIONS by COMPARTMENT"); print("="*64)

rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
    comp=ad.obs['compartment'].astype(str).values
    hot,cold=z>HOT_Z,z<COLD_Z
    rec={'label':SAMPLE_LABELS[sid],'n_hot':int(hot.sum()),'n_cold':int(cold.sum())}
    for cp in ['GM','WM']:
        m=comp==cp; rec[f'{cp}_total']=int(m.sum())
        rec[f'{cp}_hot']=int((m&hot).sum()); rec[f'{cp}_cold']=int((m&cold).sum())
    rows.append(rec)
df_frac=pd.DataFrame(rows)
for cp in ['GM','WM']:
    tot=df_frac[f'{cp}_total'].replace(0,np.nan)
    df_frac[f'{cp}_hotfrac_a']=df_frac[f'{cp}_hot']/tot
    df_frac[f'{cp}_coldfrac_a']=df_frac[f'{cp}_cold']/tot
    df_frac[f'{cp}_ratio_a']=df_frac[f'{cp}_hot']/(df_frac[f'{cp}_cold']+1)
    df_frac[f'{cp}_hotshare_b']=df_frac[f'{cp}_hot']/df_frac['n_hot'].replace(0,np.nan)
    df_frac[f'{cp}_coldshare_b']=df_frac[f'{cp}_cold']/df_frac['n_cold'].replace(0,np.nan)

print("\n  Per-sample counts (compartment × hot/cold):")
print(f"  {'Sample':<8s} {'GMtot':>6s} {'GMhot':>6s} {'GMcold':>7s} | {'WMtot':>6s} {'WMhot':>6s} {'WMcold':>7s}")
print("  "+"─"*56)
for _,r in df_frac.iterrows():
    print(f"  {r['label']:<8s} {r['GM_total']:>6d} {r['GM_hot']:>6d} {r['GM_cold']:>7d} | "
          f"{r['WM_total']:>6d} {r['WM_hot']:>6d} {r['WM_cold']:>7d}")

def msd(s): s=pd.Series(s).dropna(); return f"{s.mean()*100:.1f} ± {s.std(ddof=1)*100:.1f}" if len(s)>1 else "—"
print(f"\n  GRID (a) WITHIN-COMPARTMENT (hot-in-X/all-X; mean±SD):")
print(f"  {'':<6s} {'Hot frac':>14s} {'Cold frac':>14s} {'Hot/Cold':>16s}")
for cp in ['WM','GM']:
    rr=pd.Series(df_frac[f'{cp}_ratio_a']).dropna()
    print(f"  {cp:<6s} {msd(df_frac[f'{cp}_hotfrac_a']):>14s} {msd(df_frac[f'{cp}_coldfrac_a']):>14s} "
          f"{rr.mean():>8.1f} ± {rr.std(ddof=1):>6.1f}")
print(f"\n  GRID (b) SHARE-OF-HOT (hot-in-X/all-hot; mean±SD):")
print(f"  {'':<6s} {'Hot share':>14s} {'Cold share':>14s}")
for cp in ['WM','GM']:
    print(f"  {cp:<6s} {msd(df_frac[f'{cp}_hotshare_b']):>14s} {msd(df_frac[f'{cp}_coldshare_b']):>14s}")
df_frac.to_csv(RESULTS_DIR/'gistar_fractions_compartment.csv',index=False)
print(f"\n  ✓ saved. (b) WM hot-share should echo the Step-3 WM enrichment; WM ratio unstable.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.3 — Gi* FRACTIONS by DOMINANT CELL TYPE (both denominators)
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.3 — Gi* FRACTIONS by DOMINANT CELL TYPE"); print("="*64)
print(f"\n  dominant = argmax over {len(celltypes)} c2l_ per spot (hard label; lossy).")

rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
    hot,cold=z>HOT_Z,z<COLD_Z
    dom=np.array(celltypes)[ad.obs[c2l_cols].values.argmax(1)]
    rec={'label':SAMPLE_LABELS[sid],'n_hot':int(hot.sum()),'n_cold':int(cold.sum())}
    for ct in celltypes:
        m=dom==ct; rec[f'{ct}_total']=int(m.sum())
        rec[f'{ct}_hot']=int((m&hot).sum()); rec[f'{ct}_cold']=int((m&cold).sum())
    rows.append(rec)
df_dom=pd.DataFrame(rows)

print(f"\n  Dominant-type spot counts per sample:")
h=f"  {'Sample':<8s}"+"".join(f"{ct[:5]:>7s}" for ct in celltypes); print(h); print("  "+"─"*(len(h)-2))
for _,r in df_dom.iterrows():
    print(f"  {r['label']:<8s}"+"".join(f"{r[f'{ct}_total']:>7d}" for ct in celltypes))

for ct in celltypes:
    tot=df_dom[f'{ct}_total'].replace(0,np.nan)
    df_dom[f'{ct}_hotfrac_a']=df_dom[f'{ct}_hot']/tot
    df_dom[f'{ct}_coldfrac_a']=df_dom[f'{ct}_cold']/tot
    df_dom[f'{ct}_hotshare_b']=df_dom[f'{ct}_hot']/df_dom['n_hot'].replace(0,np.nan)
def msd(s): s=pd.Series(s).dropna(); return f"{s.mean()*100:.1f}±{s.std(ddof=1)*100:.1f}" if len(s)>1 else "—"

print(f"\n  GRID (a) WITHIN-TYPE (hot-in-T/all-T):")
print(f"  {'CellType':<16s} {'Hot frac':>12s} {'Cold frac':>12s}")
for ct in sorted(celltypes,key=lambda c:-pd.Series(df_dom[f'{c}_hotfrac_a']).dropna().mean()):
    print(f"  {ct:<16s} {msd(df_dom[f'{ct}_hotfrac_a']):>12s} {msd(df_dom[f'{ct}_coldfrac_a']):>12s}")
print(f"\n  GRID (b) SHARE-OF-HOT (hot-in-T/all-hot):")
print(f"  {'CellType':<16s} {'Hot share':>12s}")
for ct in sorted(celltypes,key=lambda c:-pd.Series(df_dom[f'{c}_hotshare_b']).dropna().mean()):
    print(f"  {ct:<16s} {msd(df_dom[f'{ct}_hotshare_b']):>12s}")
df_dom.to_csv(RESULTS_DIR/'gistar_fractions_dominant_celltype.csv',index=False)
print(f"\n  ✓ saved. Rare dominant types (low counts above) → noisy; don't over-read.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.4 — COMPOSITION STACKED BAR: hot vs cold (with Unassigned)
# ═══════════════════════════════════════════════════════════════════════════════
# Colors = canonical cell-type palette (matches 10_spatial). Hot/cold shown by ROW,
# not color. Plotted as-is (no renorm); Unassigned = deconvolution residual.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.4 — COMPOSITION STACKED BAR"); print("="*64)

hot_c,cold_c=[],[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
    hot,cold=z>HOT_Z,z<COLD_Z
    if hot.sum()<15 or cold.sum()<15: continue
    hot_c.append(ad.obs.loc[hot,c2l_cols].mean().values)
    cold_c.append(ad.obs.loc[cold,c2l_cols].mean().values)
hot_mean=np.mean(hot_c,0)*100; cold_mean=np.mean(cold_c,0)*100; n_used=len(hot_c)
hot_un=100-hot_mean.sum(); cold_un=100-cold_mean.sum()
print(f"\n  {n_used} samples; column sums hot={hot_mean.sum():.1f} cold={cold_mean.sum():.1f}")
print(f"  Unassigned: hot={hot_un:.1f}% cold={cold_un:.1f}% (deconv residual, per-spot sum≈0.91)")
print(f"\n  {'CellType':<16s} {'Hot %':>7s} {'Cold %':>7s} {'Δ':>7s}")
for i in np.argsort(-(hot_mean-cold_mean)):
    print(f"  {celltypes[i]:<16s} {hot_mean[i]:>6.1f} {cold_mean[i]:>6.1f} {hot_mean[i]-cold_mean[i]:>+7.1f}")

# ── canonical cell-type palette (from 10_spatial) — identity colors, NOT direction ──
CT_COLORS={'Excitatory':'#2ca02c','Inhibitory':'#ff7f0e','Oligodendrocyte':'#9467bd',
           'Astrocyte':'#1f77b4','Microglia':'#8c564b','OPC':'#e377c2',
           'Vascular':'#17becf','Unassigned':'#e0e0e0'}
stack=['Oligodendrocyte','OPC','Astrocyte','Microglia','Vascular','Inhibitory','Excitatory','Unassigned']
idx={ct:i for i,ct in enumerate(celltypes)}

fig,ax=plt.subplots(figsize=(7.8,2.4))
for label,vals,un,y in [('Hot spots',hot_mean,hot_un,1),('Cold spots',cold_mean,cold_un,0)]:
    left=0
    for ct in stack:
        w=un if ct=='Unassigned' else vals[idx[ct]]
        ax.barh(y,w,left=left,height=0.6,color=CT_COLORS[ct],edgecolor='white',linewidth=0.5)
        if w>=5: ax.text(left+w/2,y,f"{w:.0f}",ha='center',va='center',fontsize=7,
                         color='#666' if ct=='Unassigned' else 'white',fontweight='bold')
        left+=w
ax.set_yticks([0,1]); ax.set_yticklabels(['Cold\nspots','Hot\nspots'],fontsize=9)
ax.set_xlim(0,100); ax.set_xlabel('Cell-type composition (%); Unassigned = deconv residual',fontsize=8)
ax.set_title(f"Hot vs cold spots: cell-type composition (Gi*; {n_used} controls)",fontsize=9,fontweight='bold')
ax.tick_params(labelsize=7)
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
ax.legend([plt.Rectangle((0,0),1,1,color=CT_COLORS[ct]) for ct in stack],stack,
          fontsize=6,frameon=False,ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.38))
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_composition_hot_vs_cold.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"\n  ✓ Fig_composition_hot_vs_cold.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.4b — COMPOSITION STACKED BAR: SnC | All | Hot | Cold  (all Gi*-consistent)
# ═══════════════════════════════════════════════════════════════════════════════
# Extends 5.4 with SnC and All reference rows so the baseline control (SnC≈All) and
# the enrichment (Hot vs Cold) live in ONE panel, single definition (Gi*). For F.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.4b — COMPOSITION: SnC | All | Hot | Cold"); print("="*64)

snc_c, all_c, hot_c, cold_c = [], [], [], []
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; o=ad.obs
    z=o['gi_star_z'].astype(float).values
    snc=o['is_senescent'].astype(bool).values
    hot,cold = z>HOT_Z, z<COLD_Z
    if hot.sum()<15 or cold.sum()<15: continue        # same gate as 5.4 (Gi*)
    all_c.append(o[c2l_cols].mean().values)
    hot_c.append(o.loc[hot, c2l_cols].mean().values)
    cold_c.append(o.loc[cold,c2l_cols].mean().values)
    snc_c.append(o.loc[snc, c2l_cols].mean().values if snc.sum()>=15 else np.full(len(c2l_cols),np.nan))

# per-sample means → cohort means (nanmean so thin-SnC sections don't break the SnC row)
snc_mean = np.nanmean(snc_c,0)*100; all_mean = np.mean(all_c,0)*100
hot_mean = np.mean(hot_c,0)*100;    cold_mean= np.mean(cold_c,0)*100
n_used=len(hot_c); n_snc=int(np.sum([~np.isnan(s).any() for s in snc_c]))
un = {k:100-v.sum() for k,v in [('SnC',snc_mean),('All',all_mean),('Hot',hot_mean),('Cold',cold_mean)]}
print(f"\n  {n_used} samples (Gi* gate); SnC row from {n_snc} samples with ≥15 SnC")
print(f"\n  {'CellType':<16s} {'SnC':>6s} {'All':>6s} {'Hot':>6s} {'Cold':>6s} {'Δhot-cold':>9s}")
for i in np.argsort(-(hot_mean-cold_mean)):
    print(f"  {celltypes[i]:<16s} {snc_mean[i]:>6.1f} {all_mean[i]:>6.1f} {hot_mean[i]:>6.1f} {cold_mean[i]:>6.1f} {hot_mean[i]-cold_mean[i]:>+9.1f}")

CT_COLORS={'Excitatory':'#2ca02c','Inhibitory':'#ff7f0e','Oligodendrocyte':'#9467bd',
           'Astrocyte':'#1f77b4','Microglia':'#8c564b','OPC':'#e377c2',
           'Vascular':'#17becf','Unassigned':'#e0e0e0'}
stack=['Oligodendrocyte','OPC','Astrocyte','Microglia','Vascular','Inhibitory','Excitatory','Unassigned']
idx={ct:i for i,ct in enumerate(celltypes)}

# rows top→bottom: SnC, All, Hot, Cold. y descending so SnC is on top.
rows=[('SnC spots',snc_mean,un['SnC'],3),('All spots',all_mean,un['All'],2),
      ('Hot spots',hot_mean,un['Hot'],1),('Cold spots',cold_mean,un['Cold'],0)]

fig,ax=plt.subplots(figsize=(7.8,3.4))
for label,vals,unv,y in rows:
    left=0
    for ct in stack:
        w=unv if ct=='Unassigned' else vals[idx[ct]]
        if not np.isfinite(w): continue
        ax.barh(y,w,left=left,height=0.62,color=CT_COLORS[ct],edgecolor='white',linewidth=0.5)
        if w>=5: ax.text(left+w/2,y,f"{w:.0f}",ha='center',va='center',fontsize=7,
                         color='#666' if ct=='Unassigned' else 'white',fontweight='bold')
        left+=w
# light divider between the reference pair (SnC|All) and the contrast pair (Hot|Cold)
ax.axhline(1.5,color='#bbb',lw=0.7,ls=(0,(4,3)))
ax.text(101,2.5,'reference',fontsize=6.5,color='#999',va='center',style='italic',rotation=90,ha='left')
ax.text(101,0.5,'Gi* contrast',fontsize=6.5,color='#999',va='center',style='italic',rotation=90,ha='left')

ax.set_yticks([3,2,1,0]); ax.set_yticklabels(['SnC\nspots','All\nspots','Hot\nspots','Cold\nspots'],fontsize=8.5)
ax.set_xlim(0,100); ax.set_xlabel('Cell-type composition (%); Unassigned = deconv residual',fontsize=8)
ax.set_title(f"Composition: SnC | all | Gi* hot | Gi* cold ({n_used} controls)",fontsize=9,fontweight='bold')
ax.tick_params(labelsize=7)
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
ax.legend([plt.Rectangle((0,0),1,1,color=CT_COLORS[ct]) for ct in stack],stack,
          fontsize=6,frameon=False,ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.22))
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_composition_snc_all_hot_cold.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"\n  ✓ Fig_composition_snc_all_hot_cold.pdf / .svg")

---
## 10 · Gi\* differences and within-compartment tests

**Why.** The hot-minus-cold difference per cell type, then the same restricted
to within GM and within WM. Within-compartment is the honest version: it asks
whether hot spots differ from cold spots *of the same tissue class*.

**Feasibility first.** Section 5.6 in the source checks whether enough spots of
each class exist per compartment before testing — a test on four spots is
reported as not estimable rather than as a null.

**Display.** Diverging Δ bars, three panels — pooled, GM, WM — and the hot/cold
fraction summaries.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.5 — DIVERGING Δ BAR: hot − cold composition
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.5 — DIVERGING Δ BAR"); print("="*64)

per={ct:[] for ct in celltypes}
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
    hot,cold=z>HOT_Z,z<COLD_Z
    if hot.sum()<15 or cold.sum()<15: continue
    for ct,col in zip(celltypes,c2l_cols):
        per[ct].append(ad.obs.loc[hot,col].mean()-ad.obs.loc[cold,col].mean())
stat=[]
for ct in celltypes:
    d=np.array(per[ct]); npos=int((d>0).sum())
    stat.append({'celltype':ct,'mean_d':d.mean()*100,'sd_d':d.std(ddof=1)*100,
                 'n':len(d),'npos':npos,'unanimous':(npos==len(d) or npos==0)})
df_d=pd.DataFrame(stat).sort_values('mean_d')

print(f"\n  {'CellType':<16s} {'meanΔ%':>8s} {'sdΔ%':>7s} {'dir':>8s}")
for _,r in df_d.sort_values('mean_d',ascending=False).iterrows():
    print(f"  {r['celltype']:<16s} {r['mean_d']:>+7.1f} {r['sd_d']:>7.1f} {int(r['npos'])}/{int(r['n'])}")

HOT_C,COLD_C='#cb181d','#2b8cbe'   # hot/+Δ = red, cold/−Δ = blue
fig,ax=plt.subplots(figsize=(6.2,3.2))
for i,(_,r) in enumerate(df_d.iterrows()):
    c=HOT_C if r['mean_d']>0 else COLD_C
    ax.barh(i,r['mean_d'],height=0.66,color=c,edgecolor='white',linewidth=0.5,zorder=2)
    ax.errorbar(r['mean_d'],i,xerr=r['sd_d'],fmt='none',ecolor='#555',elinewidth=0.8,capsize=2.5,zorder=3)
    if r['unanimous']:
        ax.text(r['mean_d']+np.sign(r['mean_d'])*(r['sd_d']+1.2),i,'●',fontsize=5,va='center',
                ha='left' if r['mean_d']>0 else 'right',color=c)
    ax.text(r['mean_d']+np.sign(r['mean_d'])*0.5,i,f"{r['mean_d']:+.1f}",va='center',
            ha='left' if r['mean_d']>0 else 'right',fontsize=6.5,color='#222')
ax.axvline(0,color='#333',lw=0.8)
ax.set_yticks(range(len(df_d))); ax.set_yticklabels(df_d['celltype'],fontsize=8)
ax.set_xlabel("Δ composition: hot − cold (percentage points)",fontsize=8)
ax.set_title("Cell-type enrichment in hot vs cold spots\n(Gi*; ● = unanimous direction)",fontsize=9,fontweight='bold')
mx=(df_d['mean_d'].abs()+df_d['sd_d']).max()*1.18; ax.set_xlim(-mx,mx)
ax.text(mx*0.96,len(df_d)-0.3,'enriched\nin HOT',fontsize=6.5,ha='right',va='top',color=HOT_C,style='italic')
ax.text(-mx*0.96,0.3,'enriched\nin COLD',fontsize=6.5,ha='left',va='bottom',color=COLD_C,style='italic')
ax.tick_params(labelsize=7)
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_composition_diverging.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"\n  ✓ Fig_composition_diverging.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.6 — WITHIN-COMPARTMENT hot vs cold: FEASIBILITY
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.6 — WITHIN-COMPARTMENT FEASIBILITY"); print("="*64)
MIN_CLASS=15

feas=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
    comp=ad.obs['compartment'].astype(str).values
    rec={'label':SAMPLE_LABELS[sid]}
    for cp in ['GM','WM']:
        cm=comp==cp
        rec[f'{cp}_hot']=int((cm&(z>HOT_Z)).sum()); rec[f'{cp}_cold']=int((cm&(z<COLD_Z)).sum())
    feas.append(rec)
df_feas=pd.DataFrame(feas)

print(f"\n  {'Sample':<8s} {'GMhot':>6s} {'GMcold':>7s} {'GM✓':>4s} | {'WMhot':>6s} {'WMcold':>7s} {'WM✓':>4s}")
print("  "+"─"*52)
gm_ok=wm_ok=0
for _,r in df_feas.iterrows():
    g=r['GM_hot']>=MIN_CLASS and r['GM_cold']>=MIN_CLASS
    w=r['WM_hot']>=MIN_CLASS and r['WM_cold']>=MIN_CLASS
    gm_ok+=g; wm_ok+=w
    print(f"  {r['label']:<8s} {r['GM_hot']:>6d} {r['GM_cold']:>7d} {'✓' if g else '✗':>4s} | "
          f"{r['WM_hot']:>6d} {r['WM_cold']:>7d} {'✓' if w else '✗':>4s}")
print(f"\n  Feasible — GM: {gm_ok}/15   WM: {wm_ok}/15")
print(f"  WM expected to mostly fail (cold near-absent). WM<3 → within-WM exploratory only.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.7 — WITHIN-COMPARTMENT hot−cold Δ + PAIRED TEST
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.7 — WITHIN-COMPARTMENT Δ + TEST"); print("="*64)

def collect(comp_filter):
    out={ct:[] for ct in celltypes}
    for sid in sorted(sample_adatas):
        ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
        keep=np.ones(len(z),bool) if comp_filter is None else (ad.obs['compartment'].astype(str).values==comp_filter)
        hot,cold=keep&(z>HOT_Z),keep&(z<COLD_Z)
        if hot.sum()<MIN_CLASS or cold.sum()<MIN_CLASS: continue
        for ct,col in zip(celltypes,c2l_cols):
            out[ct].append(ad.obs.loc[hot,col].mean()-ad.obs.loc[cold,col].mean())
    return {ct:np.array(v) for ct,v in out.items()}

pooled,gm,wm=collect(None),collect('GM'),collect('WM')
n_p,n_g,n_w=len(pooled[celltypes[0]]),len(gm[celltypes[0]]),len(wm[celltypes[0]])
print(f"\n  Samples — pooled:{n_p} within-GM:{n_g} within-WM:{n_w}")
print(f"  Test: Wilcoxon signed-rank on per-sample Δ (within-GM); FDR-BH across 7 types.")
print(f"  within-WM (n={n_w}) = exploratory. Caveat: 7 c2l_ coupled (sum-constrained).\n")

rows=[]; gm_p=[]
for ct in celltypes:
    d=gm[ct]
    try: _,p=wilcoxon(d) if (len(d)>=2 and np.any(d!=0)) else (np.nan,np.nan)
    except Exception: p=np.nan
    gm_p.append(p)
    rows.append({'celltype':ct,'pooled_d':pooled[ct].mean()*100,'gm_d':d.mean()*100,'gm_p':p,
                 'gm_npos':int((d>0).sum()),'gm_n':len(d),
                 'wm_d':wm[ct].mean()*100 if n_w else np.nan,'wm_npos':int((wm[ct]>0).sum()) if n_w else 0,'wm_n':n_w})
df=pd.DataFrame(rows)
v=~df['gm_p'].isna(); df.loc[v,'gm_fdr']=multipletests(df.loc[v,'gm_p'],method='fdr_bh')[1]
df=df.sort_values('pooled_d',ascending=False).reset_index(drop=True)

print(f"  {'CellType':<16s} {'pooledΔ':>8s} | {'gmΔ':>7s} {'gm_dir':>7s} {'gm_FDR':>8s} {'retain%':>8s} | {'wmΔ':>8s}")
print("  "+"─"*72)
for _,r in df.iterrows():
    ret=(r['gm_d']/r['pooled_d']*100) if abs(r['pooled_d'])>0.5 else np.nan
    fdr=r.get('gm_fdr',np.nan); sig='★' if (not pd.isna(fdr) and fdr<0.05) else ' '
    print(f"  {r['celltype']:<16s} {r['pooled_d']:>+8.1f} | {r['gm_d']:>+7.1f} "
          f"{int(r['gm_npos'])}/{int(r['gm_n'])}{'':>3s} "
          f"{(f'{fdr:.3f}'+sig) if not pd.isna(fdr) else '—':>8s} "
          f"{(f'{ret:.0f}%') if not pd.isna(ret) else '—':>8s} | "
          f"{r['wm_d']:>+7.1f} {int(r['wm_npos'])}/{int(r['wm_n'])}")
df.to_csv(RESULTS_DIR/'within_compartment_hotcold_delta.csv',index=False)
print(f"\n  ✓ saved. retain% = fraction of pooled gap surviving within GM (the anatomy control).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.8 — THREE-PANEL DIVERGING BAR (pooled | within-GM | within-WM)
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.8 — THREE-PANEL DIVERGING BAR"); print("="*64)

strata={'Pooled\n(all spots)':pooled,'Within GM':gm,'Within WM\n(exploratory)':wm}
ns={k:len(v[celltypes[0]]) for k,v in strata.items()}
gm_sig={ct:(not pd.isna(df.loc[df['celltype']==ct,'gm_fdr'].iloc[0]) and
            df.loc[df['celltype']==ct,'gm_fdr'].iloc[0]<0.05) for ct in celltypes}
order=sorted(celltypes,key=lambda ct: pooled[ct].mean())
mx=max(abs(np.mean(v[ct]))+(np.std(v[ct],ddof=1) if len(v[ct])>1 else 0)
       for v in strata.values() for ct in celltypes)*1.2*100
HOT_C,COLD_C='#cb181d','#2b8cbe'

fig,axes=plt.subplots(1,3,figsize=(11,3.4),sharey=True)
for ax,(name,dat) in zip(axes,strata.items()):
    is_wm='WM' in name
    for i,ct in enumerate(order):
        d=dat[ct]*100; md=d.mean(); sd=d.std(ddof=1) if len(d)>1 else 0
        c=HOT_C if md>0 else COLD_C
        ax.barh(i,md,height=0.66,color=c,alpha=0.45 if is_wm else 1.0,
                hatch='///' if is_wm else None,edgecolor='white',linewidth=0.5,zorder=2)
        if len(d)>1: ax.errorbar(md,i,xerr=sd,fmt='none',ecolor='#555',elinewidth=0.7,capsize=2,zorder=3)
        if name=='Within GM' and gm_sig[ct]:
            ax.text(md+np.sign(md)*(sd+1),i,'★',fontsize=8,va='center',ha='left' if md>0 else 'right',color=c)
    ax.axvline(0,color='#333',lw=0.7); ax.set_xlim(-mx,mx)
    ax.set_title(f"{name}  (n={ns[name]})",fontsize=8.5,fontweight='bold')
    ax.tick_params(labelsize=7)
    for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels(order,fontsize=8)
axes[1].set_xlabel("Δ composition: hot − cold (percentage points)",fontsize=8.5)
fig.suptitle("Hot-spot composition enrichment shrinks within compartment (anatomy-controlled)",
             fontsize=9.5,fontweight='bold',y=1.02)
op=pooled['Oligodendrocyte'].mean()*100; og=gm['Oligodendrocyte'].mean()*100
axes[1].text(0.5,-0.30,f"Oligodendrocyte: {op:+.0f} → {og:+.0f}  ({og/op*100:.0f}% retained)",
             transform=axes[1].transAxes,fontsize=7,ha='center',color='#444',style='italic')
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_within_compartment_composition.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_within_compartment_composition.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.9 — Gi* HOT/COLD FRACTION SUMMARY (whole-section)
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.9 — Gi* HOT/COLD FRACTION SUMMARY"); print("="*64)

rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
    n=len(z); nh=int((z>HOT_Z).sum()); nc=int((z<COLD_Z).sum())
    rows.append({'label':SAMPLE_LABELS[sid],'n_total':n,'hot_frac':nh/n,'cold_frac':nc/n,'hot_to_cold':nh/(nc+1)})
df_gf=pd.DataFrame(rows)

print(f"\n  Per-sample (whole-section):")
print(f"  {'Sample':<8s} {'nTot':>5s} {'hot%':>7s} {'cold%':>7s} {'hot/cold':>9s}")
print("  "+"─"*40)
for _,r in df_gf.iterrows():
    print(f"  {r['label']:<8s} {r['n_total']:>5d} {r['hot_frac']*100:>6.1f} {r['cold_frac']*100:>6.1f} {r['hot_to_cold']:>9.2f}")

def msd(s): return f"{s.mean()*100:.1f} ± {s.std(ddof=1)*100:.1f}"
def miqr(s): s=s*100; return f"{s.median():.1f} ({s.quantile(.25):.1f}–{s.quantile(.75):.1f})"
def msd_r(s): return f"{s.mean():.2f} ± {s.std(ddof=1):.2f}"
def miqr_r(s): return f"{s.median():.2f} ({s.quantile(.25):.2f}–{s.quantile(.75):.2f})"

print(f"\n  SUMMARY across 15 samples:")
print(f"  {'Metric':<16s} {'mean ± SD':>18s} {'median (IQR)':>20s}")
print(f"  {'Hot fraction %':<16s} {msd(df_gf['hot_frac']):>18s} {miqr(df_gf['hot_frac']):>20s}")
print(f"  {'Cold fraction %':<16s} {msd(df_gf['cold_frac']):>18s} {miqr(df_gf['cold_frac']):>20s}")
print(f"  {'Hot/cold ratio':<16s} {msd_r(df_gf['hot_to_cold']):>18s} {miqr_r(df_gf['hot_to_cold']):>20s}")
df_gf.to_csv(RESULTS_DIR/'gistar_fraction_summary.csv',index=False)
print(f"\n  ✓ saved. median(IQR) is the robust summary (fractions bounded/skewed).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.9b — Gi* FRACTION SUMMARY by COMPARTMENT
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.9b — Gi* FRACTION SUMMARY by COMPARTMENT"); print("="*64)

rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; z=ad.obs['gi_star_z'].astype(float).values
    comp=ad.obs['compartment'].astype(str).values
    rec={'label':SAMPLE_LABELS[sid]}
    for cp in ['GM','WM']:
        m=comp==cp; ntot=int(m.sum())
        if ntot==0: rec[f'{cp}_hotfrac']=np.nan; rec[f'{cp}_coldfrac']=np.nan; rec[f'{cp}_ratio']=np.nan; continue
        nh=int((m&(z>HOT_Z)).sum()); nc=int((m&(z<COLD_Z)).sum())
        rec[f'{cp}_hotfrac']=nh/ntot; rec[f'{cp}_coldfrac']=nc/ntot; rec[f'{cp}_ratio']=nh/(nc+1)
    rows.append(rec)
df_cf=pd.DataFrame(rows)

print(f"\n  Per-sample within-compartment fractions:")
print(f"  {'Sample':<8s} | {'GM hot%':>7s} {'GM cold%':>8s} {'GM r':>6s} | {'WM hot%':>7s} {'WM cold%':>8s} {'WM r':>7s}")
print("  "+"─"*60)
for _,r in df_cf.iterrows():
    def f(x): return f"{x*100:.1f}" if not pd.isna(x) else "  —"
    print(f"  {r['label']:<8s} | {f(r['GM_hotfrac']):>7s} {f(r['GM_coldfrac']):>8s} {r['GM_ratio']:>6.2f} | "
          f"{f(r['WM_hotfrac']):>7s} {f(r['WM_coldfrac']):>8s} {r['WM_ratio']:>7.2f}")

def msd(s): s=s.dropna()*100; return f"{s.mean():.1f} ± {s.std(ddof=1):.1f}" if len(s)>1 else "—"
def miqr(s): s=s.dropna()*100; return f"{s.median():.1f} ({s.quantile(.25):.1f}–{s.quantile(.75):.1f})" if len(s)>1 else "—"
def msd_r(s): s=s.dropna(); return f"{s.mean():.2f} ± {s.std(ddof=1):.2f}" if len(s)>1 else "—"
def miqr_r(s): s=s.dropna(); return f"{s.median():.2f} ({s.quantile(.25):.2f}–{s.quantile(.75):.2f})" if len(s)>1 else "—"

print(f"\n  SUMMARY by compartment (mean±SD | median IQR):")
for cp in ['WM','GM']:
    print(f"\n  [{cp}]")
    print(f"    {'Hot fraction %':<16s} {msd(df_cf[f'{cp}_hotfrac']):>16s}   {miqr(df_cf[f'{cp}_hotfrac']):>20s}")
    print(f"    {'Cold fraction %':<16s} {msd(df_cf[f'{cp}_coldfrac']):>16s}   {miqr(df_cf[f'{cp}_coldfrac']):>20s}")
    print(f"    {'Hot/cold ratio':<16s} {msd_r(df_cf[f'{cp}_ratio']):>16s}   {miqr_r(df_cf[f'{cp}_ratio']):>20s}")
df_cf.to_csv(RESULTS_DIR/'gistar_fraction_summary_compartment.csv',index=False)
print(f"\n  ✓ FLAG: WM cold near-absent → WM ratio unstable; report 'cold near-absent', not the number.")

---
## 11 · LISA — local Moran's I

**Why.** A second local statistic with a different definition. Gi\* asks
whether a neighbourhood's *mean* is extreme; LISA asks whether a spot and its
neighbours *covary*. They can disagree, and agreement between them is stronger
evidence than either alone.

**Quadrants.** HH (high spot, high neighbours) is a focus; LL is a cold region;
HL and LH are spatial outliers. The contrast run here is HH versus not-significant.

**Display.** HH/NS composition with the same feasibility gate and
within-compartment test as section 10, then the three-panel diverging bar.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6.1 — LISA QUADRANT FRACTIONS
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 6.1 — LISA QUADRANT FRACTIONS"); print("="*64)
QUADS=['HH','LL','HL','LH','ns']

rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; q=ad.obs['lisa_quad'].astype(str).values; n=len(q)
    rec={'label':SAMPLE_LABELS[sid],'n_total':n}
    for k in QUADS: rec[f'n_{k}']=int((q==k).sum()); rec[f'f_{k}']=(q==k).sum()/n
    rows.append(rec)
df_lisa=pd.DataFrame(rows)

print(f"\n  Per-sample quadrant fractions (%):")
print(f"  {'Sample':<8s} {'nTot':>5s} {'HH%':>6s} {'LL%':>6s} {'HL%':>6s} {'LH%':>6s} {'ns%':>6s}")
print("  "+"─"*48)
for _,r in df_lisa.iterrows():
    print(f"  {r['label']:<8s} {r['n_total']:>5d} {r['f_HH']*100:>5.1f} {r['f_LL']*100:>5.1f} "
          f"{r['f_HL']*100:>5.1f} {r['f_LH']*100:>5.1f} {r['f_ns']*100:>5.1f}")

def miqr(s): s=s*100; return f"{s.median():.1f} ({s.quantile(.25):.1f}–{s.quantile(.75):.1f})"
print(f"\n  SUMMARY across 15 samples:")
print(f"  {'Quadrant':<10s} {'median % (IQR)':>20s} {'total spots':>13s}")
for k in QUADS:
    print(f"  {k:<10s} {miqr(df_lisa[f'f_{k}']):>20s} {int(df_lisa[f'n_{k}'].sum()):>13d}")
print(f"\n  Total HH spots: {int(df_lisa['n_HH'].sum())}   Total LL spots: {int(df_lisa['n_LL'].sum())}")
df_lisa.to_csv(RESULTS_DIR/'lisa_quadrant_fractions.csv',index=False)
print(f"\n  ✓ saved. HH≈Gi* hot, LL≈Gi* cold; HH/LL dominate, HL/LH rare.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6.2 — LISA HH vs NS COMPOSITION: PREP + FEASIBILITY
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 6.2 — HH vs NS COMPOSITION: PREP"); print("="*64)
MIN_CLASS=15
_4=['Oligodendrocyte','OPC','Astrocyte','Excitatory']

feas=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; q=ad.obs['lisa_quad'].astype(str).values
    feas.append({'label':SAMPLE_LABELS[sid],'n_HH':int((q=='HH').sum()),'n_NS':int((q=='ns').sum())})
df_f=pd.DataFrame(feas)
print(f"\n  {'Sample':<8s} {'nHH':>5s} {'nNS':>6s} {'gate':>8s}")
print("  "+"─"*30)
n_ok=0
for _,r in df_f.iterrows():
    ok=r['n_HH']>=MIN_CLASS; n_ok+=ok
    print(f"  {r['label']:<8s} {r['n_HH']:>5d} {r['n_NS']:>6d} {'✓' if ok else '✗ thin':>8s}")
print(f"\n  Passing HH gate: {n_ok}/15")

hh_d={ct:[] for ct in celltypes}
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; q=ad.obs['lisa_quad'].astype(str).values
    hh,ns=q=='HH',q=='ns'
    if hh.sum()<MIN_CLASS: continue
    for ct,col in zip(celltypes,c2l_cols):
        hh_d[ct].append(ad.obs.loc[hh,col].mean()-ad.obs.loc[ns,col].mean())

print(f"\n  POOLED HH−NS Δ preview ({len(hh_d[celltypes[0]])} samples):")
print(f"  {'CellType':<16s} {'Δ %':>8s} {'dir':>7s}  {'?':>9s}")
print("  "+"─"*44)
for ct in sorted(celltypes,key=lambda c:-np.mean(hh_d[c])):
    d=np.array(hh_d[ct]); npos=int((d>0).sum())
    print(f"  {ct:<16s} {d.mean()*100:>+7.1f} {npos}/{len(d)}{'':>3s}  {'◀ named' if ct in _4 else '':>9s}")
print(f"\n  HH≈senescent clusters; expect oligo↑ excitatory↓ (milder than Gi*; NS≠cold).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6.3 — HH vs NS Δ + WITHIN-COMPARTMENT TEST
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 6.3 — HH vs NS Δ + WITHIN-COMPARTMENT TEST"); print("="*64)

def collect_hhns(comp_filter):
    out={ct:[] for ct in celltypes}
    for sid in sorted(sample_adatas):
        ad=sample_adatas[sid]; q=ad.obs['lisa_quad'].astype(str).values
        keep=np.ones(len(q),bool) if comp_filter is None else (ad.obs['compartment'].astype(str).values==comp_filter)
        hh,ns=keep&(q=='HH'),keep&(q=='ns')
        if hh.sum()<MIN_CLASS or ns.sum()<MIN_CLASS: continue
        for ct,col in zip(celltypes,c2l_cols):
            out[ct].append(ad.obs.loc[hh,col].mean()-ad.obs.loc[ns,col].mean())
    return {ct:np.array(v) for ct,v in out.items()}

pooled_h,gm_h,wm_h=collect_hhns(None),collect_hhns('GM'),collect_hhns('WM')
n_p,n_g,n_w=len(pooled_h[celltypes[0]]),len(gm_h[celltypes[0]]),len(wm_h[celltypes[0]])
print(f"\n  Samples — pooled:{n_p} within-GM:{n_g} within-WM:{n_w}")
print(f"  Test: Wilcoxon on within-GM Δ + FDR. WM exploratory. Caveat: c2l_ coupled.\n")

rows=[]; gp=[]
for ct in celltypes:
    d=gm_h[ct]
    try: _,p=wilcoxon(d) if (len(d)>=2 and np.any(d!=0)) else (np.nan,np.nan)
    except Exception: p=np.nan
    gp.append(p)
    rows.append({'celltype':ct,'pooled_d':pooled_h[ct].mean()*100,'gm_d':d.mean()*100,'gm_p':p,
                 'gm_npos':int((d>0).sum()),'gm_n':len(d),
                 'wm_d':wm_h[ct].mean()*100 if n_w else np.nan,'wm_n':n_w})
dfh=pd.DataFrame(rows)
v=~dfh['gm_p'].isna(); dfh.loc[v,'gm_fdr']=multipletests(dfh.loc[v,'gm_p'],method='fdr_bh')[1]
dfh=dfh.sort_values('pooled_d',ascending=False).reset_index(drop=True)

print(f"  {'CellType':<16s} {'pooledΔ':>8s} | {'gmΔ':>7s} {'gm_dir':>7s} {'gm_FDR':>8s} {'retain%':>8s} | {'wmΔ':>7s}  {'rev?':>5s}")
print("  "+"─"*76)
for _,r in dfh.iterrows():
    ret=(r['gm_d']/r['pooled_d']*100) if abs(r['pooled_d'])>0.3 else np.nan
    fdr=r.get('gm_fdr',np.nan); sig='★' if (not pd.isna(fdr) and fdr<0.05) else ' '
    rev='◀' if r['celltype'] in _4 else ''
    print(f"  {r['celltype']:<16s} {r['pooled_d']:>+8.1f} | {r['gm_d']:>+7.1f} "
          f"{int(r['gm_npos'])}/{int(r['gm_n'])}{'':>3s} "
          f"{(f'{fdr:.3f}'+sig) if not pd.isna(fdr) else '—':>8s} "
          f"{(f'{ret:.0f}%') if not pd.isna(ret) else '—':>8s} | {r['wm_d']:>+7.1f}  {rev:>5s}")
dfh.to_csv(RESULTS_DIR/'lisa_hhns_within_compartment.csv',index=False)
print(f"\n  ✓ saved. retain% = HH−NS gap surviving within GM.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6.4 — HH vs NS THREE-PANEL DIVERGING BAR
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 6.4 — HH vs NS THREE-PANEL"); print("="*64)

strata={'Pooled\n(all spots)':pooled_h,'Within GM':gm_h,'Within WM\n(exploratory)':wm_h}
ns={k:len(v[celltypes[0]]) for k,v in strata.items()}
gm_sig={ct:(not pd.isna(dfh.loc[dfh['celltype']==ct,'gm_fdr'].iloc[0]) and
            dfh.loc[dfh['celltype']==ct,'gm_fdr'].iloc[0]<0.05) for ct in celltypes}
order=sorted(celltypes,key=lambda ct: pooled_h[ct].mean())
mx=max(abs(np.mean(v[ct]))+(np.std(v[ct],ddof=1) if len(v[ct])>1 else 0)
       for v in strata.values() for ct in celltypes)*1.2*100
HOT_C,COLD_C='#cb181d','#2b8cbe'

fig,axes=plt.subplots(1,3,figsize=(11,3.4),sharey=True)
for ax,(name,dat) in zip(axes,strata.items()):
    is_wm='WM' in name
    for i,ct in enumerate(order):
        d=dat[ct]*100; md=d.mean(); sd=d.std(ddof=1) if len(d)>1 else 0
        c=HOT_C if md>0 else COLD_C
        ax.barh(i,md,height=0.66,color=c,alpha=0.45 if is_wm else 1.0,
                hatch='///' if is_wm else None,edgecolor='white',linewidth=0.5,zorder=2)
        if len(d)>1: ax.errorbar(md,i,xerr=sd,fmt='none',ecolor='#555',elinewidth=0.7,capsize=2,zorder=3)
        if name=='Within GM' and gm_sig[ct]:
            ax.text(md+np.sign(md)*(sd+1),i,'★',fontsize=8,va='center',ha='left' if md>0 else 'right',color=c)
    ax.axvline(0,color='#333',lw=0.7); ax.set_xlim(-mx,mx)
    ax.set_title(f"{name}  (n={ns[name]})",fontsize=8.5,fontweight='bold'); ax.tick_params(labelsize=7)
    for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels(order,fontsize=8)
axes[1].set_xlabel("Δ composition: HH − NS (percentage points)",fontsize=8.5)
fig.suptitle("LISA HH vs NS composition enrichment by compartment",fontsize=9.5,fontweight='bold',y=1.02)
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_lisa_hhns_composition.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_lisa_hhns_composition.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6.6 — COMPOSITION STACKED BAR: all 4 LISA quadrants (HH/LL/HL/LH)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: Cell 3 (lisa_quad), c2l_*. Per-sample mean composition of each quadrant,
# plotted as-is with Unassigned. HL/LH are rare → flagged (few samples, noisy).
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 6.6 — COMPOSITION: all 4 LISA quadrants"); print("="*64)

MIN_CLASS = 15
QUAD_ORDER = ['HH','HL','LH','LL']   # HH top (senescent clusters) → LL bottom
QUAD_DESC = {'HH':'high among high (senescent clusters)','LL':'low among low (cold clusters)',
             'HL':'high among low (hot outlier)','LH':'low among high (cold outlier)'}

# per-sample mean composition per quadrant (gate ≥15 spots in that quadrant)
quad_comp = {q: [] for q in QUAD_ORDER}
quad_n    = {q: 0  for q in QUAD_ORDER}
for sid in sorted(sample_adatas):
    ad = sample_adatas[sid]
    q = ad.obs['lisa_quad'].astype(str).values
    for quad in QUAD_ORDER:
        m = q == quad
        if m.sum() >= MIN_CLASS:
            quad_comp[quad].append(ad.obs.loc[m, c2l_cols].mean().values)
            quad_n[quad] += 1

print(f"\n  Samples contributing per quadrant (gate ≥{MIN_CLASS}):")
for quad in QUAD_ORDER:
    print(f"    {quad}: {quad_n[quad]}/15  {'(thin — noisy)' if quad_n[quad] < 5 else ''}")

# mean composition per quadrant
quad_mean = {q: (np.mean(quad_comp[q],0)*100 if quad_comp[q] else np.full(len(celltypes),np.nan))
             for q in QUAD_ORDER}

print(f"\n  {'Quadrant':<6s} {'nSamp':>5s} | " + " ".join(f"{ct[:5]:>6s}" for ct in celltypes))
for quad in QUAD_ORDER:
    vals = quad_mean[quad]
    print(f"  {quad:<6s} {quad_n[quad]:>5d} | " + " ".join(f"{v:>6.1f}" if not np.isnan(v) else f"{'—':>6s}" for v in vals))

# ── plot ─────────────────────────────────────────────────────────────────────
CT_COLORS={'Excitatory':'#2ca02c','Inhibitory':'#ff7f0e','Oligodendrocyte':'#9467bd',
           'Astrocyte':'#1f77b4','Microglia':'#8c564b','OPC':'#e377c2',
           'Vascular':'#17becf','Unassigned':'#e0e0e0'}
stack=['Oligodendrocyte','OPC','Astrocyte','Microglia','Vascular','Inhibitory','Excitatory','Unassigned']
idx={ct:i for i,ct in enumerate(celltypes)}

fig,ax=plt.subplots(figsize=(8.2,3.4))
ypos={q:len(QUAD_ORDER)-1-i for i,q in enumerate(QUAD_ORDER)}   # HH on top
for quad in QUAD_ORDER:
    vals=quad_mean[quad]
    if np.all(np.isnan(vals)):
        ax.text(50,ypos[quad],f"{quad}: too few spots (n<{MIN_CLASS} in all samples)",
                ha='center',va='center',fontsize=7,color='#999',style='italic'); continue
    un=100-np.nansum(vals); left=0
    for ct in stack:
        w=un if ct=='Unassigned' else vals[idx[ct]]
        ax.barh(ypos[quad],w,left=left,height=0.62,color=CT_COLORS[ct],edgecolor='white',linewidth=0.5)
        if w>=5: ax.text(left+w/2,ypos[quad],f"{w:.0f}",ha='center',va='center',fontsize=6.5,
                         color='#666' if ct=='Unassigned' else 'white',fontweight='bold')
        left+=w
ax.set_yticks(list(ypos.values()))
ax.set_yticklabels([f"{q}\n(n={quad_n[q]})" for q in QUAD_ORDER],fontsize=8)
ax.set_xlim(0,100); ax.set_xlabel('Cell-type composition (%); Unassigned = deconv residual',fontsize=8)
ax.set_title("LISA quadrant cell-type composition (all 4 quadrants)",fontsize=9.5,fontweight='bold')
ax.tick_params(labelsize=7)
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
ax.legend([plt.Rectangle((0,0),1,1,color=CT_COLORS[ct]) for ct in stack],stack,
          fontsize=6,frameon=False,ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.30))
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_composition_lisa_4quadrants.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"\n  ✓ Fig_composition_lisa_4quadrants.pdf / .svg")

---
## 12 · Summary bars

**Why.** The three lines of evidence — residual Moran's I, Gi\* fractions, LISA
fractions — side by side per sample. Consistency across the three is what the
module can claim; a result present in only one is a property of that statistic.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7.1 — SUMMARY BARS: Point 1 (residual Moran's I)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: df_mi (Cell 4b). Per-sample raw vs residual I + aggregated mean.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 7.1 — SUMMARY BARS: Point 1"); print("="*64)

d = df_mi.sort_values('I_raw', ascending=False).reset_index(drop=True)
RAW_C, RES_C = '#6a51a3', '#bcbddc'   # raw = dark, residual = light (same hue, attenuation reads as fade)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 3.6),
                                gridspec_kw={'width_ratios':[3.2, 1]})

# ── per-sample paired bars ───────────────────────────────────────────────────
x = np.arange(len(d)); w = 0.4
axL.bar(x - w/2, d['I_raw'], w, color=RAW_C, label='Raw sen_score')
axL.bar(x + w/2, d['I_res'], w, color=RES_C, label='Residual (composition removed)')
axL.set_xticks(x); axL.set_xticklabels(d['label'], rotation=45, ha='right', fontsize=7)
axL.set_ylabel("Moran's I", fontsize=9)
axL.set_title("Per sample: clustering before vs after composition adjustment", fontsize=9, fontweight='bold')
axL.legend(fontsize=7, frameon=False)
axL.axhline(0, color='#999', lw=0.5)
for sp in ['top','right']: axL.spines[sp].set_visible(False)

# ── aggregated mean bars ─────────────────────────────────────────────────────
mr, me = d['I_raw'].mean(), d['I_res'].mean()
sr, se = d['I_raw'].std(ddof=1), d['I_res'].std(ddof=1)
axR.bar([0,1], [mr,me], 0.6, color=[RAW_C,RES_C], yerr=[sr,se], capsize=4, error_kw={'elinewidth':1})
axR.set_xticks([0,1]); axR.set_xticklabels(['Raw','Residual'], fontsize=8)
axR.set_ylabel("mean Moran's I", fontsize=9)
axR.set_title(f"Mean ± SD\n{mr:.3f} → {me:.3f} ({100*(mr-me)/mr:.0f}% drop)", fontsize=8.5, fontweight='bold')
for sp in ['top','right']: axR.spines[sp].set_visible(False)

fig.suptitle("Point 1 — Senescence clustering survives composition adjustment (15/15 samples)",
             fontsize=10, fontweight='bold', y=1.03)
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_summary_P1_moransI.{fmt}', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print(f"  ✓ Fig_summary_P1_moransI.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7.1a — TABLE-FOREST (per sample): Point 1 residual Moran's I
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: df_mi (Cell 4b). Per-sample raw vs residual I, two markers per row,
# ★ if residual p<0.05. Value columns: I_raw | I_res | %drop | p_res.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 7.1a — TABLE-FOREST (per sample): Point 1"); print("="*64)

RAW_C, RES_C = '#6a51a3', '#C0392B'   # raw = purple, residual = red (the one that matters)
d = df_mi.sort_values('I_raw', ascending=False).reset_index(drop=True)
n_rows = len(d)

fig, ax = plt.subplots(figsize=(11.5, 0.5*n_rows+1.4))
xspan = d[['I_raw','I_res']].values.max()*1.18
ax.set_xlim(-0.02, xspan)
trans = ax.get_yaxis_transform()

COLX={'raw':1.07,'res':1.24,'drop':1.42,'p':1.58}
yhead=n_rows-0.15
for key,lab in [('raw',"I raw"),('res',"I resid"),('drop',"%drop"),('p',"p resid")]:
    ax.text(COLX[key], yhead, lab, transform=trans, fontsize=8, fontweight='bold', ha='center', va='bottom')
ax.plot([COLX['raw']-0.05, COLX['p']+0.05],[yhead-0.12]*2, transform=trans, color='#ccc', lw=0.6, clip_on=False)

for i,(_,r) in enumerate(d.iterrows()):
    y=n_rows-1-i
    if i%2==0: ax.axhspan(y-0.5,y+0.5,color='#f4f4f4',zorder=0)
    # connector line raw→residual
    ax.plot([r['I_res'],r['I_raw']],[y,y],color='#bbb',lw=1.0,zorder=2)
    ax.plot(r['I_raw'],y,marker='o',color=RAW_C,ms=6,mec='black',mew=0.3,zorder=4)
    ax.plot(r['I_res'],y,marker='D',color=RES_C,ms=6,mec='black',mew=0.3,zorder=4)
    sig = r['p_res']<0.05 and r['I_res']>0
    if sig: ax.text(max(r['I_raw'],r['I_res'])+xspan*0.015, y, '★', fontsize=8, va='center', color=RES_C)
    ax.text(-0.02, y, r['label'], transform=trans, fontsize=8, ha='right', va='center',
            fontweight=('bold' if sig else 'normal'))
    ax.text(COLX['raw'], y, f"{r['I_raw']:.3f}", transform=trans, fontsize=7, ha='center', va='center')
    ax.text(COLX['res'], y, f"{r['I_res']:.3f}", transform=trans, fontsize=7, ha='center', va='center', fontweight='bold', color=RES_C)
    ax.text(COLX['drop'],y, f"{r['pct_drop']:.0f}%", transform=trans, fontsize=7, ha='center', va='center')
    pstr = f"{r['p_res']:.3f}" if r['p_res']>=0.001 else "<0.001"
    ax.text(COLX['p'], y, pstr, transform=trans, fontsize=7, ha='center', va='center',
            fontweight=('bold' if sig else 'normal'))

ax.axvline(0,color='#999',lw=0.6,ls='--')
ax.set_ylim(-0.7,n_rows+0.3); ax.set_yticks([])
ax.set_xlabel("Moran's I",fontsize=8)
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
ax.set_title("Point 1 — residual Moran's I per sample (raw vs composition-adjusted)",
             fontsize=10.5, fontweight='bold', pad=10, loc='left')
ax.text(0.0,-0.07,"● raw  ◆ residual (composition removed) · ★ residual p<0.05",
        transform=ax.transAxes, fontsize=7, color='#555')
plt.subplots_adjust(right=0.60)
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_forest_P1_persample.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_forest_P1_persample.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7.1b — POOLED SUMMARY: Point 1 residual Moran's I
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: df_mi. Mean raw → mean residual, diamonds + SD whiskers, paired Wilcoxon.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 7.1b — POOLED SUMMARY: Point 1"); print("="*64)
from scipy.stats import wilcoxon

mr, me = df_mi['I_raw'].mean(), df_mi['I_res'].mean()
sr, se = df_mi['I_raw'].std(ddof=1), df_mi['I_res'].std(ddof=1)
try: _,wp = wilcoxon(df_mi['I_raw'], df_mi['I_res'])
except Exception: wp=np.nan
nsv = int(((df_mi['p_res']<0.05)&(df_mi['I_res']>0)).sum())

fig, ax = plt.subplots(figsize=(7, 1.8))
RAW_C, RES_C = '#6a51a3', '#C0392B'
ax.errorbar(mr, 1, xerr=sr, fmt='o', color=RAW_C, ms=10, mec='black', mew=0.4,
            capsize=5, elinewidth=1.4, label='Raw')
ax.errorbar(me, 0, xerr=se, fmt='D', color=RES_C, ms=10, mec='black', mew=0.4,
            capsize=5, elinewidth=1.4, label='Residual (composition removed)')
ax.plot([me,mr],[0.5,0.5],alpha=0)  # spacing
ax.axvline(0, color='#999', lw=0.6, ls='--')
ax.set_yticks([0,1]); ax.set_yticklabels(['Residual','Raw'], fontsize=9)
ax.set_ylim(-0.6,1.6); ax.set_xlim(-0.02, (mr+sr)*1.15)
ax.set_xlabel("mean Moran's I (± SD)", fontsize=8.5)
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
ax.set_title(f"Point 1 pooled: {mr:.3f} → {me:.3f}  ({100*(mr-me)/mr:.0f}% drop)\n"
             f"{nsv}/{len(df_mi)} samples significant · paired Wilcoxon p={wp:.4f}",
             fontsize=9.5, fontweight='bold', pad=8, loc='left')
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_pooled_P1_moransI.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_pooled_P1_moransI.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7.2 — SUMMARY BARS: Point 2 (Gi* hot/cold fractions)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: df_gf (Cell 5.9). Hot/cold = direction → red/blue palette.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 7.2 — SUMMARY BARS: Point 2"); print("="*64)

d = df_gf.sort_values('hot_frac', ascending=False).reset_index(drop=True)
HOT_C, COLD_C = '#cb181d', '#2b8cbe'

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 3.6),
                                gridspec_kw={'width_ratios':[3.2, 1]})

x = np.arange(len(d)); w = 0.4
axL.bar(x - w/2, d['hot_frac']*100, w, color=HOT_C, label='Hot fraction')
axL.bar(x + w/2, d['cold_frac']*100, w, color=COLD_C, label='Cold fraction')
axL.set_xticks(x); axL.set_xticklabels(d['label'], rotation=45, ha='right', fontsize=7)
axL.set_ylabel("% of section", fontsize=9)
axL.set_title("Per sample: hot vs cold fraction (whole-section)", fontsize=9, fontweight='bold')
axL.legend(fontsize=7, frameon=False)
for sp in ['top','right']: axL.spines[sp].set_visible(False)

mh, mc = d['hot_frac'].mean()*100, d['cold_frac'].mean()*100
sh, sc = d['hot_frac'].std(ddof=1)*100, d['cold_frac'].std(ddof=1)*100
axR.bar([0,1], [mh,mc], 0.6, color=[HOT_C,COLD_C], yerr=[sh,sc], capsize=4, error_kw={'elinewidth':1})
axR.set_xticks([0,1]); axR.set_xticklabels(['Hot','Cold'], fontsize=8)
axR.set_ylabel("mean % of section", fontsize=9)
axR.set_title(f"Mean ± SD\nhot {mh:.1f}%  cold {mc:.1f}%", fontsize=8.5, fontweight='bold')
for sp in ['top','right']: axR.spines[sp].set_visible(False)

fig.suptitle("Point 2 — Gi* hot and cold fractions across samples",
             fontsize=10, fontweight='bold', y=1.03)
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_summary_P2_fractions.{fmt}', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print(f"  ✓ Fig_summary_P2_fractions.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7.3 — SUMMARY BARS: Point 3 (LISA HH/LL fractions)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: df_lisa (Cell 6.1). HH = senescent clusters (red), LL = cold clusters (blue).
# Aggregated uses MEDIAN (IQR) — quadrant fractions are bounded/skewed.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 7.3 — SUMMARY BARS: Point 3"); print("="*64)

d = df_lisa.sort_values('f_HH', ascending=False).reset_index(drop=True)
HH_C, LL_C = '#cb181d', '#2b8cbe'

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 3.6),
                                gridspec_kw={'width_ratios':[3.2, 1]})

x = np.arange(len(d)); w = 0.4
axL.bar(x - w/2, d['f_HH']*100, w, color=HH_C, label='HH (senescent clusters)')
axL.bar(x + w/2, d['f_LL']*100, w, color=LL_C, label='LL (cold clusters)')
axL.set_xticks(x); axL.set_xticklabels(d['label'], rotation=45, ha='right', fontsize=7)
axL.set_ylabel("% of section", fontsize=9)
axL.set_title("Per sample: HH vs LL fraction", fontsize=9, fontweight='bold')
axL.legend(fontsize=7, frameon=False)
for sp in ['top','right']: axL.spines[sp].set_visible(False)

def med_iqr(s):
    s=s*100; return s.median(), s.quantile(.25), s.quantile(.75)
hh_m,hh_lo,hh_hi = med_iqr(d['f_HH']); ll_m,ll_lo,ll_hi = med_iqr(d['f_LL'])
axR.bar([0,1], [hh_m,ll_m], 0.6, color=[HH_C,LL_C],
        yerr=[[hh_m-hh_lo, ll_m-ll_lo],[hh_hi-hh_m, ll_hi-ll_m]], capsize=4, error_kw={'elinewidth':1})
axR.set_xticks([0,1]); axR.set_xticklabels(['HH','LL'], fontsize=8)
axR.set_ylabel("% of section", fontsize=9)
axR.set_title(f"Median (IQR)\nHH {hh_m:.1f}%  LL {ll_m:.1f}%", fontsize=8.5, fontweight='bold')
for sp in ['top','right']: axR.spines[sp].set_visible(False)

fig.suptitle(f"Point 3 — LISA HH and LL fractions (total HH = {int(df_lisa['n_HH'].sum())} spots)",
             fontsize=10, fontweight='bold', y=1.03)
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_summary_P3_lisa.{fmt}', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print(f"  ✓ Fig_summary_P3_lisa.pdf / .svg")

---
## 13 · Senescent pathways in hot versus cold

**Why.** Moves from "where" to "what". If Gi\* hot spots are genuinely
senescent neighbourhoods rather than compositional artifacts, the senescence
hallmark pathways should be elevated in them — and the elevation should not be
explained by which cell types are present.

**Display.** Pathway scores hot versus cold, with the composition of each group
reported alongside so the two readings stay attached.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.10 — MODULE SCORING (Sloan et al. senescence hallmark gene lists)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: ctrl (Cell 2), sample_adatas (Cell 3), lognorm layer.
# Scores 10 modules on lognorm via sc.tl.score_genes; writes Score_* into ctrl and
# propagates into sample_adatas. Same method as 10_spatial Cell 6.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.10 — MODULE SCORING"); print("="*64)

import scanpy as sc  

MARKERS_DIR = Path(SEN_REF_MARKERS)
SLOAN_FILE  = MARKERS_DIR / '1-s2.0-S2666979X25003830-mmc10.xlsx'

# ── score on lognorm layer (don't disturb ctrl.X permanently) ────────────────
ctrl_scoring = ctrl.copy()
if 'lognorm' in ctrl_scoring.layers:
    ctrl_scoring.X = ctrl_scoring.layers['lognorm'].copy()
    print(f"\n  Scoring on 'lognorm' layer: [{ctrl_scoring.X.min():.2f}, {ctrl_scoring.X.max():.2f}]")
else:
    print(f"  ⚠ no lognorm layer; using .X as-is")

# ── load Sloan gene lists ────────────────────────────────────────────────────
sloan_raw = pd.read_excel(SLOAN_FILE, sheet_name='Sen Gene Lists')
hallmark_names = ['p53_Targets','CellCycleArrest','SASP','AntiApoptosis','DDR',
                  'CellSurfaceMarkers','LysosomalContent','SD_TMC','SenMayo','Fridman_Up']

print(f"\n  Loading {len(hallmark_names)} gene lists from Sloan Table S9:")
gene_lists={}
for i,name in enumerate(hallmark_names):
    g=sloan_raw.iloc[1:,i].dropna().astype(str).str.strip()
    g=g[g!=''].tolist()
    detected=[x for x in g if x in ctrl_scoring.var_names]
    gene_lists[name]=detected
    print(f"    {name:<20s}: {len(detected):>3d}/{len(g):<3d} detected")

# ── score ────────────────────────────────────────────────────────────────────
print(f"\n  Scoring (sc.tl.score_genes)...")
for name,genes in gene_lists.items():
    if len(genes)<5:
        print(f"    {name:<20s}: SKIP (<5 genes)"); continue
    sc.tl.score_genes(ctrl_scoring, gene_list=genes, score_name=f'Score_{name}',
                      ctrl_size=min(100,len(genes)))
    v=ctrl_scoring.obs[f'Score_{name}']
    print(f"    {name:<20s}: mean={v.mean():.4f} sd={v.std():.4f}")

score_cols=[c for c in ctrl_scoring.obs.columns if c.startswith('Score_')]

# ── propagate Score_* back into ctrl and each sample_adatas (index-aligned) ───
for sc_col in score_cols:
    ctrl.obs[sc_col]=ctrl_scoring.obs[sc_col]
for sid in sample_adatas:
    ad=sample_adatas[sid]
    for sc_col in score_cols:
        ad.obs[sc_col]=ctrl.obs.loc[ad.obs_names, sc_col].values

module_cols=[(f'Score_{n}',n) for n in hallmark_names if f'Score_{n}' in score_cols]
print(f"\n  ✓ {len(score_cols)} modules scored, propagated to ctrl + {len(sample_adatas)} sample objects.")
print(f"  module_cols ready: {[m for _,m in module_cols]}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.11 — SENESCENT PATHWAYS in HOT vs COLD spots
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: 5.10 (Score_* in sample_adatas), gi_star_z, compartment.
# Per module: hot−cold mean score Δ, per sample → pooled + within-GM (Wilcoxon+FDR).
# Same anatomy-control logic as the composition test.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.11 — PATHWAYS in HOT vs COLD"); print("="*64)

HOT_Z, COLD_Z, MIN_CLASS = 1.96, -1.96, 15
module_names = [n for _, n in module_cols]
score_of = lambda m: f'Score_{m}'

def collect_mod(comp_filter):
    out = {m: [] for m in module_names}
    for sid in sorted(sample_adatas):
        ad = sample_adatas[sid]
        z = ad.obs['gi_star_z'].astype(float).values
        keep = np.ones(len(z), bool) if comp_filter is None else \
               (ad.obs['compartment'].astype(str).values == comp_filter)
        hot, cold = keep & (z > HOT_Z), keep & (z < COLD_Z)
        if hot.sum() < MIN_CLASS or cold.sum() < MIN_CLASS:
            continue
        for m in module_names:
            v = ad.obs[score_of(m)].astype(float).values
            out[m].append(v[hot].mean() - v[cold].mean())
    return {m: np.array(x) for m, x in out.items()}

pooled_m, gm_m = collect_mod(None), collect_mod('GM')
n_p = len(pooled_m[module_names[0]])
n_g = len(gm_m[module_names[0]])
print(f"\n  Samples — pooled:{n_p} within-GM:{n_g}")
print(f"  Test: Wilcoxon on per-sample Δ (hot−cold module score) + FDR across modules.\n")

# ── pooled + within-GM tests ─────────────────────────────────────────────────
rows = []
for m in module_names:
    dp = pooled_m[m]; dg = gm_m[m]
    try:
        _, p_p = wilcoxon(dp) if (len(dp) >= 2 and np.any(dp != 0)) else (np.nan, np.nan)
    except Exception:
        p_p = np.nan
    try:
        _, p_g = wilcoxon(dg) if (len(dg) >= 2 and np.any(dg != 0)) else (np.nan, np.nan)
    except Exception:
        p_g = np.nan
    rows.append({'module': m,
                 'pooled_d': dp.mean(), 'pooled_p': p_p,
                 'pooled_npos': int((dp > 0).sum()), 'pooled_n': len(dp),
                 'gm_d': dg.mean(), 'gm_p': p_g,
                 'gm_npos': int((dg > 0).sum()), 'gm_n': len(dg)})
df_m = pd.DataFrame(rows)

vp = ~df_m['pooled_p'].isna()
df_m.loc[vp, 'pooled_fdr'] = multipletests(df_m.loc[vp, 'pooled_p'], method='fdr_bh')[1]
vg = ~df_m['gm_p'].isna()
df_m.loc[vg, 'gm_fdr'] = multipletests(df_m.loc[vg, 'gm_p'], method='fdr_bh')[1]
df_m = df_m.sort_values('pooled_d', ascending=False).reset_index(drop=True)

# ── results table (all f-string nesting pulled out) ──────────────────────────
print(f"  {'Module':<18s} {'pooledΔ':>8s} {'p_dir':>7s} {'p_FDR':>7s} | {'gmΔ':>8s} {'gm_dir':>7s} {'gm_FDR':>7s}")
print("  " + "─"*68)
for _, r in df_m.iterrows():
    pf = r.get('pooled_fdr', np.nan)
    gf = r.get('gm_fdr', np.nan)
    ps = '★' if (not pd.isna(pf) and pf < 0.05) else ' '
    gs = '★' if (not pd.isna(gf) and gf < 0.05) else ' '
    pf_str = (f"{pf:.3f}" + ps) if not pd.isna(pf) else "—"
    gf_str = (f"{gf:.3f}" + gs) if not pd.isna(gf) else "—"
    pdir = f"{int(r['pooled_npos'])}/{int(r['pooled_n'])}"
    gdir = f"{int(r['gm_npos'])}/{int(r['gm_n'])}"
    print(f"  {r['module']:<18s} {r['pooled_d']:>+8.4f} {pdir:>7s} {pf_str:>7s} | "
          f"{r['gm_d']:>+8.4f} {gdir:>7s} {gf_str:>7s}")

df_m.to_csv(RESULTS_DIR/'pathways_hot_vs_cold.csv', index=False)
print(f"\n  ✓ saved: {RESULTS_DIR/'pathways_hot_vs_cold.csv'}")
print(f"  ★ = FDR<0.05.  pooledΔ>0 = pathway higher in hot spots.")
print(f"  Read: which pathways are elevated in hot (pooled), and which survive within GM (anatomy control).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.12 — TABLE-FOREST: pathways hot vs cold (pooled)
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.12 — TABLE-FOREST: pathways (pooled)"); print("="*64)

RED, BLUE = '#C0392B', '#3D7C9A'

def stats(arr):
    a=np.asarray(arr,float); n=len(a)
    if n<2: return a.mean(),np.nan,np.nan,np.nan
    se=a.std(ddof=1)/np.sqrt(n)
    return a.mean(), a.mean()-1.96*se, a.mean()+1.96*se, se

rows=[]
for _,r in df_m.iterrows():
    m=r['module']; d,lo,hi,se=stats(pooled_m[m])
    rows.append({'module':m,'d':d,'lo':lo,'hi':hi,'se':se,
                 'p':r['pooled_p'],'fdr':r.get('pooled_fdr',np.nan),
                 'npos':int(r['pooled_npos']),'n':int(r['pooled_n'])})
S=pd.DataFrame(rows)
n_rows=len(S); nval=int(S['n'].iloc[0])

# wider figure + forest occupies only the LEFT ~45%, columns get the rest with clear gaps
fig,ax=plt.subplots(figsize=(12.5, 0.6*n_rows+1.5))
xspan=np.nanmax(np.abs(np.r_[S['lo'].values,S['hi'].values]))*1.2
ax.set_xlim(-xspan,xspan)

# column x-positions in axes-fraction — spread out with real gaps
COLX={'d':1.10, 'ci':1.42, 'dir':1.72, 'p':1.92, 'fdr':2.10}
trans=ax.get_yaxis_transform()

yhead=n_rows-0.15
for key,lab in [('d',"Δ (SE)"),('ci',"[95% CI]"),('dir',"dir"),('p',"p"),('fdr',"FDR")]:
    ax.text(COLX[key], yhead, lab, transform=trans, fontsize=8, fontweight='bold',
            ha='center', va='bottom')
# header underline
ax.plot([COLX['d']-0.06, COLX['fdr']+0.06],[yhead-0.15]*2, transform=trans,
        color='#ccc', lw=0.6, clip_on=False)

for i,(_,r) in enumerate(S.iterrows()):
    y=n_rows-1-i
    if i%2==0: ax.axhspan(y-0.5,y+0.5,color='#f4f4f4',zorder=0)
    color=RED if r['d']>0 else BLUE
    if not np.isnan(r['lo']):
        ax.plot([r['lo'],r['hi']],[y,y],color=color,lw=1.5,solid_capstyle='round',zorder=2)
    ax.plot(r['d'],y,marker='D',color=color,ms=6.5,mec='black',mew=0.3,zorder=4)

    sig_lbl = r['fdr']<0.05 if not pd.isna(r['fdr']) else False
    lbl=('★ ' if sig_lbl else '')+r['module']
    ax.text(-0.03, y, lbl, transform=trans, fontsize=8.5,
            fontweight=('bold' if sig_lbl else 'normal'),
            color=(RED if sig_lbl else '#222'), ha='right', va='center')

    ax.text(COLX['d'], y, f"{r['d']:+.3f}", transform=trans, fontsize=7.5, ha='center', va='center', fontweight='bold')
    ax.text(COLX['d'], y-0.32, f"({r['se']:.3f})", transform=trans, fontsize=6, ha='center', va='center', color='#888')
    ci=f"[{r['lo']:+.3f}, {r['hi']:+.3f}]" if not np.isnan(r['lo']) else "—"
    ax.text(COLX['ci'], y, ci, transform=trans, fontsize=7, ha='center', va='center', color='#333')
    ax.text(COLX['dir'],y, f"{r['npos']}/{r['n']}", transform=trans, fontsize=7, ha='center', va='center', color='#333')
    pstr=f"{r['p']:.3f}" if not pd.isna(r['p']) else "—"
    fstr=f"{r['fdr']:.3f}" if not pd.isna(r['fdr']) else "—"
    ax.text(COLX['p'], y, pstr, transform=trans, fontsize=7, ha='center', va='center',
            fontweight=('bold' if (not pd.isna(r['p']) and r['p']<0.05) else 'normal'), color='#222')
    ax.text(COLX['fdr'],y, fstr, transform=trans, fontsize=7, ha='center', va='center',
            fontweight=('bold' if sig_lbl else 'normal'), color=(RED if sig_lbl else '#222'))

ax.axvline(0,color='#999',lw=0.6,ls='--',zorder=1)
ax.set_ylim(-0.7,n_rows+0.3)
ax.set_yticks([])
ax.set_xlabel("Δ module score: hot − cold",fontsize=8)
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
ax.tick_params(labelsize=7)
ax.set_title(f"Senescence pathways in hot vs cold spots  (pooled, n={nval})",
             fontsize=11, fontweight='bold', pad=12, loc='left')
ax.text(0.0,-0.10,"◆ red = higher in HOT · blue = higher in COLD · ★ FDR<0.05",
        transform=ax.transAxes, fontsize=7, color='#555')

plt.subplots_adjust(right=0.50)   # forest on left half, columns get the right half
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_pathways_forest.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_pathways_forest.pdf / .svg")

---
## 14 · Composition of senescent spots

**Why.** The inverse framing. Rather than asking what is in a spatially hot
region, ask what is in the spots called senescent — aggregated, then per sample,
then across all four groupings (SnC · all spots · HH · LL) so the definitions
can be compared directly.

Per-sample matters: an aggregated composition can be dominated by whichever
sample contributed the most spots.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.13 — COMPOSITION: SnC spots vs all spots (aggregated, mean proportion)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: sample_adatas with is_senescent + c2l_*, celltypes/c2l_cols.
# Per-sample mean composition → averaged across samples (per-sample-first, not pooled).
# SnC = is_senescent; gate ≥15 SnC spots per sample to contribute.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.13 — COMPOSITION: SnC vs all (aggregated)"); print("="*64)

c2l_cols  = [c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes = [c.replace('c2l_','') for c in c2l_cols]
MIN_SNC = 15

snc_rows, all_rows, n_used = [], [], 0
for sid in sorted(sample_adatas):
    ad = sample_adatas[sid]
    snc = ad.obs['is_senescent'].astype(bool).values
    if snc.sum() < MIN_SNC:
        continue
    n_used += 1
    snc_rows.append(ad.obs.loc[snc, c2l_cols].mean().values)   # mean comp of SnC spots
    all_rows.append(ad.obs[c2l_cols].mean().values)            # mean comp of all spots

snc_mean = np.mean(snc_rows, 0)*100
all_mean = np.mean(all_rows, 0)*100
snc_un, all_un = 100-snc_mean.sum(), 100-all_mean.sum()

print(f"\n  Samples contributing (≥{MIN_SNC} SnC spots): {n_used}/{len(sample_adatas)}")
print(f"  Column sums: SnC={snc_mean.sum():.1f}  All={all_mean.sum():.1f}  (Unassigned SnC={snc_un:.1f} All={all_un:.1f})")
print(f"\n  {'CellType':<16s} {'SnC %':>7s} {'All %':>7s} {'Δ':>7s}")
for i in np.argsort(-(snc_mean-all_mean)):
    print(f"  {celltypes[i]:<16s} {snc_mean[i]:>6.1f} {all_mean[i]:>6.1f} {snc_mean[i]-all_mean[i]:>+7.1f}")
pd.DataFrame({'celltype':celltypes+['Unassigned'],
             'snc_pct':list(snc_mean)+[snc_un],'all_pct':list(all_mean)+[all_un]}
            ).to_csv(RESULTS_DIR/'snc_vs_all_composition.csv',index=False)

# ── plot ──────────────────────────────────────────────────────────────────────
CT_COLORS={'Excitatory':'#2ca02c','Inhibitory':'#ff7f0e','Oligodendrocyte':'#9467bd',
           'Astrocyte':'#1f77b4','Microglia':'#8c564b','OPC':'#e377c2',
           'Vascular':'#17becf','Unassigned':'#e0e0e0'}
stack=['Oligodendrocyte','OPC','Astrocyte','Microglia','Vascular','Inhibitory','Excitatory','Unassigned']
idx={ct:i for i,ct in enumerate(celltypes)}

fig,ax=plt.subplots(figsize=(5.2,4.2))
for xp,(lab,vals,un) in zip([0,1],[('SnC spots',snc_mean,snc_un),('All spots',all_mean,all_un)]):
    bottom=0
    for ct in stack:
        w=un if ct=='Unassigned' else vals[idx[ct]]
        ax.bar(xp,w,bottom=bottom,width=0.55,color=CT_COLORS[ct],edgecolor='white',linewidth=0.5)
        if w>=4: ax.text(xp,bottom+w/2,f"{w:.0f}",ha='center',va='center',fontsize=8,
                         color='#666' if ct=='Unassigned' else 'white',fontweight='bold')
        bottom+=w
ax.set_xticks([0,1]); ax.set_xticklabels(['SnC\nspots','All\nspots'],fontsize=10)
ax.set_ylabel("Cell-type composition (%)",fontsize=9); ax.set_ylim(0,100); ax.set_xlim(-0.6,1.6)
ax.set_title(f"Composition: SnC spots vs all spots\n(aggregated, {n_used} samples, mean proportion)",
             fontsize=10,fontweight='bold')
for sp in ['top','right']: ax.spines[sp].set_visible(False)
ax.legend([plt.Rectangle((0,0),1,1,color=CT_COLORS[ct]) for ct in stack],stack,
          fontsize=7,frameon=False,ncol=2,loc='center left',bbox_to_anchor=(1.02,0.5))
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_snc_vs_all_composition.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"\n  ✓ Fig_snc_vs_all_composition.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.13b — COMPOSITION: SnC vs all spots, PER SAMPLE
# ═══════════════════════════════════════════════════════════════════════════════
# Thin-SnC (<15) dropped. SnC = hatched, All = solid. Oligo + Excitatory labeled
# (outlined text, size-guarded to avoid collisions).
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.13b — COMPOSITION: SnC vs all, per sample"); print("="*64)
import matplotlib.patheffects as pe

c2l_cols  = [c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes = [c.replace('c2l_','') for c in c2l_cols]
MIN_SNC = 15

recs=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; snc=ad.obs['is_senescent'].astype(bool).values
    if snc.sum() < MIN_SNC: continue
    recs.append({'label':SAMPLE_LABELS[sid],'pct_snc':100*snc.sum()/ad.n_obs,
                 'snc_comp':ad.obs.loc[snc,c2l_cols].mean().values*100,
                 'all_comp':ad.obs[c2l_cols].mean().values*100})
recs=sorted(recs,key=lambda r:-r['pct_snc'])
order=[r['label'] for r in recs]
print(f"\n  {len(recs)} samples shown; {len(sample_adatas)-len(recs)} dropped (thin SnC).")

CT_COLORS={'Excitatory':'#2ca02c','Inhibitory':'#ff7f0e','Oligodendrocyte':'#9467bd',
           'Astrocyte':'#1f77b4','Microglia':'#8c564b','OPC':'#e377c2',
           'Vascular':'#17becf','Unassigned':'#e0e0e0'}
stack=['Oligodendrocyte','OPC','Astrocyte','Microglia','Vascular','Inhibitory','Excitatory','Unassigned']
idx={ct:i for i,ct in enumerate(celltypes)}
LABEL_CTS=['Oligodendrocyte','Excitatory']
outline=[pe.withStroke(linewidth=1.4, foreground='black')]   # text outline for legibility

fig,ax=plt.subplots(figsize=(13.5,5.2))
w=0.40
for i,r in enumerate(recs):
    for off,vals,hatch in [(-w/2-0.03, r['snc_comp'], '////'),
                           ( w/2+0.03, r['all_comp'], None)]:
        xp=i+off; un=100-np.nansum(vals); bottom=0
        for ct in stack:
            v=un if ct=='Unassigned' else vals[idx[ct]]
            ax.bar(xp,v,bottom=bottom,width=w,color=CT_COLORS[ct],
                   edgecolor='white',linewidth=0.3,hatch=hatch)
            # label oligo+excit only if the segment is tall enough to hold text (≥6%)
            if ct in LABEL_CTS and v>=6:
                ax.text(xp, bottom+v/2, f"{v:.0f}", ha='center', va='center',
                        fontsize=6.5, fontweight='bold', color='white',
                        path_effects=outline, zorder=10, clip_on=True)
            bottom+=v
ax.set_xticks(range(len(order))); ax.set_xticklabels(order,rotation=90,fontsize=8)
ax.set_ylabel("Cell-type composition (%)",fontsize=9); ax.set_ylim(0,100); ax.set_xlim(-0.7,len(recs)-0.3)
ax.set_title("Composition per sample — SnC spots (hatched) vs all spots (solid)\nlabels = Oligodendrocyte & Excitatory %",
             fontsize=10,fontweight='bold')
for sp in ['top','right']: ax.spines[sp].set_visible(False)

ct_handles=[plt.Rectangle((0,0),1,1,color=CT_COLORS[ct]) for ct in stack]
hatch_handles=[plt.Rectangle((0,0),1,1,facecolor='#bbb',hatch='////',edgecolor='white'),
               plt.Rectangle((0,0),1,1,facecolor='#bbb',edgecolor='white')]
leg1=ax.legend(ct_handles,stack,fontsize=6.5,frameon=False,ncol=8,
               loc='upper center',bbox_to_anchor=(0.5,-0.16))
ax.add_artist(leg1)
ax.legend(hatch_handles,['SnC spots (hatched)','All spots (solid)'],fontsize=7,
          frameon=False,ncol=2,loc='upper center',bbox_to_anchor=(0.5,-0.25))
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_snc_vs_all_composition_persample.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_snc_vs_all_composition_persample.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.13c — LAYER COMPOSITION: SnC vs all spots, PER SAMPLE  (matches 5.13b style)
# ═══════════════════════════════════════════════════════════════════════════════
# Thin-SnC (<15) dropped. SnC = hatched, All = solid. WM layers labeled.
# Shares sample order with 5.13b via COMP_ORDER for bar-for-bar consistency.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.13c — LAYER COMPOSITION: SnC vs all, per sample"); print("="*64)
import matplotlib.patheffects as pe

LAYER_COL = 'annotation'
MIN_SNC   = 15

stack = ['L1','L2/3','L3/4','L3/4/5','L5/6','L6b','WM1','WM2','WM3','Unknown']
LAYER_COLORS = {'L1':'#FEE5D9','L2/3':'#FCBBA1','L3/4':'#FC9272','L3/4/5':'#FB6A4A',
                'L5/6':'#EF3B2C','L6b':'#CB181D','WM1':'#C6DBEF','WM2':'#6BAED6',
                'WM3':'#2171B5','Unknown':'#E0E0E0'}
LABEL_LAYERS = ['WM1','WM2','WM3']                  # label WM blocks (the senescence story)

def layer_props(o, mask):
    lay = o[LAYER_COL].astype(str).where(o[LAYER_COL].astype(str).isin(stack), 'Unknown')
    return (lay[mask].value_counts(normalize=True).reindex(stack).fillna(0)*100).values

recs=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; o=ad.obs
    if LAYER_COL not in o: continue
    snc=o['is_senescent'].astype(bool).values
    if snc.sum() < MIN_SNC: continue
    recs.append({'label':SAMPLE_LABELS[sid],'pct_snc':100*snc.sum()/ad.n_obs,
                 'snc_comp':layer_props(o, snc),
                 'all_comp':layer_props(o, np.ones(len(o),bool))})

# ── consistency: reuse CT plot's order if set, else sort here and define it ────
if 'COMP_ORDER' in globals():
    rec_by_label = {r['label']: r for r in recs}
    recs  = [rec_by_label[l] for l in COMP_ORDER if l in rec_by_label]
    order = [r['label'] for r in recs]
else:
    recs  = sorted(recs, key=lambda r:-r['pct_snc'])
    order = [r['label'] for r in recs]
    COMP_ORDER = order            # first composition plot to run defines the shared order
print(f"\n  {len(recs)} samples shown; {len(sample_adatas)-len(recs)} dropped (thin SnC).")

idx={lay:i for i,lay in enumerate(stack)}
outline=[pe.withStroke(linewidth=1.4, foreground='black')]

fig,ax=plt.subplots(figsize=(13.5,5.2))
w=0.40
for i,r in enumerate(recs):
    for off,vals,hatch in [(-w/2-0.03, r['snc_comp'], '////'),
                           ( w/2+0.03, r['all_comp'], None)]:
        xp=i+off; bottom=0
        for lay in stack:
            v=vals[idx[lay]]
            if v<=0:
                continue
            ax.bar(xp,v,bottom=bottom,width=w,color=LAYER_COLORS[lay],
                   edgecolor='white',linewidth=0.3,hatch=hatch)
            if lay in LABEL_LAYERS and v>=6:
                ax.text(xp, bottom+v/2, f"{v:.0f}", ha='center', va='center',
                        fontsize=6.5, fontweight='bold', color='white',
                        path_effects=outline, zorder=10, clip_on=True)
            bottom+=v

ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=0, ha='center', fontsize=8)
ax.set_ylabel("Layer composition (%)",fontsize=9); ax.set_ylim(0,100); ax.set_xlim(-0.7,len(recs)-0.3)
ax.set_title("Layer composition per sample — SnC spots (hatched) vs all spots (solid)\nlabels = white-matter layer %",
             fontsize=10,fontweight='bold')
for sp in ['top','right']: ax.spines[sp].set_visible(False)

# ── single legend block, bottom center ────────────────────────────────────────
layer_handles=[plt.Rectangle((0,0),1,1,color=LAYER_COLORS[lay]) for lay in stack]
hatch_handles=[plt.Rectangle((0,0),1,1,facecolor='#bbb',hatch='////',edgecolor='white'),
               plt.Rectangle((0,0),1,1,facecolor='#bbb',edgecolor='white')]
leg1=ax.legend(layer_handles, stack, fontsize=6.5, frameon=False, ncol=10,
               loc='upper center', bbox_to_anchor=(0.5,-0.12))
ax.add_artist(leg1)
ax.legend(hatch_handles, ['SnC spots (hatched)','All spots (solid)'], fontsize=7,
          frameon=False, ncol=2, loc='upper center', bbox_to_anchor=(0.5,-0.22))

plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_snc_vs_all_layers_persample.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_snc_vs_all_layers_persample.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.14 — COMPOSITION: SnC | All | HH | LL (aggregated, mean proportion)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: sample_adatas (is_senescent, lisa_quad, c2l_*), SAMPLE_LABELS.
# Per-sample mean composition per group → averaged across samples (per-sample-first).
# Gate ≥15 spots in a group for a sample to contribute to that group.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.14 — COMPOSITION: SnC | All | HH | LL"); print("="*64)
import matplotlib.patheffects as pe

c2l_cols  = [c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes = [c.replace('c2l_','') for c in c2l_cols]
MIN_CLASS = 15

# group masks per sample
def group_masks(ad):
    snc = ad.obs['is_senescent'].astype(bool).values
    q   = ad.obs['lisa_quad'].astype(str).values
    return {'SnC spots': snc, 'All spots': np.ones(ad.n_obs,bool),
            'HH\n(clusters)': q=='HH', 'LL\n(cold)': q=='LL'}

groups=['SnC spots','All spots','HH\n(clusters)','LL\n(cold)']
acc={g:[] for g in groups}; ncontrib={g:0 for g in groups}
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; masks=group_masks(ad)
    for g in groups:
        m=masks[g]
        if m.sum()>=MIN_CLASS:
            acc[g].append(ad.obs.loc[m,c2l_cols].mean().values*100); ncontrib[g]+=1

means={g:(np.mean(acc[g],0) if acc[g] else np.full(len(celltypes),np.nan)) for g in groups}

print(f"\n  Samples contributing per group (gate ≥{MIN_CLASS}):")
for g in groups: print(f"    {g.replace(chr(10),' '):<16s}: {ncontrib[g]}/{len(sample_adatas)}")
print(f"\n  {'CellType':<16s} " + " ".join(f"{g.split(chr(10))[0]:>8s}" for g in groups))
for ct in celltypes:
    print(f"  {ct:<16s} " + " ".join(f"{means[g][idx]:>8.1f}" if not np.isnan(means[g][idx]) else f"{'—':>8s}"
          for g,idx in [(g,celltypes.index(ct)) for g in groups]))
pd.DataFrame({**{'celltype':celltypes},**{g.split(chr(10))[0]:means[g] for g in groups}}
            ).to_csv(RESULTS_DIR/'composition_snc_all_hh_ll.csv',index=False)

# ── plot ──────────────────────────────────────────────────────────────────────
CT_COLORS = {
    "Excitatory":"#0072B2", "Inhibitory":"#E69F00", "Astrocyte":"#009E73",
    "Oligodendrocyte":"#56B4E9", "Microglia":"#D55E00", "OPC":"#CC79A7",
    "Vascular":"#7F7F7F", "Unassigned":"#E6E6E6",
}
stack=['Oligodendrocyte','OPC','Astrocyte','Microglia','Vascular','Inhibitory','Excitatory','Unassigned']
cidx={ct:i for i,ct in enumerate(celltypes)}
LABEL=['Oligodendrocyte','Excitatory']
outline=[pe.withStroke(linewidth=1.3,foreground='black')]

fig,ax=plt.subplots(figsize=(7,4.6))
for xp,g in enumerate(groups):
    vals=means[g]
    if np.all(np.isnan(vals)): continue
    un=100-np.nansum(vals); bottom=0
    for ct in stack:
        v=un if ct=='Unassigned' else vals[cidx[ct]]
        ax.bar(xp,v,bottom=bottom,width=0.6,color=CT_COLORS[ct],edgecolor='white',linewidth=0.5)
        if ct in LABEL and v>=5:
            ax.text(xp,bottom+v/2,f"{v:.0f}",ha='center',va='center',fontsize=8,
                    color='white',fontweight='bold',path_effects=outline)
        bottom+=v
ax.set_xticks(range(len(groups)))
ax.set_xticklabels([f"{g}\n(n={ncontrib[g]})" for g in groups],fontsize=8.5)
ax.set_ylabel("Cell-type composition (%)",fontsize=9); ax.set_ylim(0,100); ax.set_xlim(-0.6,len(groups)-0.4)
ax.axvline(1.5,color='#ccc',ls='--',lw=0.8)   # divider: spots | clusters
ax.set_title("Composition: senescent spots vs spatial clusters\nSnC | all | HH (hot clusters) | LL (cold clusters)",
             fontsize=10,fontweight='bold')
for sp in ['top','right']: ax.spines[sp].set_visible(False)
ax.legend([plt.Rectangle((0,0),1,1,color=CT_COLORS[ct]) for ct in stack],stack,
          fontsize=7,frameon=False,ncol=2,loc='center left',bbox_to_anchor=(1.02,0.5))
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_composition_snc_all_hh_ll.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"\n  ✓ Fig_composition_snc_all_hh_ll.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.14b — COMPOSITION: SnC | All | HH | LL, PER SAMPLE (polished)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: sample_adatas (is_senescent, lisa_quad, c2l_*), SAMPLE_LABELS.
# Hatch = group. Sample DROPPED if any of the 4 groups has <15 spots.
# Tighter bars, horizontal labels, larger fonts, values table before plot.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.14b — COMPOSITION 4 groups per sample"); print("="*64)
import matplotlib.patheffects as pe

c2l_cols  = [c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes = [c.replace('c2l_','') for c in c2l_cols]
MIN_CLASS = 15
grp_order = ['SnC','All','HH','LL']
HATCH = {'SnC':None, 'All':'...', 'HH':'///', 'LL':'\\\\\\'}

def grp_comp(ad):
    snc=ad.obs['is_senescent'].astype(bool).values
    q=ad.obs['lisa_quad'].astype(str).values
    masks={'SnC':snc,'All':np.ones(ad.n_obs,bool),'HH':q=='HH','LL':q=='LL'}
    counts={g:int(m.sum()) for g,m in masks.items()}
    if any(c<MIN_CLASS for c in counts.values()): return None, counts
    return {g: ad.obs.loc[m,c2l_cols].mean().values*100 for g,m in masks.items()}, counts

recs=[]; dropped=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; comp,counts=grp_comp(ad)
    snc=ad.obs['is_senescent'].astype(bool).values
    if comp is None: dropped.append((SAMPLE_LABELS[sid],counts)); continue
    recs.append({'label':SAMPLE_LABELS[sid],'pct_snc':100*snc.sum()/ad.n_obs,'comp':comp})
recs=sorted(recs,key=lambda r:-r['pct_snc'])
order=[r['label'] for r in recs]; n=len(recs)

print(f"\n  {n} samples shown; {len(dropped)} dropped (group <{MIN_CLASS}):")
for lab,cts in dropped:
    print(f"    {lab}: " + ", ".join(f"{g}={cts[g]}" for g in grp_order if cts[g]<MIN_CLASS))

# ── values table (oligo | excit per group) ───────────────────────────────────
oi,ei=celltypes.index('Oligodendrocyte'),celltypes.index('Excitatory')
print(f"\n  Oligodendrocyte % | Excitatory %  by group")
print(f"  {'Sample':<8s} | " + " | ".join(f"{g:^13s}" for g in grp_order))
print("  "+"─"*64)
for r in recs:
    print(f"  {r['label']:<8s} | " + " | ".join(f"{r['comp'][g][oi]:>5.1f} {r['comp'][g][ei]:>5.1f}  " for g in grp_order))
print("  "+"─"*64)
agg={g:np.mean([r['comp'][g] for r in recs],0) for g in grp_order}
print(f"  {'MEAN':<8s} | " + " | ".join(f"{agg[g][oi]:>5.1f} {agg[g][ei]:>5.1f}  " for g in grp_order))

# ── plot ──────────────────────────────────────────────────────────────────────
CT_COLORS = {
    "Excitatory":"#0072B2", "Inhibitory":"#E69F00", "Astrocyte":"#009E73",
    "Oligodendrocyte":"#56B4E9", "Microglia":"#D55E00", "OPC":"#CC79A7",
    "Vascular":"#7F7F7F", "Unassigned":"#E6E6E6",
}
stack=['Oligodendrocyte','OPC','Astrocyte','Microglia','Vascular','Inhibitory','Excitatory','Unassigned']
cidx={ct:i for i,ct in enumerate(celltypes)}
LABEL=['Oligodendrocyte','Excitatory']
outline=[pe.withStroke(linewidth=1.1,foreground='black')]

fig,ax=plt.subplots(figsize=(13,5.4))
bw=0.21; gap=0.16; block=4*bw+gap          # tighter: small gap between samples
for si,r in enumerate(recs):
    for gi,g in enumerate(grp_order):
        xp=si*block+gi*bw; vals=r['comp'][g]; un=100-np.nansum(vals); bottom=0
        for ct in stack:
            v=un if ct=='Unassigned' else vals[cidx[ct]]
            ax.bar(xp,v,bottom=bottom,width=bw*0.95,color=CT_COLORS[ct],
                   edgecolor='white',linewidth=0.3,hatch=HATCH[g])
            if ct in LABEL and v>=6:
                ax.text(xp,bottom+v/2,f"{v:.0f}",ha='center',va='center',fontsize=7,color='white',
                        fontweight='bold',path_effects=outline)
            bottom+=v
centers=[si*block+1.5*bw for si in range(n)]
ax.set_xticks(centers); ax.set_xticklabels(order,rotation=0,fontsize=10)   # horizontal labels
ax.set_xlim(-0.3,n*block-gap+0.05)
ax.set_ylabel("Cell-type composition (%)",fontsize=12); ax.set_ylim(0,100)
ax.tick_params(axis='y',labelsize=10)
ax.set_title("Composition per sample:  SnC (solid) · All (dots) · HH (///) · LL (\\\\\\)",
             fontsize=13,fontweight='bold',pad=10)
for sp in ['top','right']: ax.spines[sp].set_visible(False)

ct_h=[plt.Rectangle((0,0),1,1,color=CT_COLORS[ct]) for ct in stack]
grp_h=[plt.Rectangle((0,0),1,1,facecolor='#bbb',edgecolor='white',hatch=HATCH[g]) for g in grp_order]
l1=ax.legend(ct_h,stack,fontsize=9,frameon=False,ncol=8,loc='upper center',bbox_to_anchor=(0.5,-0.09)); ax.add_artist(l1)
ax.legend(grp_h,['SnC (solid)','All (dots)','HH (///)','LL (\\\\\\)'],
          fontsize=9,frameon=False,ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.16))
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_composition_4groups_persample.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_composition_4groups_persample.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.14d — HORIZONTAL 4-COLUMN COMPOSITION: SnC | All | HH | LL (aggregated)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: means dict (Cell 5.14), celltypes. Okabe-Ito palette.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.14d — HORIZONTAL 4-COLUMN COMPOSITION"); print("="*64)
import matplotlib.patheffects as pe

groups_src=['SnC spots','All spots','HH\n(clusters)','LL\n(cold)']
disp={'SnC spots':'SnC\nspots','All spots':'All\nspots','HH\n(clusters)':'HH\nclusters','LL\n(cold)':'LL\nclusters'}

OI={'Oligodendrocyte':'#0072B2','OPC':'#56B4E9','Astrocyte':'#009E73','Microglia':'#8C6D31',
    'Vascular':'#7F7F7F','Inhibitory':'#E69F00','Excitatory':'#D55E00','Unassigned':'#E6E6E6'}
stack=['Oligodendrocyte','OPC','Astrocyte','Microglia','Vascular','Inhibitory','Excitatory','Unassigned']
cidx={ct:i for i,ct in enumerate(celltypes)}
LABEL=['Oligodendrocyte','Excitatory']
outline=[pe.withStroke(linewidth=1.3,foreground='white')]

plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans']})
fig,axes=plt.subplots(1,4,figsize=(13,2.7),sharex=True)
for ax,gsrc in zip(axes,groups_src):
    v=means[gsrc]; un=100-np.nansum(v); left=0
    for ct in stack:
        w=un if ct=='Unassigned' else v[cidx[ct]]
        ax.barh(0,w,left=left,height=0.6,color=OI[ct],edgecolor='white',linewidth=0.8)
        if ct in LABEL and w>=5:
            ax.text(left+w/2,0,f"{w:.0f}",ha='center',va='center',fontsize=10,
                    color='white',fontweight='bold',path_effects=outline)
        left+=w
    ax.set_title(disp[gsrc],fontsize=11,fontweight='bold',pad=8)
    ax.set_xlim(0,100); ax.set_ylim(-0.5,0.5); ax.set_yticks([])
    ax.set_xlabel("composition (%)",fontsize=8); ax.tick_params(labelsize=7)
    for sp in ['top','right','left']: ax.spines[sp].set_visible(False)

# spots / clusters group labels above columns 1-2 and 3-4
axes[0].annotate('SPOTS', xy=(0.5,1.28), xytext=(1.0,1.28), xycoords='axes fraction',
                 ha='center', fontsize=9.5, color='#666', style='italic', fontweight='bold', annotation_clip=False)
axes[2].annotate('CLUSTERS', xy=(0.5,1.28), xytext=(1.0,1.28), xycoords='axes fraction',
                 ha='center', fontsize=9.5, color='#666', style='italic', fontweight='bold', annotation_clip=False)

fig.legend([plt.Rectangle((0,0),1,1,color=OI[ct]) for ct in stack],stack,
           fontsize=8.5,frameon=False,ncol=8,loc='lower center',bbox_to_anchor=(0.5,-0.28))
fig.suptitle("Cell-type composition across groups (aggregated, mean proportion)",
             fontsize=11.5,fontweight='bold',y=1.18)
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_composition_4groups_horizontal.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_composition_4groups_horizontal.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5.14e — PER-SAMPLE GRID: composition by group (single-axes, tight)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: sample_adatas (is_senescent, lisa_quad, c2l_*), celltypes, SAMPLE_LABELS.
# Group blocks offset on x, samples as y-rows. Sample dropped if any group <15.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 5.14e — PER-SAMPLE GRID (single-axes)"); print("="*64)
import matplotlib.patheffects as pe

c2l_cols=[c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes=[c.replace('c2l_','') for c in c2l_cols]
MIN_CLASS=15; groups=['SnC','All','HH','LL']

def grp_comp(ad):
    snc=ad.obs['is_senescent'].astype(bool).values; q=ad.obs['lisa_quad'].astype(str).values
    masks={'SnC':snc,'All':np.ones(ad.n_obs,bool),'HH':q=='HH','LL':q=='LL'}
    if any(m.sum()<MIN_CLASS for m in masks.values()): return None
    return {g: ad.obs.loc[m,c2l_cols].mean().values*100 for g,m in masks.items()}

recs=[]; dropped=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; comp=grp_comp(ad); snc=ad.obs['is_senescent'].astype(bool).values
    if comp is None: dropped.append(SAMPLE_LABELS[sid]); continue
    recs.append({'label':SAMPLE_LABELS[sid],'pct_snc':100*snc.sum()/ad.n_obs,'comp':comp})
recs=sorted(recs,key=lambda r:-r['pct_snc']); order=[r['label'] for r in recs]; n=len(recs)
print(f"\n  {n} samples; {len(dropped)} dropped (group <{MIN_CLASS}): {dropped}")

OI={'Oligodendrocyte':'#0072B2','OPC':'#56B4E9','Astrocyte':'#009E73','Microglia':'#8C6D31',
    'Vascular':'#7F7F7F','Inhibitory':'#E69F00','Excitatory':'#D55E00','Unassigned':'#E6E6E6'}
stack=['Oligodendrocyte','OPC','Astrocyte','Microglia','Vascular','Inhibitory','Excitatory','Unassigned']
cidx={ct:i for i,ct in enumerate(celltypes)}
LABEL=['Oligodendrocyte','Excitatory']; outline=[pe.withStroke(linewidth=0.9,foreground='white')]

plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans']})
BLOCKW=100; GAP=22; bh=0.74
fig,ax=plt.subplots(figsize=(13,0.40*n+1.1))
for gi,g in enumerate(groups):
    x_off=gi*(BLOCKW+GAP)
    for ri,r in enumerate(recs):
        y=n-1-ri; vals=r['comp'][g]; un=100-np.nansum(vals); left=x_off
        for ct in stack:
            w=un if ct=='Unassigned' else vals[cidx[ct]]
            ax.barh(y,w,left=left,height=bh,color=OI[ct],edgecolor='white',linewidth=0.3)
            if ct in LABEL and w>=10:
                ax.text(left+w/2,y,f"{w:.0f}",ha='center',va='center',fontsize=5.5,
                        color='white',fontweight='bold',path_effects=outline)
            left+=w
    ax.text(x_off+BLOCKW/2, n-0.15, g, ha='center', va='bottom', fontsize=11.5, fontweight='bold')
ax.set_yticks(range(n)); ax.set_yticklabels(order[::-1], fontsize=9)
ax.set_xticks([gi*(BLOCKW+GAP)+x for gi in range(4) for x in (0,50,100)])
ax.set_xticklabels(['0','50','100']*4, fontsize=6.5)
ax.set_xlim(-2, 4*BLOCKW+3*GAP+2); ax.set_ylim(-0.6,n+0.35)
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
ax.tick_params(left=False)
fig.legend([plt.Rectangle((0,0),1,1,color=OI[ct]) for ct in stack],stack,
           fontsize=8,frameon=False,ncol=8,loc='lower center',bbox_to_anchor=(0.5,-0.04))
fig.suptitle("Composition per sample by group  —  SnC | All | HH | LL",
             fontsize=11.5,fontweight='bold',y=1.0)
plt.tight_layout()
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_composition_4groups_grid.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print(f"  ✓ Fig_composition_4groups_grid.pdf / .svg")

---
## 15 · Proportion, burden and susceptibility in space

**Why.** The module 07 decomposition, computed on spots instead of nuclei.
`Burden = Proportion × Susceptibility` holds the same way, and having it in both
modalities is a cross-platform check on the decomposition itself — not just on
the individual numbers.

**Display.** The three quantities aggregated, per sample, and split by WM / GM.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL — PROPORTION · BURDEN · SUSCEPTIBILITY (3-panel, tables + figure)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: sample_adatas (is_senescent, c2l_*), celltypes, SAMPLE_LABELS.
# Dominant CT = argmax. Tables printed for all 3 stratifications, then figure.
# Panel C thin bars (<MIN_CELLS dominant spots) faded.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("PROPORTION · BURDEN · SUSCEPTIBILITY"); print("="*64)

c2l_cols=[c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes=[c.replace('c2l_','') for c in c2l_cols]
MIN_CELLS=20

CT_COLORS={'Oligodendrocyte':'#0072B2','OPC':'#56B4E9','Astrocyte':'#009E73','Microglia':'#8C6D31',
           'Vascular':'#7F7F7F','Inhibitory':'#E69F00','Excitatory':'#D55E00'}
active_cts=[c for c in CT_COLORS if c in celltypes]

# ── compute per sample × CT ──────────────────────────────────────────────────
rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; N=ad.n_obs
    snc=ad.obs['is_senescent'].astype(bool).values
    dom=np.array(celltypes)[ad.obs[c2l_cols].values.argmax(1)]
    for ct in active_cts:
        m=dom==ct; nt=int(m.sum())
        rows.append({'label':SAMPLE_LABELS[sid],'pct_snc':100*snc.sum()/N,'Cell_Type':ct,'n_dom':nt,
                     'CellProp':100*nt/N,
                     'SenBurden':100*int(snc[m].sum())/N,
                     'SenFrac':(100*snc[m].sum()/nt) if nt>=MIN_CELLS else np.nan})
df=pd.DataFrame(rows)
ord_samples=(df[['label','pct_snc']].drop_duplicates().sort_values('pct_snc',ascending=False)['label'].tolist())

def piv(col): return df.pivot(index='label',columns='Cell_Type',values=col).reindex(ord_samples)[active_cts]
prop_m=piv('CellProp').fillna(0); burden_m=piv('SenBurden').fillna(0); senf_m=piv('SenFrac'); ndom_m=piv('n_dom').fillna(0)
prop_norm=prop_m.div(prop_m.sum(axis=1),axis=0)*100

# ── TABLES ───────────────────────────────────────────────────────────────────
def show_table(mat, title, fmt="{:>6.1f}", note=""):
    print(f"\n  {title}")
    print(f"  {'Sample':<8s} " + "".join(f"{c[:6]:>7s}" for c in active_cts))
    print("  "+"─"*(9+7*len(active_cts)))
    for s in ord_samples:
        print(f"  {s:<8s} " + "".join((fmt.format(mat.loc[s,c]) if not pd.isna(mat.loc[s,c]) else f"{'·':>6s}") for c in active_cts))
    if note: print(f"  {note}")

show_table(prop_norm, "A · CELL-TYPE PROPORTION (%, normalized to 100)")
show_table(burden_m,  "B · SnC BURDEN (% of all spots; row sum = sample %SnC)")
print(f"    row sums: " + ", ".join(f"{s}={burden_m.loc[s].sum():.1f}" for s in ord_samples[:4]) + " ...")
show_table(senf_m,    "C · SnC SUSCEPTIBILITY (% senescent within type; '·' = <{} spots)".format(MIN_CELLS))
show_table(ndom_m,    "    dominant-spot counts (n per type)", fmt="{:>6.0f}")

for nm,mat in [('proportion',prop_norm),('burden',burden_m),('susceptibility',senf_m),('ndom',ndom_m)]:
    mat.to_csv(RESULTS_DIR/f'strat_{nm}_per_sample.csv')
print(f"\n  ✓ saved 4 CSVs (proportion/burden/susceptibility/ndom)")

# ── FIGURE ───────────────────────────────────────────────────────────────────
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'],'axes.linewidth':0.6})
n=len(ord_samples); y=np.arange(n)
fig,(axA,axB,axC)=plt.subplots(1,3,figsize=(12.5,max(3.2,n*0.42+1.4)),
                               gridspec_kw={'width_ratios':[1,1,1.1],'wspace':0.12})

def stackp(ax,mat,letter,title,xlabel,norm):
    left=np.zeros(n)
    for c in active_cts:
        v=mat[c].values
        ax.barh(y,v,left=left,color=CT_COLORS[c],edgecolor='white',linewidth=0.5,height=0.7,zorder=2)
        left+=v
    ax.set_yticks(y); ax.invert_yaxis()
    ax.set_xlabel(xlabel,fontsize=8.5,color='#333')
    ax.set_title(f"{letter}   {title}",fontsize=10.5,fontweight='bold',loc='left',pad=8)
    if norm: ax.set_xlim(0,100); ax.set_xticks([0,50,100])
    else:
        ax.set_xlim(0,left.max()*1.15)
        for yi,tot in zip(y,mat.sum(axis=1).values):
            ax.text(tot+left.max()*0.02,yi,f"{tot:.0f}",va='center',ha='left',fontsize=6.5,color='#777')
    ax.tick_params(labelsize=7.5,length=2,color='#999')
    for s in ['top','right']: ax.spines[s].set_visible(False)
    ax.spines['left'].set_color('#999'); ax.spines['bottom'].set_color('#999')
    ax.set_axisbelow(True); ax.xaxis.grid(True,color='#eee',lw=0.6,zorder=0)

stackp(axA,prop_norm,'a',"Cell-type proportion","% of spots",True); axA.set_yticklabels(ord_samples,fontsize=8)
stackp(axB,burden_m,'b',"Senescent burden","% of all spots",False); axB.set_yticklabels([])

# C grouped, fade thin bars
nct=len(active_cts); bh=0.72/nct
axC.set_axisbelow(True); axC.xaxis.grid(True,color='#eee',lw=0.6)
for ci,c in enumerate(active_cts):
    yy=y+(ci-(nct-1)/2)*bh
    vals=senf_m[c].values; nsp=ndom_m[c].values
    for k in range(n):
        v=vals[k]
        if pd.isna(v): continue
        thin = nsp[k]<MIN_CELLS
        axC.barh(yy[k],v,height=bh*0.9,color=CT_COLORS[c],
                 alpha=0.30 if thin else 1.0, hatch='///' if thin else None,
                 edgecolor='white' if thin else 'none', linewidth=0.3, zorder=2)
axC.set_yticks(y); axC.set_yticklabels([]); axC.invert_yaxis()
axC.set_xlabel("% senescent within type",fontsize=8.5,color='#333')
axC.set_title("c   Susceptibility",fontsize=10.5,fontweight='bold',loc='left',pad=8)
axC.tick_params(labelsize=7.5,length=2,color='#999')
for s in ['top','right']: axC.spines[s].set_visible(False)
axC.spines['left'].set_color('#999'); axC.spines['bottom'].set_color('#999')

handles=[plt.Rectangle((0,0),1,1,color=CT_COLORS[c]) for c in active_cts]
fig.legend(handles,active_cts,loc='lower center',ncol=len(active_cts),frameon=False,fontsize=8,
           bbox_to_anchor=(0.5,-0.02),columnspacing=1.2,handlelength=1.1)
fig.text(0.985,0.02,"faded = <%d spots"%MIN_CELLS,ha='right',fontsize=6,color='#999',style='italic')
fig.suptitle("Cell-type proportion · senescent burden · susceptibility (per sample)",fontsize=11,y=1.0)
fig.tight_layout(rect=[0,0.05,1,0.98])
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_prop_burden_susceptibility.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print("\n  ✓ Fig_prop_burden_susceptibility.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL — PROPORTION · BURDEN · SUSCEPTIBILITY (AGGREGATED, whole-section)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: df (per-sample×CT from the per-sample cell), active_cts, CT_COLORS, MIN_CELLS.
# Aggregates per-sample-first (mean across samples). One row, 3 panels.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("PROP · BURDEN · SUSCEPTIBILITY — AGGREGATED"); print("="*64)
import matplotlib.patheffects as pe

# aggregate per-sample metrics → mean across samples (NaN-skipped for susceptibility)
g = (df.groupby('Cell_Type',observed=True)
       .agg(CellProp=('CellProp','mean'),
            SenBurden=('SenBurden','mean'),
            SenFrac=('SenFrac','mean'),
            SenFrac_sd=('SenFrac','std'),
            n_rate=('SenFrac', lambda s: s.notna().sum()))
       .reindex(active_cts))
g['CellProp_norm'] = 100*g['CellProp']/g['CellProp'].sum()

# ── table ────────────────────────────────────────────────────────────────────
print(f"\n  {'CellType':<16s} {'Prop%':>7s} {'Burden%':>8s} {'Suscept%':>9s} {'±SD':>6s} {'n_samp':>7s}")
print("  "+"─"*58)
for ct in active_cts:
    sf=g.loc[ct,'SenFrac']; sd=g.loc[ct,'SenFrac_sd']; nr=int(g.loc[ct,'n_rate'])
    sf_s=f"{sf:.1f}" if not pd.isna(sf) else "·"; sd_s=f"{sd:.1f}" if not pd.isna(sd) else "·"
    print(f"  {ct:<16s} {g.loc[ct,'CellProp_norm']:>7.1f} {g.loc[ct,'SenBurden']:>8.2f} {sf_s:>9s} {sd_s:>6s} {nr:>7d}")
print(f"  burden sum (≈ mean %SnC): {g['SenBurden'].sum():.2f}%")
g[['CellProp_norm','SenBurden','SenFrac','SenFrac_sd','n_rate']].to_csv(RESULTS_DIR/'strat_aggregated_wholesection.csv')
print(f"  ✓ saved")

# ── figure: 1 row × 3 panels ─────────────────────────────────────────────────
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'],'axes.linewidth':0.6})
outline=[pe.withStroke(linewidth=1.3,foreground='white')]
fig,(axA,axB,axC)=plt.subplots(1,3,figsize=(12,3.0),gridspec_kw={'width_ratios':[0.7,0.9,1.05],'wspace':0.30})

# A proportion — single stacked bar
left=0
for ct in active_cts:
    w=g.loc[ct,'CellProp_norm']
    axA.barh(0,w,left=left,color=CT_COLORS[ct],edgecolor='white',linewidth=0.6,height=0.5)
    if ct in ['Oligodendrocyte','Excitatory'] and w>=6:
        axA.text(left+w/2,0,f"{w:.0f}",ha='center',va='center',fontsize=9,color='white',fontweight='bold',path_effects=outline)
    left+=w
axA.set_xlim(0,100); axA.set_ylim(-0.5,0.5); axA.set_yticks([]); axA.set_xticks([0,50,100])
axA.set_title("a   Cell-type proportion",fontsize=10.5,fontweight='bold',loc='left',pad=8)
axA.set_xlabel("% of spots",fontsize=8.5)
for s in ['top','right','left']: axA.spines[s].set_visible(False)
axA.tick_params(labelsize=8)

# B burden — single stacked bar (un-normalized)
left=0; tot=g['SenBurden'].sum()
for ct in active_cts:
    w=g.loc[ct,'SenBurden']
    axB.barh(0,w,left=left,color=CT_COLORS[ct],edgecolor='white',linewidth=0.6,height=0.5)
    left+=w
axB.text(tot*1.03,0,f"{tot:.1f}%",va='center',ha='left',fontsize=8.5,color='#555')
axB.set_xlim(0,tot*1.25); axB.set_ylim(-0.5,0.5); axB.set_yticks([])
axB.set_title("b   Senescent burden",fontsize=10.5,fontweight='bold',loc='left',pad=8)
axB.set_xlabel("% of all spots",fontsize=8.5)
for s in ['top','right','left']: axB.spines[s].set_visible(False)
axB.tick_params(labelsize=8)

# C susceptibility — one bar per cell type + SD whisker, fade thin
yy=np.arange(len(active_cts))[::-1]
for i,ct in enumerate(active_cts):
    sf=g.loc[ct,'SenFrac']; sd=g.loc[ct,'SenFrac_sd']; nr=int(g.loc[ct,'n_rate'])
    if pd.isna(sf):
        axC.text(0.05,yy[i],'· insufficient',va='center',fontsize=6.5,color='#bbb'); continue
    thin = nr<3
    axC.barh(yy[i],sf,height=0.62,color=CT_COLORS[ct],alpha=0.35 if thin else 1.0,
             hatch='///' if thin else None,edgecolor='white' if thin else 'none',linewidth=0.4,zorder=2)
    if not pd.isna(sd) and not thin:
        axC.plot([sf-sd,sf+sd],[yy[i],yy[i]],color='#555',lw=0.9,zorder=3)
    axC.text(sf+(sd if not pd.isna(sd) else 0)+0.08,yy[i],f"{sf:.1f}",va='center',ha='left',fontsize=7,color='#444')
axC.set_yticks(yy); axC.set_yticklabels(active_cts,fontsize=8)
axC.set_xlim(0, np.nanmax(g['SenFrac'].values+g['SenFrac_sd'].fillna(0).values)*1.2)
axC.set_title("c   Susceptibility",fontsize=10.5,fontweight='bold',loc='left',pad=8)
axC.set_xlabel("% senescent within type",fontsize=8.5)
for s in ['top','right']: axC.spines[s].set_visible(False)
axC.tick_params(labelsize=8); axC.set_axisbelow(True); axC.xaxis.grid(True,color='#eee',lw=0.6)

handles=[plt.Rectangle((0,0),1,1,color=CT_COLORS[c]) for c in active_cts]
fig.legend(handles,active_cts,loc='lower center',ncol=len(active_cts),frameon=False,fontsize=8,bbox_to_anchor=(0.5,-0.10))
fig.text(0.985,-0.02,"susceptibility: mean±SD across samples; faded = <3 samples",ha='right',fontsize=6,color='#999',style='italic')
fig.suptitle("Cell-type proportion · senescent burden · susceptibility (aggregated)",fontsize=11.5,fontweight='bold',y=1.04)
fig.tight_layout(rect=[0,0.06,1,0.96])
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_prop_burden_susceptibility_agg.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print("\n  ✓ Fig_prop_burden_susceptibility_agg.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL — PROP·BURDEN·SUSCEPTIBILITY, WM/GM per sample (SELF-CONTAINED compute+fig)
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("PROP·BURDEN·SUSCEPTIBILITY — WM vs GM, per sample"); print("="*64)

c2l_cols=[c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes=[c.replace('c2l_','') for c in c2l_cols]
MIN_CELLS=20
CT_COLORS={'Oligodendrocyte':'#0072B2','OPC':'#56B4E9','Astrocyte':'#009E73','Microglia':'#8C6D31',
           'Vascular':'#7F7F7F','Inhibitory':'#E69F00','Excitatory':'#D55E00'}
active_cts=[c for c in CT_COLORS if c in celltypes]

# ── compute per sample × CT × compartment ────────────────────────────────────
def metrics_for(ad, comp_mask):
    Ncomp=int(comp_mask.sum())
    if Ncomp==0: return {ct:(np.nan,np.nan,np.nan,0) for ct in active_cts}, 0
    snc=ad.obs['is_senescent'].astype(bool).values
    dom=np.array(celltypes)[ad.obs[c2l_cols].values.argmax(1)]
    out={}
    for ct in active_cts:
        m=comp_mask & (dom==ct); nt=int(m.sum())
        senf=(snc[m].sum()/nt) if nt>=MIN_CELLS else np.nan
        out[ct]=(100*nt/Ncomp, 100*int(snc[m].sum())/Ncomp,
                 (100*senf if not np.isnan(senf) else np.nan), nt)
    return out, Ncomp

rows={'WM':[],'GM':[]}
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; comp=ad.obs['compartment'].astype(str).values
    snc=ad.obs['is_senescent'].astype(bool).values
    for cp in ['WM','GM']:
        cm=comp==cp; met,Ncomp=metrics_for(ad,cm)
        for ct in active_cts:
            p,b,s,nd=met[ct]
            rows[cp].append({'label':SAMPLE_LABELS[sid],'Cell_Type':ct,
                             'CellProp':p,'SenBurden':b,'SenFrac':s,'n_dom':nd})
dfs={cp:pd.DataFrame(rows[cp]) for cp in ['WM','GM']}

# sample order by whole-section %SnC
ov=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; snc=ad.obs['is_senescent'].astype(bool).values
    ov.append((SAMPLE_LABELS[sid],100*snc.sum()/ad.n_obs))
ord_samples=[s for s,_ in sorted(ov,key=lambda t:-t[1])]
def piv(dfc,col): return dfc.pivot(index='label',columns='Cell_Type',values=col).reindex(ord_samples)[active_cts]

# ── FIGURE: 2 rows (WM,GM) × 3 cols ──────────────────────────────────────────
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'],'axes.linewidth':0.6})
n=len(ord_samples); y=np.arange(n)
fig,axes=plt.subplots(2,3,figsize=(13,max(8.4,n*0.42*2+1.8)),
                      gridspec_kw={'width_ratios':[1,1,1.1]}, layout='constrained')
fig.set_constrained_layout_pads(w_pad=0.04, h_pad=0.06, hspace=0.10, wspace=0.06)

def stackp(ax,mat,title,xlabel,norm,show_y):
    left=np.zeros(n)
    for c in active_cts:
        v=np.nan_to_num(mat[c].values); ax.barh(y,v,left=left,color=CT_COLORS[c],edgecolor='white',linewidth=0.5,height=0.72,zorder=2); left+=v
    ax.set_yticks(y); ax.invert_yaxis(); ax.set_yticklabels(ord_samples if show_y else [],fontsize=8)
    ax.set_xlabel(xlabel,fontsize=8,color='#333'); ax.set_title(title,fontsize=10,fontweight='bold',loc='left',pad=5)
    if norm: ax.set_xlim(0,100); ax.set_xticks([0,50,100])
    else:
        ax.set_xlim(0,max(left.max()*1.15,0.1))
        for yi,tot in zip(y,np.nan_to_num(mat.values).sum(1)):
            if tot>0: ax.text(tot+left.max()*0.02,yi,f"{tot:.0f}",va='center',ha='left',fontsize=6,color='#888')
    ax.tick_params(labelsize=7,length=2,color='#999')
    for s in ['top','right']: ax.spines[s].set_visible(False)
    ax.spines['left'].set_color('#999'); ax.spines['bottom'].set_color('#999')
    ax.set_axisbelow(True); ax.xaxis.grid(True,color='#eee',lw=0.6,zorder=0)

def grouped(ax,senf,ndom,title):
    nct=len(active_cts); bh=0.74/nct
    ax.set_axisbelow(True); ax.xaxis.grid(True,color='#eee',lw=0.6)
    for ci,c in enumerate(active_cts):
        yy=y+(ci-(nct-1)/2)*bh; vals=senf[c].values; nsp=ndom[c].values
        for k in range(n):
            if pd.isna(vals[k]): continue
            thin=nsp[k]<MIN_CELLS
            ax.barh(yy[k],vals[k],height=bh*0.9,color=CT_COLORS[c],
                    alpha=0.3 if thin else 1.0,hatch='///' if thin else None,
                    edgecolor='white' if thin else 'none',linewidth=0.3,zorder=2)
    ax.set_yticks(y); ax.set_yticklabels([]); ax.invert_yaxis()
    ax.set_xlabel("% senescent within type",fontsize=8,color='#333')
    ax.set_title(title,fontsize=10,fontweight='bold',loc='left',pad=5)
    ax.tick_params(labelsize=7,length=2,color='#999')
    for s in ['top','right']: ax.spines[s].set_visible(False)
    ax.spines['left'].set_color('#999'); ax.spines['bottom'].set_color('#999')

for ri,cp in enumerate(['WM','GM']):
    propn=piv(dfs[cp],'CellProp'); propn=propn.div(propn.sum(axis=1),axis=0)*100
    stackp(axes[ri,0],propn,f"{cp}  ·  proportion","% of spots",True,show_y=True)
    stackp(axes[ri,1],piv(dfs[cp],'SenBurden'),f"{cp}  ·  burden","% of spots",False,show_y=False)
    grouped(axes[ri,2],piv(dfs[cp],'SenFrac'),piv(dfs[cp],'n_dom'),f"{cp}  ·  susceptibility")

handles=[plt.Rectangle((0,0),1,1,color=CT_COLORS[c]) for c in active_cts]
fig.legend(handles,active_cts,loc='outside lower center',ncol=len(active_cts),frameon=False,fontsize=8)
fig.suptitle("Proportion · burden · susceptibility — white matter (top) vs gray matter (bottom)",
             fontsize=12,fontweight='bold')
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_prop_burden_susc_WMvsGM.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print("\n  ✓ Fig_prop_burden_susc_WMvsGM.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL — PROP·BURDEN·SUSCEPTIBILITY, WM vs GM, AGGREGATED
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: dfs (the per-sample WM/GM dataframes from the previous cell), active_cts,
#           CT_COLORS, RESULTS_DIR, FIGURES_DIR, MIN_CELLS.
# Aggregates per-sample metrics across samples (per-sample-first mean). 2×3 grid.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("PROP·BURDEN·SUSCEPTIBILITY — WM vs GM, AGGREGATED"); print("="*64)

# aggregate per-sample values → mean across samples (per-sample-first; ignore NaN susceptibility)
agg={}
for cp in ['WM','GM']:
    d=dfs[cp]
    g=d.groupby('Cell_Type',observed=True).agg(
        CellProp=('CellProp','mean'),
        SenBurden=('SenBurden','mean'),
        SenFrac=('SenFrac','mean'),               # mean of per-sample rates (NaN-skipped)
        n_samples_rate=('SenFrac',lambda s: s.notna().sum())
    ).reindex(active_cts)
    agg[cp]=g

# normalize proportion to 100%
for cp in ['WM','GM']:
    p=agg[cp]['CellProp']; agg[cp]['CellProp_norm']=100*p/p.sum()

# ── tables ───────────────────────────────────────────────────────────────────
for cp in ['WM','GM']:
    g=agg[cp]
    print(f"\n  ════ {cp} (aggregated across samples) ════")
    print(f"  {'CellType':<16s} {'Prop%':>7s} {'Burden%':>8s} {'Suscept%':>9s} {'n_samp':>7s}")
    print("  "+"─"*52)
    for ct in active_cts:
        sf=g.loc[ct,'SenFrac']; nr=int(g.loc[ct,'n_samples_rate'])
        sf_s=f"{sf:.1f}" if not pd.isna(sf) else "·"
        print(f"  {ct:<16s} {g.loc[ct,'CellProp_norm']:>7.1f} {g.loc[ct,'SenBurden']:>8.2f} {sf_s:>9s} {nr:>7d}")
    print(f"  burden sum (≈ mean %SnC in {cp}): {g['SenBurden'].sum():.2f}%")
    g[['CellProp_norm','SenBurden','SenFrac','n_samples_rate']].to_csv(RESULTS_DIR/f'strat_{cp}_aggregated.csv')
print(f"\n  ✓ saved 2 CSVs")

# ── figure: 2 rows (WM,GM) × 3 cols, single bars (aggregated) ────────────────
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'],'axes.linewidth':0.6})
import matplotlib.patheffects as pe
outline=[pe.withStroke(linewidth=1.3,foreground='white')]
fig,axes=plt.subplots(2,3,figsize=(12,5.4),gridspec_kw={'width_ratios':[0.7,1,1.05],'wspace':0.28,'hspace':0.35})

for ri,cp in enumerate(['WM','GM']):
    g=agg[cp]
    # A proportion — single stacked horizontal bar
    axA=axes[ri,0]; left=0
    for ct in active_cts:
        w=g.loc[ct,'CellProp_norm']
        axA.barh(0,w,left=left,color=CT_COLORS[ct],edgecolor='white',linewidth=0.6,height=0.55)
        if ct in ['Oligodendrocyte','Excitatory'] and w>=6:
            axA.text(left+w/2,0,f"{w:.0f}",ha='center',va='center',fontsize=9,color='white',fontweight='bold',path_effects=outline)
        left+=w
    axA.set_xlim(0,100); axA.set_ylim(-0.5,0.5); axA.set_yticks([]); axA.set_xticks([0,50,100])
    axA.set_title(f"{cp}  ·  proportion",fontsize=10,fontweight='bold',loc='left',pad=6)
    axA.set_xlabel("% of spots",fontsize=8)
    for s in ['top','right','left']: axA.spines[s].set_visible(False)

    # B burden — single stacked bar (un-normalized)
    axB=axes[ri,1]; left=0; tot=g['SenBurden'].sum()
    for ct in active_cts:
        w=g.loc[ct,'SenBurden']
        axB.barh(0,w,left=left,color=CT_COLORS[ct],edgecolor='white',linewidth=0.6,height=0.55)
        left+=w
    axB.text(tot*1.03,0,f"{tot:.1f}%",va='center',ha='left',fontsize=8,color='#555')
    axB.set_xlim(0,max(tot*1.2,0.5)); axB.set_ylim(-0.5,0.5); axB.set_yticks([])
    axB.set_title(f"{cp}  ·  burden",fontsize=10,fontweight='bold',loc='left',pad=6)
    axB.set_xlabel("% of spots senescent",fontsize=8)
    for s in ['top','right','left']: axB.spines[s].set_visible(False)
    axB.tick_params(labelsize=7)

    # C susceptibility — grouped (one bar per cell type), fade thin
    axC=axes[ri,2]; yy=np.arange(len(active_cts))[::-1]
    for i,ct in enumerate(active_cts):
        sf=g.loc[ct,'SenFrac']; nr=int(g.loc[ct,'n_samples_rate'])
        if pd.isna(sf): 
            axC.text(0.1,yy[i],'· insufficient',va='center',fontsize=6,color='#bbb'); continue
        thin = nr<3
        axC.barh(yy[i],sf,height=0.66,color=CT_COLORS[ct],alpha=0.35 if thin else 1.0,
                 hatch='///' if thin else None,edgecolor='white' if thin else 'none',linewidth=0.4)
        axC.text(sf+0.05,yy[i],f"{sf:.1f}",va='center',ha='left',fontsize=7,color='#444')
    axC.set_yticks(yy); axC.set_yticklabels(active_cts,fontsize=7.5)
    axC.set_xlim(0,max(4,np.nanmax(g['SenFrac'].values)*1.25))
    axC.set_title(f"{cp}  ·  susceptibility",fontsize=10,fontweight='bold',loc='left',pad=6)
    axC.set_xlabel("% senescent within type",fontsize=8)
    for s in ['top','right']: axC.spines[s].set_visible(False)
    axC.tick_params(labelsize=7)
    axC.set_axisbelow(True); axC.xaxis.grid(True,color='#eee',lw=0.6)

handles=[plt.Rectangle((0,0),1,1,color=CT_COLORS[c]) for c in active_cts]
fig.legend(handles,active_cts,loc='lower center',ncol=len(active_cts),frameon=False,fontsize=8,bbox_to_anchor=(0.5,-0.02))
fig.suptitle("Proportion · burden · susceptibility — white matter (top) vs gray matter (bottom), aggregated",
             fontsize=11.5,fontweight='bold',y=1.0)
fig.tight_layout(rect=[0,0.04,1,0.98])
for fmt in ['pdf','svg']:
    plt.savefig(FIGURES_DIR/f'Fig_prop_burden_susc_WMvsGM_agg.{fmt}',dpi=300,bbox_inches='tight',facecolor='white')
plt.show(); print("\n  ✓ Fig_prop_burden_susc_WMvsGM_agg.pdf / .svg")

---
## 16 · Per-sample senescence rates

**Why.** The denominators behind everything above. %SnC per sample, then the
SnC rate per cell type per sample — which is where a sample carrying the pooled
result becomes visible.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL — %SnC PER SAMPLE (table)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: sample_adatas (is_senescent), SAMPLE_LABELS.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("%SnC PER SAMPLE"); print("="*64)

rows=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; snc=ad.obs['is_senescent'].astype(bool).values
    rows.append({'label':SAMPLE_LABELS[sid],'n_total':ad.n_obs,
                 'n_snc':int(snc.sum()),'pct_snc':100*snc.sum()/ad.n_obs})
df_pctsnc=pd.DataFrame(rows).sort_values('pct_snc',ascending=False).reset_index(drop=True)

print(f"\n  {'Sample':<8s} {'nSpots':>7s} {'nSnC':>6s} {'%SnC':>7s}")
print("  "+"─"*32)
for _,r in df_pctsnc.iterrows():
    print(f"  {r['label']:<8s} {r['n_total']:>7d} {r['n_snc']:>6d} {r['pct_snc']:>6.1f}")
print("  "+"─"*32)

p=df_pctsnc['pct_snc']
print(f"  {'mean ± SD':<8s} {'':<7s} {'':<6s} {p.mean():>5.1f} ± {p.std(ddof=1):.1f}")
print(f"  {'median':<8s} {'':<7s} {'':<6s} {p.median():>6.1f} (IQR {p.quantile(.25):.1f}–{p.quantile(.75):.1f})")
print(f"  {'range':<8s} {'':<7s} {'':<6s} {p.min():.1f}–{p.max():.1f}")
print(f"  total SnC spots: {int(df_pctsnc['n_snc'].sum()):,} / {int(df_pctsnc['n_total'].sum()):,}")

df_pctsnc.to_csv(RESULTS_DIR/'pct_snc_per_sample.csv',index=False)
print(f"\n  ✓ saved: {RESULTS_DIR/'pct_snc_per_sample.csv'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL — SnC RATE per CELL TYPE per SAMPLE
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: sample_adatas (is_senescent, c2l_*), celltypes, SAMPLE_LABELS.
# Dominant CT = argmax c2l_. Rate = % of that CT's spots that are SnC. Gate ≥20 spots.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("SnC RATE per CELL TYPE per SAMPLE"); print("="*64)

c2l_cols  = [c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes = [c.replace('c2l_','') for c in c2l_cols]
MIN_CELLS = 20   # min dominant-type spots in a sample to report its rate

# sort samples by overall %SnC (high→low) for column order
sample_pct=[]
for sid in sorted(sample_adatas):
    ad=sample_adatas[sid]; snc=ad.obs['is_senescent'].astype(bool).values
    sample_pct.append((sid, SAMPLE_LABELS[sid], 100*snc.sum()/ad.n_obs))
sample_pct.sort(key=lambda t:-t[2])
sids=[t[0] for t in sample_pct]; labels=[t[1] for t in sample_pct]

# rate[ct][sid] = % SnC among dominant-ct spots; also store n
rate={ct:{} for ct in celltypes}; ncnt={ct:{} for ct in celltypes}
for sid in sids:
    ad=sample_adatas[sid]
    snc=ad.obs['is_senescent'].astype(bool).values
    dom=np.array(celltypes)[ad.obs[c2l_cols].values.argmax(1)]
    for ct in celltypes:
        m=dom==ct; n=int(m.sum()); ncnt[ct][sid]=n
        rate[ct][sid]= (100*snc[m].sum()/n) if n>=MIN_CELLS else np.nan

# order cell types by mean rate (desc)
ct_mean={ct:np.nanmean([rate[ct][s] for s in sids]) for ct in celltypes}
ct_order=sorted(celltypes, key=lambda c:-(ct_mean[c] if not np.isnan(ct_mean[c]) else -1))

# ── table ─────────────────────────────────────────────────────────────────────
print(f"\n  SnC rate (%) — rows=dominant cell type, cols=sample (sorted by overall %SnC)")
print(f"  cells with <{MIN_CELLS} dominant spots shown as '·'\n")
hdr = f"  {'CellType':<16s}" + "".join(f"{lab[:6]:>7s}" for lab in labels) + f" {'mean':>6s}"
print(hdr); print("  "+"─"*(len(hdr)-2))
for ct in ct_order:
    row=f"  {ct:<16s}"
    for sid in sids:
        v=rate[ct][sid]
        row += f"{v:>7.1f}" if not np.isnan(v) else f"{'·':>7s}"
    mv=ct_mean[ct]
    row += f" {mv:>6.1f}" if not np.isnan(mv) else f" {'·':>6s}"
    print(row)

# dominant-spot counts (so thin rows are visible)
print(f"\n  Dominant-spot counts (n per CT per sample):")
print("  "+"─"*(len(hdr)-2))
for ct in ct_order:
    row=f"  {ct:<16s}"
    for sid in sids: row += f"{ncnt[ct][sid]:>7d}"
    print(row)

# save tidy
tidy=[]
for ct in celltypes:
    for sid in sids:
        tidy.append({'celltype':ct,'sample':SAMPLE_LABELS[sid],
                     'n_dominant':ncnt[ct][sid],'snc_rate_pct':rate[ct][sid]})
pd.DataFrame(tidy).to_csv(RESULTS_DIR/'snc_rate_per_celltype_per_sample.csv',index=False)
print(f"\n  ✓ saved: {RESULTS_DIR/'snc_rate_per_celltype_per_sample.csv'}")
print(f"  Read: does any cell type have a HIGHER SnC rate? (oligo vs neuron especially)")
print(f"  Caveat: dominant assignment is lossy; rare types (·) are unreliable.")

---
## 17 · Susceptibility — non-circular check, and the compartment gate

**Why.** A susceptibility estimate computed from the same spots that defined the
senescence call is partly circular. This section recomputes it in a way that
breaks the loop.

**The WM/GM gate** then evaluates whether the compartment split is doing what
sections 09-11 assume — whether the two compartments are separable enough for a
within-compartment test to mean anything.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SUSCEPTIBILITY — non-circular check (global cutoff + continuous score)
# ═══════════════════════════════════════════════════════════════════════════════
# A: single GLOBAL mean+2SD cutoff across ALL spots → % of each CT above it (free to vary)
# B: mean continuous sen_score per CT (no threshold)
# Dominant CT = argmax c2l_. Pools all control spots.
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("SUSCEPTIBILITY — non-circular (global cut + continuous)"); print("="*64)

c2l_cols  = [c for c in ctrl.obs.columns if c.startswith('c2l_')]
celltypes = [c.replace('c2l_','') for c in c2l_cols]

# pooled across all control spots
score = ctrl.obs['sen_score'].astype(float).values
dom   = np.array(celltypes)[ctrl.obs[c2l_cols].values.argmax(1)]

# A — single global cutoff (one number for ALL spots)
gmean, gsd = score.mean(), score.std(ddof=1)
gcut = gmean + 2*gsd
above = score > gcut
print(f"\n  Global cutoff = mean+2SD = {gmean:.4f} + 2*{gsd:.4f} = {gcut:.4f}")
print(f"  Overall % above global cutoff: {100*above.mean():.2f}%  (n={above.sum()}/{len(score)})\n")

rows=[]
for ct in celltypes:
    m = dom==ct; n=int(m.sum())
    if n==0: continue
    rows.append({
        'CellType':ct,'n_spots':n,
        'pct_above_global':100*above[m].mean(),      # A: free to vary per CT
        'mean_score':score[m].mean(),                # B: continuous
        'median_score':np.median(score[m]),
    })
df_s=pd.DataFrame(rows).sort_values('mean_score',ascending=False).reset_index(drop=True)

print(f"  {'CellType':<16s} {'n':>7s} {'%>global':>9s} {'meanScore':>10s} {'medScore':>9s}")
print("  "+"─"*56)
for _,r in df_s.iterrows():
    print(f"  {r['CellType']:<16s} {r['n_spots']:>7d} {r['pct_above_global']:>8.2f}% "
          f"{r['mean_score']:>10.4f} {r['median_score']:>9.4f}")

print(f"\n  READ:")
print(f"   A (%>global): if neurons > oligo here, neurons genuinely exceed a")
print(f"      cell-type-agnostic cutoff more often → real susceptibility difference.")
print(f"   B (meanScore): if neurons > oligo here too, the bulk distribution is")
print(f"      shifted up in neurons → confirms it's not a tail/threshold artifact.")
print(f"   If A and B agree → finding is real. If only the old per-CT binary showed")
print(f"      it → it was circular.")
df_s.to_csv(RESULTS_DIR/'susceptibility_noncircular.csv',index=False)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — WM/GM GATE EVALUATION  (select samples with trustworthy labels)
# ═══════════════════════════════════════════════════════════════════════════════
# REQUIRES: Cell 2 (df_validation in memory).
# ═══════════════════════════════════════════════════════════════════════════════
print("="*64); print("CELL 3 — WM/GM GATE  (WM must exceed GM on every metric)"); print("="*64)

METRICS = markers_present + (['Oligo'] if OLIGO_COL in ctrl.obs else [])

rows = []
for _, r in df_validation.iterrows():
    deltas = {met: r.get(f'{met}_WM', np.nan) - r.get(f'{met}_GM', np.nan) for met in METRICS}
    evaluable = all(np.isfinite(d) for d in deltas.values())   # needs both WM & GM present
    rec = {'sample_id': r['sample_id']}
    rec.update({f'Δ{met}': deltas[met] for met in METRICS})
    rec['evaluable'] = evaluable
    rec['PASS'] = bool(evaluable and all(deltas[met] > 0 for met in METRICS))
    rows.append(rec)

df_gate = pd.DataFrame(rows).set_index('sample_id')

# ── display ──────────────────────────────────────────────────────────────────
disp = df_gate.drop(columns='evaluable').copy()
for met in METRICS:
    disp[f'Δ{met}'] = disp[f'Δ{met}'].map(lambda v: f'{v:+.3f}' if np.isfinite(v) else '  n/a')
disp['PASS'] = df_gate['PASS'].map({True: '✓', False: '✗'})
print("\n  Δ = mean(WM) − mean(GM).  Positive on every metric ⇒ PASS.\n")
print(disp.to_string())

# ── summary + selection ──────────────────────────────────────────────────────
passing  = df_gate.index[df_gate['PASS']].tolist()
failing  = df_gate.index[~df_gate['PASS'] &  df_gate['evaluable']].tolist()
noteval  = df_gate.index[~df_gate['evaluable']].tolist()

print(f"\n  {'─'*58}")
print(f"  GATE: {len(passing)}/{len(df_gate)} pass"
      + (f"   ({len(noteval)} not evaluable — missing WM or GM)" if noteval else ""))
if failing: print(f"  ✗ WM ≤ GM somewhere: {failing}")
if noteval: print(f"  ⚠ not evaluable:     {noteval}")
print(f"  ✓ pass: {passing}")

df_gate.to_csv(RESULTS_DIR / 'wmgm_gate.csv')
print(f"\n  ✓ saved: {RESULTS_DIR / 'wmgm_gate.csv'}")
print(f"  → `passing` = candidate pool for downstream subsetting / exemplar pick.")

In [ ]:
#!/usr/bin/env python
# ═══════════════════════════════════════════════════════════════════════════════
# EXEMPLAR SPATIAL MAPS — dominant cell type + cortical layers (+ compartment)
# Self-contained: runs standalone or inside the 10_spatial kernel.
# ═══════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ── CONFIG ─────────────────────────────────────────────────────────────────────
PROJECT_DIR = Path(f'{SEN_DATA}/spatial/'
                   '04_validation/morabito_dsad_control')
ADATA_PATH  = globals().get('ADATA_PATH', None)   # set to the controls .h5ad if not in memory
FIGURES_DIR = Path(globals().get('FIGURES_DIR', PROJECT_DIR / 'figures'))
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EXEMPLAR     = globals().get('EXEMPLAR_SID', 'Dec_20_2021_Human5')   # set from ranking
CONTROL_VAL  = 'Control'
SPOT         = 9
RARE_ON_TOP  = True          # draw abundant types first so rare ones stay visible

SID_COL   = 'sample_id'
GROUP_COL = 'group'
CT_COL    = 'dominant_celltype'
LAYER_COL = 'annotation'
COMP_COL  = 'compartment'

CT_COLORS = globals().get('ct_colors', {
    'Excitatory':'#2ca02c','Inhibitory':'#ff7f0e','Oligodendrocyte':'#9467bd',
    'Astrocyte':'#1f77b4','Microglia':'#8c564b','OPC':'#e377c2','Endothelial':'#d62728',
    'Pericyte':'#17becf','PVM':'#bcbd22','VLMC':'#f7b6d2','VSMC':'#7f7f7f','Adaptive':'#aec7e8'})
LAYER_ORDER  = globals().get('LAYER_ORDER',
    ['L1','L2/3','L3/4','L3/4/5','L5/6','L6b','WM1','WM2','WM3'])
LAYER_COLORS = globals().get('LAYER_COLORS', {
    'L1':'#fee5d9','L2/3':'#fcbba1','L3/4':'#fc9272','L3/4/5':'#fb6a4a','L5/6':'#ef3b2c',
    'L6b':'#cb181d','WM1':'#a6bddb','WM2':'#74a9cf','WM3':'#2b8cbe'})
COMP_COLORS  = {'GM':'#BBBBBB','WM':'#4477AA','Unknown':'#000000'}

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8,'figure.dpi':300,
                     'savefig.dpi':300,'pdf.fonttype':42})

# ── LOAD: in-memory first, else from disk ───────────────────────────────────────
def _resolve_exemplar(sid):
    if 'sample_adatas' in globals() and sid in sample_adatas:
        return sample_adatas[sid]
    if 'adata' in globals() and (adata.obs[SID_COL] == sid).any():
        return adata[adata.obs[SID_COL] == sid].copy()
    if ADATA_PATH is None:
        raise RuntimeError(
            "No in-memory adata/sample_adatas and ADATA_PATH is None.\n"
            "  → set ADATA_PATH to the controls .h5ad, or run in the kernel that built sample_adatas.")
    import scanpy as sc
    a = sc.read_h5ad(ADATA_PATH)
    if GROUP_COL in a.obs:
        a = a[a.obs[GROUP_COL] == CONTROL_VAL]
    if not (a.obs[SID_COL] == sid).any():
        raise KeyError(f"{sid!r} not in {ADATA_PATH} (controls). Available: "
                       f"{sorted(a.obs[SID_COL].unique())[:6]} ...")
    return a[a.obs[SID_COL] == sid].copy()

ex = _resolve_exemplar(EXEMPLAR)
assert 'spatial' in ex.obsm, "ex.obsm['spatial'] missing — no coordinates to plot"

# ── display label (no get_meta dependency) ──────────────────────────────────────
def _meta(sid):
    if 'get_meta' in globals():
        return get_meta(sid)
    if 'SAMPLE_REGISTRY' in globals() and sid in SAMPLE_REGISTRY:
        r = SAMPLE_REGISTRY[sid]; return r.get('name', sid), r.get('dataset', '')
    o = ex.obs
    sex = str(o['sex'].iloc[0]) if 'sex' in o else ''
    age = str(o['age'].iloc[0]) if 'age'  in o else ''
    grp = str(o[GROUP_COL].iloc[0]) if GROUP_COL in o else CONTROL_VAL
    return (f"{sex}-{age}".strip('-') or sid), grp

name, group = _meta(EXEMPLAR)
print(f"Exemplar: {EXEMPLAR}  →  {name} ({group}),  {ex.n_obs:,} spots")

# ── backfill dominant_celltype from c2l_* if absent ─────────────────────────────
if CT_COL not in ex.obs:
    c2l = [c for c in ex.obs.columns if c.startswith('c2l_')]
    assert c2l, f"no '{CT_COL}' and no c2l_* columns to derive it from"
    ex.obs[CT_COL] = ex.obs[c2l].idxmax(axis=1).str.replace('c2l_', '', regex=False)
    print(f"  derived {CT_COL} from {len(c2l)} c2l_* columns")

# ── reusable single-panel plotter ───────────────────────────────────────────────
def plot_label_map(col, color_map, order, title, fname):
    if col not in ex.obs:
        print(f"  ⚠ '{col}' not in obs — skipping {title}"); return
    coords = ex.obsm['spatial']
    labels = ex.obs[col].astype(str).values
    valid  = ~np.isnan(coords).any(axis=1)
    order  = [l for l in order if (labels == l).any()] or sorted(set(labels[valid]))
    if RARE_ON_TOP:                                  # abundant first, rare last
        order = sorted(order, key=lambda l: -(labels == l).sum())

    fig, ax = plt.subplots(figsize=(3.4, 3.4))
    for lab in order:
        m = (labels == lab) & valid
        if m.sum():
            ax.scatter(coords[m,0], coords[m,1], c=color_map.get(lab,'#999'),
                       s=SPOT, alpha=0.9, edgecolors='none', rasterized=True)
    ax.invert_yaxis(); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_linewidth(0.4); sp.set_edgecolor('#444')
    ax.set_title(f"{name} — {title}", fontsize=9, fontweight='bold', pad=4)
    leg = [l for l in (order if col != CT_COL else list(color_map)) if (labels == l).any()]
    handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=color_map.get(l,'#999'),
                      markersize=5, markeredgewidth=0, label=l) for l in leg]
    ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.01,0.5),
              frameon=False, fontsize=6.5, labelspacing=0.7, handletextpad=0.3)
    for ext in ('pdf','svg'):
        fig.savefig(FIGURES_DIR / f'{fname}_{EXEMPLAR}.{ext}',
                    dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print("  " + "  ".join(f"{l}:{int((labels==l).sum())}" for l in leg))
    print(f"  ✓ {fname}_{EXEMPLAR}.pdf / .svg")

# ── render ──────────────────────────────────────────────────────────────────────
plot_label_map(CT_COL,    CT_COLORS,    list(CT_COLORS), 'dominant cell type', 'exemplar_celltype')
plot_label_map(LAYER_COL, LAYER_COLORS, LAYER_ORDER,     'cortical layers',    'exemplar_layers')
plot_label_map(COMP_COL,  COMP_COLORS,  ['GM','WM'],     'compartment',        'exemplar_compartment')

In [ ]:
#!/usr/bin/env python
# ═══════════════════════════════════════════════════════════════════════════════
# PANEL A — EXEMPLAR sen_score MAP + Gi* NICHE OUTLINES (single section)
# Self-contained: reuses pipeline gi_star_z / niche_id if present, else computes them.
# ═══════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from scipy.spatial import ConvexHull
from sklearn.neighbors import NearestNeighbors

# ── CONFIG ─────────────────────────────────────────────────────────────────────
FIGURES_DIR   = Path(globals().get('FIGURES_DIR',
                  f'{SEN_DATA}/spatial/'
                  '04_validation/morabito_dsad_control/figures'))
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEN_COL    = 'sen_score'
K          = 6           # Visium first ring — match pipeline
HOT_Z      = 1.96        # Gi* hot-spot threshold
MIN_NICHE  = 3           # min spots per niche (connected components)
SPOT       = 10
CMAP       = 'YlOrRd'     # high sen_score = dark red (salient), low = pale (recedes)
CLIP       = (2, 98)     # percentile clip for color scale
OUTLINE    = '#225ea8'   # niche boundary colour
SID_COL, GROUP_COL = 'sample_id', 'group'

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8,
                     'figure.dpi':300,'savefig.dpi':300,'pdf.fonttype':42})

# ── resolve exemplar (in-memory first, else from disk via ADATA_PATH) ───────────
def _resolve(sid):
    if 'sample_adatas' in globals() and sid in sample_adatas: return sample_adatas[sid]
    if 'ex' in globals() and getattr(globals()['ex'], 'n_obs', 0) and \
       (globals()['ex'].obs[SID_COL] == sid).all(): return globals()['ex']
    if 'adata' in globals() and (adata.obs[SID_COL] == sid).any():
        return adata[adata.obs[SID_COL] == sid].copy()
    ap = globals().get('ADATA_PATH')
    if ap is None: raise RuntimeError("No in-memory adata and ADATA_PATH unset.")
    import scanpy as sc
    a = sc.read_h5ad(ap)
    if GROUP_COL in a.obs: a = a[a.obs[GROUP_COL] == 'Control']
    return a[a.obs[SID_COL] == sid].copy()

ex = _resolve(EXEMPLAR)
assert SEN_COL in ex.obs, f"'{SEN_COL}' not in obs"
coords = np.asarray(ex.obsm['spatial'], float)
sen    = np.asarray(ex.obs[SEN_COL], float)
n      = ex.n_obs

def _meta(sid):
    if 'get_meta' in globals(): return get_meta(sid)
    o = ex.obs
    sex = str(o['sex'].iloc[0]) if 'sex' in o else ''
    age = str(o['age'].iloc[0]) if 'age' in o else ''
    return (f"{sex}-{age}".strip('-') or sid), (str(o[GROUP_COL].iloc[0]) if GROUP_COL in o else '')
name, group = _meta(EXEMPLAR)

# ── k=6 binary graph (reuse pipeline connectivities if present) ─────────────────
if 'spatial_connectivities' in ex.obsp:
    W = (ex.obsp['spatial_connectivities'] > 0).astype(float).tocsr()
else:
    nn = NearestNeighbors(n_neighbors=K + 1).fit(coords)        # +1: drops self
    W  = nn.kneighbors_graph(coords, mode='connectivity').tolil()
    W.setdiag(0); W = W.tocsr(); W.eliminate_zeros()

# ── Gi* z (reuse if present, else analytical Getis-Ord incl. focal spot) ────────
if 'gi_star_z' in ex.obs:
    giz = np.asarray(ex.obs['gi_star_z'], float)
    print("  using pipeline gi_star_z")
else:
    Ws = (W + sparse.eye(n, format='csr'))                      # Gi* includes i
    xbar, S = sen.mean(), sen.std()
    Wsum = np.asarray(Ws.sum(1)).ravel()
    W2   = np.asarray(Ws.multiply(Ws).sum(1)).ravel()
    lag  = np.asarray(Ws @ sen).ravel()
    denom = S * np.sqrt((n * W2 - Wsum**2) / (n - 1))
    giz   = (lag - xbar * Wsum) / np.where(denom == 0, np.nan, denom)
    print("  computed analytical Gi* (regenerate from 10b-4 for paper-identical values)")

# ── niches: connected components of hot spots, size ≥ MIN_NICHE ─────────────────
if 'niche_id' in ex.obs:
    niche = np.asarray(ex.obs['niche_id'], float)
    print("  using pipeline niche_id")
else:
    hot = giz > HOT_Z
    niche = np.full(n, -1)
    if hot.sum():
        idx = np.where(hot)[0]
        sub = W[idx][:, idx]
        ncomp, lab = connected_components(sub, directed=False)
        nid = 0
        for c in range(ncomp):
            members = idx[lab == c]
            if members.size >= MIN_NICHE:
                niche[members] = nid; nid += 1
    print(f"  segmented {int(np.nanmax(niche)) + 1 if np.any(niche >= 0) else 0} niches")

# ── plot: sen_score heat + niche hull outlines ──────────────────────────────────
vmin, vmax = np.nanpercentile(sen, CLIP)
fig, ax = plt.subplots(figsize=(3.8, 3.4))
sc_ = ax.scatter(coords[:,0], coords[:,1], c=sen, cmap=CMAP, vmin=vmin, vmax=vmax,
                 s=SPOT, edgecolors='none', rasterized=True)
for nid in np.unique(niche[niche >= 0]).astype(int):
    pts = coords[niche == nid]
    if len(pts) >= 3:
        try:
            h = ConvexHull(pts); poly = np.append(h.vertices, h.vertices[0])
            ax.plot(pts[poly,0], pts[poly,1], color=OUTLINE, lw=1.1, alpha=0.9)
        except Exception:
            pass
ax.invert_yaxis(); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
for sp in ax.spines.values(): sp.set_linewidth(0.4); sp.set_edgecolor('#444')
n_niche = int((np.unique(niche[niche >= 0])).size)
ax.set_title(f"{name} — senescence score + niches (n={n_niche})",
             fontsize=9, fontweight='bold', pad=4)
cb = fig.colorbar(sc_, ax=ax, fraction=0.046, pad=0.02)
cb.set_label('sen_score', fontsize=7); cb.ax.tick_params(labelsize=6)
cb.outline.set_linewidth(0.4)
for ext in ('pdf','svg'):
    fig.savefig(FIGURES_DIR / f'exemplar_panelA_senscore_niches_{EXEMPLAR}.{ext}',
                dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"  spots in niches: {int((niche>=0).sum())}/{n} ({(niche>=0).mean()*100:.1f}%)")
print(f"  ✓ exemplar_panelA_senscore_niches_{EXEMPLAR}.pdf / .svg")

In [ ]:
#!/usr/bin/env python
# ═══════════════════════════════════════════════════════════════════════════════
# LISA SPATIAL MAP — Dec_20_2021_Human5 : tissue colored by LISA quadrant
# Reuses obs LISA labels if present; else computes local Moran on sen_score.
# ═══════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ── CONFIG ───────────────────────────────────────────────────────────────────
SID         = 'Dec_20_2021_Human5'
FIGURES_DIR = Path(globals().get('FIGURES_DIR',
                  f'{SEN_DATA}/spatial/'
                  '04_validation/morabito_dsad_control/figures')); FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SEN_COL, SID_COL, GROUP_COL = 'sen_score', 'sample_id', 'group'
K, ALPHA, N_PERM, SEED, SPOT = 6, 0.05, 999, 0, 10
LISA_COLORS = {'HH':'#cb181d','LL':'#2b8cbe','HL':'#fb9a99','LH':'#a6cee3','NS':'#E2E2E2'}
ORDER = ['NS','HL','LH','LL','HH']                 # NS first (under), HH on top

plt.rcParams.update({'font.family':'sans-serif','axes.linewidth':0.6,
    'pdf.fonttype':42,'ps.fonttype':42,'figure.dpi':300})

# ── resolve section ───────────────────────────────────────────────────────────
def _resolve(sid):
    if 'sample_adatas' in globals() and sid in sample_adatas: return sample_adatas[sid].copy()
    if 'adata' in globals() and (adata.obs[SID_COL]==sid).any(): return adata[adata.obs[SID_COL]==sid].copy()
    ap = globals().get('ADATA_PATH')
    if ap is None: raise RuntimeError("No in-memory adata and ADATA_PATH unset.")
    import scanpy as sc
    a = sc.read_h5ad(ap)
    if GROUP_COL in a.obs: a = a[a.obs[GROUP_COL]=='Control']
    return a[a.obs[SID_COL]==sid].copy()

ex = _resolve(SID); n = ex.n_obs
coords = np.asarray(ex.obsm['spatial'], float)
print(f"{SID}: {n:,} spots")

# ── LISA quadrant (reuse if present, else local Moran on sen_score) ───────────
lisa_cols = [c for c in ('lisa_quadrant','lisa_label','lisa_class') if c in ex.obs]
if lisa_cols:
    quad = ex.obs[lisa_cols[0]].astype(str).values
    print(f"  using pipeline LISA labels: {lisa_cols[0]}")
else:
    from sklearn.neighbors import NearestNeighbors
    if 'spatial_connectivities' in ex.obsp:
        W = (ex.obsp['spatial_connectivities']>0).astype(float).tocsr()
    else:
        nn = NearestNeighbors(n_neighbors=K+1).fit(coords)
        W = nn.kneighbors_graph(mode='connectivity').tolil(); W.setdiag(0); W = W.tocsr()
    Wr = W.multiply(1/np.asarray(W.sum(1)).clip(1)).tocsr()         # row-normalized
    x  = np.asarray(ex.obs[SEN_COL],float); z = (x-x.mean())/x.std()
    lag = np.asarray(Wr@z).ravel(); Ii = z*lag
    rng = np.random.default_rng(SEED); null = np.empty((N_PERM,n))
    for i in range(N_PERM):
        null[i] = z*np.asarray(Wr@z[rng.permutation(n)]).ravel()
    p = (np.sum(np.abs(null)>=np.abs(Ii),axis=0)+1)/(N_PERM+1)
    quad = np.where(z>0, np.where(lag>0,'HH','HL'), np.where(lag>0,'LH','LL'))
    quad[p>=ALPHA] = 'NS'
    print(f"  computed LISA (k={K}, {N_PERM} perms): "
          + ", ".join(f"{q}:{int((quad==q).sum())}" for q in ORDER))

# ── spatial plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(3.8, 3.4))
for q in ORDER:
    m = quad==q
    if m.sum():
        ax.scatter(coords[m,0], coords[m,1],
                   s=SPOT if q!='NS' else SPOT*0.8, c=LISA_COLORS[q],
                   alpha=0.95 if q!='NS' else 0.6, edgecolors='none',
                   rasterized=True, zorder=3 if q=='HH' else (2 if q!='NS' else 1))
ax.invert_yaxis(); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
for sp in ax.spines.values(): sp.set_linewidth(0.4); sp.set_edgecolor('#444')
ax.set_title(f'{SID} — LISA senescence clusters', fontsize=8, fontweight='bold', pad=4)
present = [q for q in ['HH','LL','HL','LH','NS'] if (quad==q).any()]
labelmap = {'HH':'High–High','LL':'Low–Low','HL':'High–Low','LH':'Low–High','NS':'n.s.'}
ax.legend(handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor=LISA_COLORS[q],
                          markersize=5,label=f'{labelmap[q]} ({int((quad==q).sum())})') for q in present],
          loc='center left', bbox_to_anchor=(1.01,0.5), frameon=False, fontsize=6.5, labelspacing=0.7)
for ext in ('pdf','svg'):
    fig.savefig(FIGURES_DIR/f'lisa_spatial_{SID}.{ext}', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print(f"  ✓ lisa_spatial_{SID}.pdf / .svg")

In [ ]:
#!/usr/bin/env python
# ═══════════════════════════════════════════════════════════════════════════════
# PANELS B + C — self-contained: compute niche fraction + join-count z, then plot
#   No saved-file dependency, no shared `df` (uses df_bc). Anchors placed outside frame.
#   b: %SnC-in-niche + N heatmap strip   c: clustering (join-count z), anchored
# ═══════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap, Normalize
from scipy import sparse, stats
from scipy.sparse.csgraph import connected_components
from sklearn.neighbors import NearestNeighbors

PROJECT_DIR = Path(f'{SEN_DATA}/spatial/'
                   '04_validation/morabito_dsad_control')
FIGURES_DIR = Path(globals().get('FIGURES_DIR', PROJECT_DIR/'figures')); FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ADATA_PATH  = globals().get('ADATA_PATH', None)

SID_COL, GROUP_COL, SNC_COL, SEN_COL = 'sample_id', 'group', 'is_senescent', 'sen_score'
K, HOT_Z, MIN_NICHE, N_PERM, MIN_SNC, SEED = 6, 1.96, 3, 1999, 15, 0
SWEEP_Z, SWEEP_MIN = [1.64, 1.96, 2.33], [3, 5]
USE_PASSING = False     # all controls by default

# ── controls ──────────────────────────────────────────────────────────────────
def _controls():
    if 'adata' in globals() and (adata.obs.get(GROUP_COL)=='Control').any():
        return adata[adata.obs[GROUP_COL]=='Control']
    if ADATA_PATH is None: raise RuntimeError("No in-memory controls and ADATA_PATH unset.")
    import scanpy as sc
    a = sc.read_h5ad(ADATA_PATH); return a[a.obs[GROUP_COL]=='Control'] if GROUP_COL in a.obs else a
ad_all  = _controls()
all_ctrl = sorted(ad_all.obs[SID_COL].unique())
samples  = (globals().get('passing') or all_ctrl) if USE_PASSING else all_ctrl
print(f"controls: {len(all_ctrl)} available | processing {len(samples)}")

def _meta(sid,o):
    if 'get_meta' in globals(): return get_meta(sid)
    sex=str(o['sex'].iloc[0]) if 'sex' in o else ''; age=str(o['age'].iloc[0]) if 'age' in o else ''
    return (f"{sex}-{age}".strip('-') or sid),'Control'

# ── per-sample helpers ────────────────────────────────────────────────────────
def graph_k6(sub):
    if 'spatial_connectivities' in sub.obsp:
        return (sub.obsp['spatial_connectivities']>0).astype(float).tocsr()
    nn=NearestNeighbors(n_neighbors=K+1).fit(np.asarray(sub.obsm['spatial'],float))
    W=nn.kneighbors_graph(mode='connectivity').tolil(); W.setdiag(0); return W.tocsr()

def gi_star(sub,W):
    if 'gi_star_z' in sub.obs: return np.asarray(sub.obs['gi_star_z'],float)
    x=np.asarray(sub.obs[SEN_COL],float); n=sub.n_obs
    Ws=(W+sparse.eye(n,format='csr')); Wsum=np.asarray(Ws.sum(1)).ravel()
    W2=np.asarray(Ws.multiply(Ws).sum(1)).ravel(); lag=np.asarray(Ws@x).ravel()
    den=x.std()*np.sqrt((n*W2-Wsum**2)/(n-1))
    return (lag-x.mean()*Wsum)/np.where(den==0,np.nan,den)

def niche_frac(W,giz,snc_mask,hz,mn):
    n=giz.shape[0]; nid=np.full(n,-1); hot=giz>hz
    if hot.sum()>=mn:
        idx=np.where(hot)[0]; nc,lab=connected_components(W[idx][:,idx],directed=False); k=0
        for c in range(nc):
            mem=idx[lab==c]
            if mem.size>=mn: nid[mem]=k; k+=1
    nnich=int(nid.max()+1) if nid.max()>=0 else 0
    frac=float((nid[snc_mask]>=0).mean()) if snc_mask.sum() else np.nan
    return frac,nnich

def join_count(W,snc,rng):
    Wu=sparse.triu(W,k=1); r,c=Wu.nonzero(); obs=float(np.sum(snc[r]*snc[c]))
    n,m=snc.shape[0],int(snc.sum()); null=np.empty(N_PERM)
    for i in range(N_PERM):
        f=np.zeros(n); f[rng.choice(n,m,replace=False)]=1.0; null[i]=np.sum(f[r]*f[c])
    sd=null.std(); z=(obs-null.mean())/sd if sd>0 else np.nan
    p=(np.sum(null>=obs)+1)/(N_PERM+1); return obs,z,p

# ── compute → df_bc (unique name, never clobbered) ────────────────────────────
rng=np.random.default_rng(SEED); rows=[]; sweep_rows=[]
for sid in samples:
    sub=ad_all[ad_all.obs[SID_COL]==sid]; o=sub.obs; nm,_=_meta(sid,o)
    if SNC_COL not in o: print(f"  ⚠ {nm}: no '{SNC_COL}' — skipped"); continue
    snc=(o[SNC_COL].values==1).astype(float); n_snc=int(snc.sum())
    if n_snc<MIN_SNC: print(f"  – {nm}: n_snc={n_snc}<{MIN_SNC} — gated out"); continue
    sm=snc.astype(bool); W=graph_k6(sub); giz=gi_star(sub,W)
    frac,nnich=niche_frac(W,giz,sm,HOT_Z,MIN_NICHE)
    _,z,p=join_count(W,snc,rng)
    rows.append(dict(sid=str(sid),sample=nm,n_snc=n_snc,n_niches=nnich,
                     frac_in_niche=frac,jc_z=z,jc_p=p))
    for hz in SWEEP_Z:
        for mn in SWEEP_MIN:
            f_s,_=niche_frac(W,giz,sm,hz,mn); sweep_rows.append(dict(sample=nm,hot_z=hz,min_niche=mn,frac=f_s))
df_bc=pd.DataFrame(rows); sweep=pd.DataFrame(sweep_rows)
assert len(df_bc), "no sections survived gating — check SNC_COL / MIN_SNC"

# unique display labels (F-64·1, F-64·2 …)
cnt=df_bc['sample'].value_counts(); seen={}; lab={}
for sid,nm in zip(df_bc['sid'],df_bc['sample']):
    if cnt[nm]==1: lab[sid]=nm
    else: seen[nm]=seen.get(nm,0)+1; lab[sid]=f'{nm}·{seen[nm]}'
df_bc['label']=df_bc['sid'].map(lab)

# ── style ─────────────────────────────────────────────────────────────────────
for fam in ('Arial','Helvetica','Liberation Sans'):
    if any(fam in f.name for f in font_manager.fontManager.ttflist):
        plt.rcParams['font.sans-serif']=[fam]; break
plt.rcParams.update({'font.family':'sans-serif','axes.linewidth':0.5,
    'xtick.major.width':0.5,'xtick.major.size':2.5,'pdf.fonttype':42,'ps.fonttype':42,'figure.dpi':300})
INK='#1a1a1a'; SUB='#777'; MUTE='#9a9a9a'; SIG='#D85A30'; NSDOT='#FFFFFF'; NSEDGE='#B4B2A9'
STEM='#E7D2C6'; POOL='#185FA5'
CMAP_F=LinearSegmentedColormap.from_list('f',['#FAECE7','#F5C4B3','#F0997B','#D85A30','#993C1D'])
CMAP_N=LinearSegmentedColormap.from_list('n',['#F1EFE8','#D3D1C7','#B4B2A9','#888780','#5F5E5A'])

# ── order + pooled ────────────────────────────────────────────────────────────
d=df_bc.sort_values('jc_z',ascending=False).reset_index(drop=True)
frac=d['frac_in_niche'].values*100; nn=d['n_niches'].values; z=d['jc_z'].values; sig=(d['jc_p']<0.05).values
fr_mu=frac.mean(); fr_ci=stats.t.ppf(.975,len(frac)-1)*frac.std(ddof=1)/np.sqrt(len(frac))
z_mu=z.mean(); z_ci=stats.t.ppf(.975,len(z)-1)*z.std(ddof=1)/np.sqrt(len(z))
stouffer=z.sum()/np.sqrt(len(z)); p_pool=stats.norm.sf(stouffer)
Q=np.sum((z-z.mean())**2); I2=max(0,(Q-(len(z)-1))/Q)*100 if Q>0 else 0; n_pos=int((z>0).sum())
n=len(d); GAP=1.0; y=np.arange(n); ypool=n-1+GAP
normF=Normalize(0,max(55,frac.max())); normN=Normalize(0,max(16,nn.max()))
tc=lambda nv:'#FFFFFF' if nv>0.55 else '#4A1B0C'

# ── figure (margins opened to hold the outside labels) ────────────────────────
fig,(axL,axR)=plt.subplots(1,2,sharey=True,figsize=(4.9,0.235*n+1.05),
                           gridspec_kw={'width_ratios':[0.85,2.6],'wspace':0.04})
fig.subplots_adjust(left=0.165,right=0.99,top=0.86,bottom=0.16)

# ----- panel b: heatmap strip (frameless) -----
for i in range(n):
    axL.barh(y[i],1,left=-0.5,height=0.9,color=CMAP_F(normF(frac[i])),edgecolor='white',lw=0.5,zorder=2)
    axL.barh(y[i],1,left= 0.5,height=0.9,color=CMAP_N(normN(nn[i])), edgecolor='white',lw=0.5,zorder=2)
    axL.text(0,y[i],f'{frac[i]:.0f}',ha='center',va='center',fontsize=5.6,color=tc(normF(frac[i])))
    axL.text(1,y[i],f'{nn[i]:.0f}',  ha='center',va='center',fontsize=5.6,color=tc(normN(nn[i])))
axL.barh(ypool,1,left=-0.5,height=0.9,color=CMAP_F(normF(fr_mu)),edgecolor='white',lw=0.5,zorder=2)
axL.text(0,ypool,f'{fr_mu:.0f}',ha='center',va='center',fontsize=5.6,color=tc(normF(fr_mu)))
axL.set_xlim(-0.62,1.62); axL.set_xticks([0,1]); axL.set_xticklabels(['%SnC','N'],fontsize=6,color=SUB)
axL.xaxis.set_ticks_position('top'); axL.tick_params(axis='x',length=0,pad=1)
axL.set_yticks(list(y)+[ypool]); axL.set_yticklabels(list(d['label'])+['pooled'],fontsize=6,color=INK)
axL.get_yticklabels()[-1].set_color(POOL); axL.get_yticklabels()[-1].set_fontstyle('italic')
axL.tick_params(axis='y',length=0,pad=2)
for sp in axL.spines.values(): sp.set_visible(False)

# ----- panel c: clustering lollipops, full border -----
axR.axvline(0,color=INK,lw=0.7,zorder=1); axR.axvline(1.96,color='#CCCCCC',lw=0.6,ls=(0,(3,3)),zorder=1)
for i in range(n): axR.plot([0,z[i]],[y[i],y[i]],color=STEM,lw=1.5,solid_capstyle='round',zorder=2)
axR.scatter(z[sig], y[sig], s=22,color=SIG,edgecolors='white',linewidths=0.5,zorder=4)
axR.scatter(z[~sig],y[~sig],s=22,color=NSDOT,edgecolors=NSEDGE,linewidths=1.0,zorder=4)
axR.plot([z_mu-z_ci,z_mu+z_ci],[ypool,ypool],color=POOL,lw=1.0,solid_capstyle='butt',zorder=3)
axR.scatter([z_mu],[ypool],marker='D',s=30,color=POOL,edgecolors='white',linewidths=0.5,zorder=4)
axR.annotate(f'z̄ {z_mu:.1f}',(z_mu,ypool),xytext=(6,0),textcoords='offset points',
             ha='left',va='center',fontsize=5.8,color=POOL,fontweight='bold')
xlo,xhi=min(-2,z.min()*1.12),max(10.5,z.max()*1.06)
axR.set_xlim(xlo,xhi); axR.set_xticks([0,2.5,5,7.5,10]); axR.tick_params(axis='x',labelsize=6,colors=INK)
axR.set_xlabel('senescent-spot clustering ($z$)',fontsize=6.5,labelpad=4)
for sp in axR.spines.values(): sp.set_visible(True); sp.set_linewidth(0.6)

# ----- anchors: axes-fraction coords, OUTSIDE the frame (no border collisions) -
fx=lambda zv:(zv-xlo)/(xhi-xlo)
axR.text(fx(1.96),1.015,'p<0.05',transform=axR.transAxes,
         ha='center',va='bottom',fontsize=5.4,color=MUTE,style='italic')
axR.text(1.0,1.015,'more clustered →',transform=axR.transAxes,
         ha='right',va='bottom',fontsize=5.4,color=MUTE,style='italic')
axR.text(fx(0.0),-0.115,'0 = random',transform=axR.transAxes,
         ha='center',va='top',fontsize=5.4,color=MUTE,style='italic')

for ax in (axL,axR): ax.set_ylim(ypool+0.7,-0.9)

for ext in ('pdf','svg','png'):
    fig.savefig(FIGURES_DIR/f'fig_BC_strip_lollipop.{ext}',dpi=600,bbox_inches='tight',pad_inches=0.03,facecolor='white')
plt.show()
sw=sweep.groupby(['hot_z','min_niche'])['frac'].mean()*100
print(f"b: pooled {fr_mu:.0f}% SnC-in-niche (CI {fr_mu-fr_ci:.0f}–{fr_mu+fr_ci:.0f}); cutoff {sw.min():.0f}–{sw.max():.0f}%")
print(f"c: {n_pos}/{len(z)} positive ({N_PERM} perms), z̄={z_mu:.1f}, Stouffer={stouffer:.1f}, p={p_pool:.0e}, I²={I2:.0f}%")

---
## 18 · Bivariate Moran's I

**Why.** Everything above is univariate — does one variable cluster with itself.
Bivariate asks whether variable A at a spot predicts variable B in the
*neighbouring* spots. That is the spatial form of a cross-variable claim: not
"senescence and inflammation co-occur" but "senescence here predicts
inflammation next door".

**What it can and cannot support.** A positive bivariate I is consistent with
signalling between neighbourhoods; it is equally consistent with both variables
tracking a third spatial factor. It is a spatial association, not a direction
of influence, and a claim about propagation needs more than this statistic can
give.

**Test.** Bivariate Moran's I per sample pair, permutation null, then DL
random-effects pooling with I² — the same machinery as section 05.

**Read I² carefully here.** Bivariate estimates are noisier than univariate
ones, so high heterogeneity is expected and a pooled estimate over
strongly disagreeing samples describes the average of a disagreement.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODULE 11b — BIVARIATE MORAN'S I: CROSS-DATASET META-ANALYSIS
# Cell 0: Config + Imports + Gene Sets
# ═══════════════════════════════════════════════════════════════════════════════
# Env: omicverse
#
# GOAL: For each control sample × cell type, test whether senescence score
#       (or binary label) spatially co-localizes with hallmark module scores.
#       Pool across 21 controls (6 Harari + 15 Morabito) via DL meta-analysis.
#
# QUESTION (Sara): "Can you do an aggregate measure to see if senescent cells
#                   are enriched in areas high in SASPs?"
# ═══════════════════════════════════════════════════════════════════════════════

import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import KDTree
from scipy.stats import norm
from scipy import sparse as sp_sparse
from esda.moran import Moran_BV
from libpysal.weights import W as LibW
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("MODULE 11b — BIVARIATE MORAN'S I META-ANALYSIS")
print("Cell 0: Config + Imports + Gene Sets")
print("=" * 60)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE        = Path(SEN_DATA)
DATA_DIR    = BASE / 'data' / '10_spatial'
RESULTS_DIR = BASE / 'results' / '10_spatial' / 'meta_bivariate'
FIGURES_DIR = BASE / 'figures' / '10_spatial' / 'meta_bivariate'
MARKERS_DIR = Path(SEN_REF_MARKERS)
SLOAN_FILE  = MARKERS_DIR / '1-s2.0-S2666979X25003830-mmc10.xlsx'

for d in [RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Checkpoints ──────────────────────────────────────────────────────────────
CHECKPOINTS = {
    'harari':   DATA_DIR / 'spatial_an1792_senepy_labeled.h5ad',
    'morabito': DATA_DIR / 'morabito_dsad_senepy_labeled.h5ad',
}

# ── Harari sample metadata ──────────────────────────────────────────────────
HARARI_META = {
    'GSM5820546': {'name': 'NND-1', 'sex': 'M', 'age': 77, 'group': 'Harari'},
    'GSM5820547': {'name': 'NND-2', 'sex': 'F', 'age': 74, 'group': 'Harari'},
    'GSM5820548': {'name': 'NND-3', 'sex': 'M', 'age': 70, 'group': 'Harari'},
    'GSM5820549': {'name': 'NND-4', 'sex': 'F', 'age': 84, 'group': 'Harari'},
    'GSM5820550': {'name': 'NND-5', 'sex': 'M', 'age': 83, 'group': 'Harari'},
    'GSM5820551': {'name': 'NND-6', 'sex': 'F', 'age': 68, 'group': 'Harari'},
}

# ── Cell type abbreviations ─────────────────────────────────────────────────
CT_ABBREV = {
    'Astrocyte': 'Astro', 'Excitatory': 'Exc', 'Inhibitory': 'Inh',
    'Oligodendrocyte': 'Oligo', 'OPC': 'OPC', 'Microglia': 'Micro',
    'Vascular': 'Vasc', 'VSMC': 'VSMC', 'Endothelial': 'Endo',
    'Pericyte': 'Peri', 'PVM': 'PVM', 'VLMC': 'VLMC',
}

# ── Analysis parameters ─────────────────────────────────────────────────────
N_PERMS      = 999    # permutations for Moran_BV
K_NEIGH      = 6      # Visium hexagonal grid neighbors
MIN_SPOTS_CT = 50     # minimum spots per cell type per sample
MIN_SAMPLES  = 3      # minimum samples to run DL meta-analysis

# ── Load gene lists from Sloan et al. Excel ─────────────────────────────────
print(f"\n{'─'*60}")
print("LOADING SENESCENCE GENE LISTS")
print(f"{'─'*60}")
print(f"  Source: {SLOAN_FILE}")

HALLMARK_NAMES = [
    'p53_Targets', 'CellCycleArrest', 'SASP', 'AntiApoptosis',
    'DDR', 'CellSurfaceMarkers', 'LysosomalContent',
    'SD_TMC', 'SenMayo', 'Fridman_Up',
]
HALLMARK_TYPES = ['hallmark'] * 7 + ['multi_hallmark'] * 3

sloan_raw = pd.read_excel(SLOAN_FILE, sheet_name='Sen Gene Lists')

# Parse each column → raw gene list (filtering to detected genes happens in Cell 2)
GENE_LISTS_RAW = {}
for i, name in enumerate(HALLMARK_NAMES):
    genes = sloan_raw.iloc[1:, i].dropna().astype(str).str.strip()
    genes = genes[genes != ''].tolist()
    GENE_LISTS_RAW[name] = genes
    print(f"  {name:<22s}: {len(genes):>3d} genes  [{HALLMARK_TYPES[i]}]")

print(f"\n  Total: {len(HALLMARK_NAMES)} gene lists loaded from Excel")
print(f"  (Filtering to detected genes happens per-sample in Cell 2)")

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("CONFIG SUMMARY")
print(f"{'='*60}")
print(f"  Checkpoints:  {', '.join(CHECKPOINTS.keys())}")
print(f"  Gene lists:   {len(GENE_LISTS_RAW)} hallmarks from Sloan et al.")
print(f"  Permutations: {N_PERMS}")
print(f"  K neighbors:  {K_NEIGH}")
print(f"  Min spots/CT: {MIN_SPOTS_CT}")
print(f"  Min samples:  {MIN_SAMPLES}")
print(f"  X variables:  sen_score (continuous) + is_senescent (binary)")
print(f"  Output:")
print(f"    Results: {RESULTS_DIR}")
print(f"    Figures: {FIGURES_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 1: Load Data — Both Checkpoints, Controls Only
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 1: LOAD DATA")
print("=" * 60)
print()

# ── Load Harari ──────────────────────────────────────────────────────────────
print("Loading Harari checkpoint...")
adata_harari = sc.read_h5ad(CHECKPOINTS['harari'])
print(f"  Shape: {adata_harari.shape}")
print(f"  Layers: {list(adata_harari.layers.keys())}")

# Identify sample column
harari_sid_col = 'gsm_id' if 'gsm_id' in adata_harari.obs.columns else 'sample_id'
harari_samples = sorted(adata_harari.obs[harari_sid_col].unique())
print(f"  Sample col: '{harari_sid_col}' → {len(harari_samples)} samples")

for sid in harari_samples:
    n = (adata_harari.obs[harari_sid_col] == sid).sum()
    meta = HARARI_META.get(sid, {})
    name = meta.get('name', sid)
    print(f"    {name} ({sid}): {n:,} spots")

# ── Load Morabito ────────────────────────────────────────────────────────────
print(f"\nLoading Morabito checkpoint...")
adata_morabito_full = sc.read_h5ad(CHECKPOINTS['morabito'])
print(f"  Full shape: {adata_morabito_full.shape}")

morabito_sid_col = 'sample_id'

# ── EXPLICIT control filter ──────────────────────────────────────────────────
# Find the condition column
cond_col = None
for col in ['condition', 'disease', 'diagnosis', 'group']:
    if col in adata_morabito_full.obs.columns:
        cond_col = col
        break

if cond_col is None:
    raise KeyError("No condition column found in Morabito checkpoint")

print(f"  Condition column: '{cond_col}'")
print(f"  Values: {dict(adata_morabito_full.obs[cond_col].value_counts())}")

ctrl_mask = adata_morabito_full.obs[cond_col] == 'Control'
adata_morabito = adata_morabito_full[ctrl_mask].copy()
del adata_morabito_full

print(f"  Filtered to Control: {adata_morabito.shape[0]:,} spots")

morabito_samples = sorted(adata_morabito.obs[morabito_sid_col].unique())
print(f"  Control samples: {len(morabito_samples)}")

for sid in morabito_samples:
    n = (adata_morabito.obs[morabito_sid_col] == sid).sum()
    print(f"    {sid}: {n:,} spots")

# ── Verify required columns ─────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("COLUMN VERIFICATION")
print(f"{'─'*60}")

required = ['sen_score', 'is_senescent', 'dominant_celltype']
for label, ad in [('Harari', adata_harari), ('Morabito', adata_morabito)]:
    print(f"  {label}:")
    for col in required:
        present = col in ad.obs.columns
        print(f"    {col:<22s}: {'✓' if present else '✗ MISSING'}")
    has_lognorm = 'lognorm' in ad.layers
    print(f"    layers['lognorm']:    {'✓' if has_lognorm else '✗ MISSING'}")

# ── Build short names for Morabito samples ───────────────────────────────────
# Full IDs like 'Dec_13_2021_Human1' → 'D13-H1'
#              'Nov_24_2021_VisiumHuman_9' → 'N24-VH9'
#              'Oct_2021_5' → 'Oct-5'

def morabito_short_name(sid):
    """Create readable short name from Morabito sample_id."""
    s = str(sid)
    if s.startswith('Dec_13'):
        num = s.split('Human')[-1]
        return f'D13-H{num}'
    elif s.startswith('Dec_20'):
        num = s.split('Human')[-1]
        return f'D20-H{num}'
    elif s.startswith('Nov_24'):
        num = s.split('_')[-1]
        return f'N24-VH{num}'
    elif s.startswith('Oct_'):
        num = s.split('_')[-1]
        return f'Oct-{num}'
    else:
        return s[:15]

# ── Build sample registry ───────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("SAMPLE REGISTRY")
print(f"{'─'*60}")

SAMPLE_REGISTRY = {}

for sid in harari_samples:
    meta = HARARI_META.get(sid, {})
    # If GSM IDs don't match HARARI_META, use the raw ID
    if meta:
        name = meta['name']
    else:
        name = str(sid)[:12]

    SAMPLE_REGISTRY[sid] = {
        'name':      name,
        'dataset':   'Harari',
        'sex':       meta.get('sex', '?'),
        'age':       meta.get('age', '?'),
        'adata_key': 'harari',
        'sid_col':   harari_sid_col,
    }

for sid in morabito_samples:
    short = morabito_short_name(sid)
    SAMPLE_REGISTRY[sid] = {
        'name':      short,
        'dataset':   'Morabito',
        'sex':       '?',
        'age':       '?',
        'adata_key': 'morabito',
        'sid_col':   morabito_sid_col,
    }

ADATA_DICT = {'harari': adata_harari, 'morabito': adata_morabito}

n_harari = len(harari_samples)
n_morabito = len(morabito_samples)
n_total = n_harari + n_morabito

print(f"  Harari:   {n_harari} samples")
print(f"  Morabito: {n_morabito} samples")
print(f"  Total:    {n_total} controls")
print()

# Verify no duplicate short names
all_names = [info['name'] for info in SAMPLE_REGISTRY.values()]
if len(all_names) != len(set(all_names)):
    dupes = [n for n in all_names if all_names.count(n) > 1]
    print(f"  ⚠ DUPLICATE NAMES: {set(dupes)}")
else:
    print(f"  ✓ All {n_total} sample names unique")

for sid, info in SAMPLE_REGISTRY.items():
    print(f"    {info['name']:<12s} ({info['dataset']:<8s})  {sid}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 2: Score Modules Per Sample (on lognorm layer)
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 2: SCORE MODULES PER SAMPLE")
print("=" * 60)
print()
print("Scoring each sample independently on lognorm layer")
print("(background gene distribution differs per sample)")
print()

# Track gene detection across datasets
gene_detection = {'harari': {}, 'morabito': {}}

for sid, info in SAMPLE_REGISTRY.items():
    name = info['name']
    adata_key = info['adata_key']
    sid_col = info['sid_col']
    ad_full = ADATA_DICT[adata_key]

    # Subset to this sample
    mask = ad_full.obs[sid_col] == sid
    ad_sample = ad_full[mask]

    # Switch to lognorm layer for scoring
    ad_tmp = ad_sample.copy()
    ad_tmp.X = ad_tmp.layers['lognorm'].copy()

    n_scored = 0
    for list_name, genes_raw in GENE_LISTS_RAW.items():
        score_col = f'Score_{list_name}'

        # Filter to genes detected in this sample
        detected = [g for g in genes_raw if g in ad_tmp.var_names]

        # Track detection
        if list_name not in gene_detection[adata_key]:
            gene_detection[adata_key][list_name] = len(detected)

        if len(detected) < 5:
            # Too few genes — fill with NaN
            ad_full.obs.loc[mask, score_col] = np.nan
            continue

        # Score on the temporary lognorm copy
        sc.tl.score_genes(
            ad_tmp, gene_list=detected,
            score_name=score_col,
            ctrl_size=min(100, len(detected)),
        )

        # Write scores back to the full object
        ad_full.obs.loc[mask, score_col] = ad_tmp.obs[score_col].values
        n_scored += 1

    print(f"  {name:<12s}: {n_scored}/10 modules scored ({mask.sum():,} spots)")
    del ad_tmp

# ── Gene detection summary ──────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("GENE DETECTION SUMMARY")
print(f"{'─'*60}")
print(f"  {'Module':<22s} {'Harari':>8s} {'Morabito':>8s} {'Total':>8s}")
print(f"  {'─'*50}")
for list_name in HALLMARK_NAMES:
    n_raw = len(GENE_LISTS_RAW[list_name])
    n_h = gene_detection['harari'].get(list_name, 0)
    n_m = gene_detection['morabito'].get(list_name, 0)
    print(f"  {list_name:<22s} {n_h:>5d}/{n_raw:<3d} {n_m:>5d}/{n_raw:<3d} {n_raw:>5d}")

# ── Verify scores exist ────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("VERIFICATION")
print(f"{'─'*60}")
score_cols = [f'Score_{name}' for name in HALLMARK_NAMES]
for label, ad in [('Harari', adata_harari), ('Morabito', adata_morabito)]:
    present = [c for c in score_cols if c in ad.obs.columns]
    missing = [c for c in score_cols if c not in ad.obs.columns]
    has_nan = [c for c in present if ad.obs[c].isna().any()]
    print(f"  {label}: {len(present)}/10 scored, {len(has_nan)} with NaNs")
    if missing:
        print(f"    Missing: {missing}")

print("\n✓ Module scoring complete")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 3: Helper Functions
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 3: HELPER FUNCTIONS")
print("=" * 60)
print()

def build_spatial_w(coords, k=6):
    """Build a PySAL W object from spatial coordinates using KDTree k-NN."""
    tree = KDTree(coords)
    _, indices = tree.query(coords, k=k+1)  # +1 because self is included
    neighbors = {}
    weights = {}
    for i in range(len(coords)):
        neigh = indices[i, 1:].tolist()  # exclude self
        neighbors[i] = neigh
        weights[i] = [1.0] * len(neigh)
    return LibW(neighbors, weights, silence_warnings=True)


def dersimonian_laird(effects, se_values):
    """
    DerSimonian-Laird random-effects meta-analysis.
    Returns dict with pooled_I, pooled_se, ci_lo, ci_hi, z, p, tau2, I2, Q, k.
    """
    effects = np.array(effects, dtype=float)
    se_values = np.array(se_values, dtype=float)

    # Fixed-effects weights
    w = 1.0 / (se_values ** 2 + 1e-12)
    W = w.sum()
    theta_fe = (w * effects).sum() / W

    # Q statistic
    Q = (w * (effects - theta_fe) ** 2).sum()
    k = len(effects)
    df = k - 1

    # Between-study variance (tau²)
    C = W - (w ** 2).sum() / W
    tau2 = max(0, (Q - df) / C) if C > 0 else 0.0

    # Random-effects weights
    w_re = 1.0 / (se_values ** 2 + tau2 + 1e-12)
    W_re = w_re.sum()
    theta_re = (w_re * effects).sum() / W_re
    se_re = np.sqrt(1.0 / W_re)

    ci_lo = theta_re - 1.96 * se_re
    ci_hi = theta_re + 1.96 * se_re
    z_val = theta_re / (se_re + 1e-12)
    p_val = 2 * (1 - norm.cdf(abs(z_val)))

    # I²
    I2 = max(0, (Q - df) / Q * 100) if Q > 0 else 0.0

    return {
        'pooled_I': theta_re, 'pooled_se': se_re,
        'ci_lo': ci_lo, 'ci_hi': ci_hi,
        'z': z_val, 'p': p_val,
        'tau2': tau2, 'I2': I2, 'Q': Q, 'k': k,
    }


def fisher_combined_p(pvals):
    """Fisher's method for combining p-values."""
    from scipy.stats import chi2
    pvals = np.array(pvals, dtype=float)
    pvals = pvals[pvals > 0]  # drop zeros
    if len(pvals) == 0:
        return 1.0
    stat = -2.0 * np.sum(np.log(pvals))
    return chi2.sf(stat, df=2 * len(pvals))


print("  ✓ build_spatial_w(coords, k)")
print("  ✓ dersimonian_laird(effects, se_values)")
print("  ✓ fisher_combined_p(pvals)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 4: Discover Testable Cell Types
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 4: DISCOVER TESTABLE CELL TYPES")
print("=" * 60)
print()
print(f"Filter: ≥{MIN_SPOTS_CT} spots per cell type per sample")
print(f"        ≥{MIN_SAMPLES} samples to include in meta-analysis")
print()

# Count spots per cell type per sample
ct_sample_counts = {}  # {cell_type: {sample_name: n_spots}}

for sid, info in SAMPLE_REGISTRY.items():
    name = info['name']
    ad = ADATA_DICT[info['adata_key']]
    sid_col = info['sid_col']
    mask = ad.obs[sid_col] == sid
    cts = ad.obs.loc[mask, 'dominant_celltype'].value_counts()
    for ct, count in cts.items():
        if ct not in ct_sample_counts:
            ct_sample_counts[ct] = {}
        ct_sample_counts[ct][name] = count

# Determine which pass the filter
TESTABLE_CTS = {}
print(f"  {'Cell Type':<20s} {'Samples≥50':>10s} {'Harari':>8s} {'Morabito':>8s} {'Status':>10s}")
print(f"  {'─'*60}")

for ct in sorted(ct_sample_counts.keys()):
    counts = ct_sample_counts[ct]
    passing = {s: n for s, n in counts.items() if n >= MIN_SPOTS_CT}
    n_pass = len(passing)

    # Count by dataset
    n_harari = sum(1 for s in passing if any(
        SAMPLE_REGISTRY[sid]['name'] == s and SAMPLE_REGISTRY[sid]['dataset'] == 'Harari'
        for sid in SAMPLE_REGISTRY
    ))
    n_morabito = sum(1 for s in passing if any(
        SAMPLE_REGISTRY[sid]['name'] == s and SAMPLE_REGISTRY[sid]['dataset'] == 'Morabito'
        for sid in SAMPLE_REGISTRY
    ))

    status = '✓ TESTABLE' if n_pass >= MIN_SAMPLES else '✗ skip'
    print(f"  {ct:<20s} {n_pass:>10d} {n_harari:>8d} {n_morabito:>8d} {status:>10s}")

    if n_pass >= MIN_SAMPLES:
        TESTABLE_CTS[ct] = {
            'n_samples': n_pass,
            'n_harari': n_harari,
            'n_morabito': n_morabito,
            'passing_samples': passing,
        }

print(f"\n  Testable cell types: {len(TESTABLE_CTS)}")
for ct, info in TESTABLE_CTS.items():
    print(f"    {ct}: {info['n_samples']} samples "
          f"(H={info['n_harari']}, M={info['n_morabito']})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 5: Compute Bivariate Moran's I — All Samples × Cell Types × Modules
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 5: COMPUTE BIVARIATE MORAN'S I")
print("=" * 60)
print()
print(f"Loop: {len(SAMPLE_REGISTRY)} samples × {len(TESTABLE_CTS)} cell types "
      f"× {len(HALLMARK_NAMES)} modules × 2 X-vars × {N_PERMS} perms")
print()

score_cols = [f'Score_{name}' for name in HALLMARK_NAMES]
all_results = []

for sid, s_info in SAMPLE_REGISTRY.items():
    name = s_info['name']
    dataset = s_info['dataset']
    ad = ADATA_DICT[s_info['adata_key']]
    sid_col = s_info['sid_col']

    mask = ad.obs[sid_col] == sid
    ad_sample = ad[mask]

    print(f"  {name} ({dataset}, {ad_sample.n_obs:,} spots)")

    for ct in TESTABLE_CTS:
        ct_mask = ad_sample.obs['dominant_celltype'] == ct
        n_ct = ct_mask.sum()
        if n_ct < MIN_SPOTS_CT:
            continue

        ad_ct = ad_sample[ct_mask].copy()
        coords = ad_ct.obsm['spatial']

        # ── Filter out spots with non-finite spatial coordinates ─────────
        finite_mask = np.all(np.isfinite(coords), axis=1)
        n_bad = (~finite_mask).sum()
        if n_bad > 0:
            print(f"    ⚠ {ct}: dropping {n_bad} spots with NaN/inf coordinates")
            ad_ct = ad_ct[finite_mask].copy()
            coords = ad_ct.obsm['spatial']
            n_ct = ad_ct.n_obs
        if n_ct < MIN_SPOTS_CT:
            continue

        w = build_spatial_w(coords, k=K_NEIGH)

        # X variables
        x_sen = ad_ct.obs['sen_score'].astype(float).values
        x_bin = ad_ct.obs['is_senescent'].astype(float).values

        if x_sen.std() < 1e-10:
            continue

        ct_abbr = CT_ABBREV.get(ct, ct[:5])
        n_mods_done = 0

        for list_name in HALLMARK_NAMES:
            score_col = f'Score_{list_name}'
            if score_col not in ad_ct.obs.columns:
                continue
            y_mod = ad_ct.obs[score_col].astype(float).values
            if np.isnan(y_mod).all() or y_mod.std() < 1e-10:
                continue

            # ── sen_score × module ───────────────────────────────────────
            bv_sen = Moran_BV(x_sen, y_mod, w, permutations=N_PERMS)
            I_sen = bv_sen.I
            p_sen = bv_sen.p_sim
            SE_sen = np.std(bv_sen.sim)

            # ── is_senescent × module ────────────────────────────────────
            if x_bin.std() > 1e-10:
                bv_bin = Moran_BV(x_bin, y_mod, w, permutations=N_PERMS)
                I_bin = bv_bin.I
                p_bin = bv_bin.p_sim
                SE_bin = np.std(bv_bin.sim)
            else:
                I_bin, p_bin, SE_bin = np.nan, np.nan, np.nan

            all_results.append({
                'sample': name, 'sample_id': sid, 'dataset': dataset,
                'sex': s_info['sex'], 'age': s_info['age'],
                'cell_type': ct, 'module': list_name,
                'n_spots': n_ct,
                'I_sen': I_sen, 'p_sen': p_sen, 'SE_sen': SE_sen,
                'I_bin': I_bin, 'p_bin': p_bin, 'SE_bin': SE_bin,
            })
            n_mods_done += 1

        if n_mods_done > 0:
            print(f"    {ct_abbr:<8s} (n={n_ct:>5d}): {n_mods_done} modules")

    print()

df_biv = pd.DataFrame(all_results)

# ── Save raw results ────────────────────────────────────────────────────────
df_biv.to_csv(RESULTS_DIR / 'bivariate_morans_I_all_controls.csv', index=False)
print(f"✓ Saved: {RESULTS_DIR / 'bivariate_morans_I_all_controls.csv'}")
print(f"  {len(df_biv)} rows: {df_biv['sample'].nunique()} samples × "
      f"{df_biv['cell_type'].nunique()} cell types × "
      f"{df_biv['module'].nunique()} modules")

# ── Quick summary ───────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("SIGNIFICANCE SUMMARY (p < 0.05)")
print(f"{'─'*60}")
for xvar, pcol in [('sen_score', 'p_sen'), ('is_senescent', 'p_bin')]:
    valid = df_biv[pcol].notna()
    sig = df_biv[valid & (df_biv[pcol] < 0.05)]
    print(f"  {xvar}: {len(sig)}/{valid.sum()} tests significant "
          f"({len(sig)/max(valid.sum(),1)*100:.1f}%)")

In [ ]:
# ── Harari sample column: prefer readable label (sample_name = NND-x) ────────
harari_sid_col = next((c for c in ['sample_name', 'gsm_id', 'sample_id']
                       if c in adata_harari.obs.columns), 'gsm_id')
harari_samples = sorted(adata_harari.obs[harari_sid_col].unique())

# ── Morabito F-/M-age labeler (sex-age collides → '.N' suffix, sample_id order) ──
def build_morabito_labels(adata, sid_col):
    from collections import Counter
    obs = adata.obs
    raw = {}
    for sid in sorted(obs[sid_col].unique()):
        sub = obs[obs[sid_col] == sid]
        sex = str(sub['sex'].iloc[0]) if 'sex' in sub else '?'
        age = sub['age'].iloc[0]      if 'age' in sub else '?'
        raw[sid] = f"{sex}-{age}"
    counts = Counter(raw.values())
    seen, labels = {}, {}
    for sid in sorted(raw):
        lab = raw[sid]
        if counts[lab] > 1:
            seen[lab] = seen.get(lab, 0) + 1
            labels[sid] = f"{lab}.{seen[lab]}"
        else:
            labels[sid] = lab
    return labels

# ── Build registry — name/sex/age from obs, no hardcoded dicts ───────────────
SAMPLE_REGISTRY = {}

for sid in harari_samples:                       # key + label = NND-x
    sub = adata_harari.obs[adata_harari.obs[harari_sid_col] == sid]
    SAMPLE_REGISTRY[sid] = {
        'name':      sid,
        'dataset':   'Harari',
        'sex':       str(sub['sex'].iloc[0]) if 'sex' in sub else '?',
        'age':       sub['age'].iloc[0]      if 'age' in sub else '?',
        'adata_key': 'harari',
        'sid_col':   harari_sid_col,
    }

morabito_labels = build_morabito_labels(adata_morabito, morabito_sid_col)
for sid in morabito_samples:                     # key = sample_id, label = F-/M-age(.N)
    sub = adata_morabito.obs[adata_morabito.obs[morabito_sid_col] == sid]
    SAMPLE_REGISTRY[sid] = {
        'name':      morabito_labels[sid],
        'dataset':   'Morabito',
        'sex':       str(sub['sex'].iloc[0]) if 'sex' in sub else '?',
        'age':       sub['age'].iloc[0]      if 'age' in sub else '?',
        'adata_key': 'morabito',
        'sid_col':   morabito_sid_col,
    }

In [ ]:
for sid, info in SAMPLE_REGISTRY.items():
    print(f"  {info['name']:<10s} {info['dataset']:<8s} sex={info['sex']:<3s} age={str(info['age']):<5s} key={sid}")

names = [i['name'] for i in SAMPLE_REGISTRY.values()]
print(f"\n  {len(names)} samples, {len(set(names))} unique labels",
      "✓" if len(names) == len(set(names)) else "✗ DUPLICATES")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 6: DerSimonian-Laird Meta-Analysis
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 6: DL META-ANALYSIS")
print("=" * 60)
print()

meta_results = []

for x_label, I_col, SE_col, p_col in [
    ('sen_score',    'I_sen', 'SE_sen', 'p_sen'),
    ('is_senescent', 'I_bin', 'SE_bin', 'p_bin'),
]:
    print(f"  ── X = {x_label} ──")
    for ct in sorted(TESTABLE_CTS.keys()):
        for mod in HALLMARK_NAMES:
            sub = df_biv[(df_biv['cell_type'] == ct) &
                         (df_biv['module'] == mod)].dropna(subset=[I_col, SE_col])
            if len(sub) < MIN_SAMPLES:
                continue

            effects = sub[I_col].values
            ses = sub[SE_col].values
            pvals = sub[p_col].values

            ma = dersimonian_laird(effects, ses)
            fisher_p = fisher_combined_p(pvals)

            direction = 'co-localize' if ma['pooled_I'] > 0 else 'segregate'
            sig = '***' if ma['p'] < 0.001 else ('**' if ma['p'] < 0.01 else ('*' if ma['p'] < 0.05 else ''))

            meta_results.append({
                'x_variable': x_label,
                'cell_type': ct, 'module': mod,
                'pooled_I': ma['pooled_I'], 'pooled_se': ma['pooled_se'],
                'ci_lo': ma['ci_lo'], 'ci_hi': ma['ci_hi'],
                'z': ma['z'], 'p': ma['p'],
                'tau2': ma['tau2'], 'I2': ma['I2'], 'Q': ma['Q'], 'k': ma['k'],
                'fisher_p': fisher_p,
                'direction': direction if ma['p'] < 0.05 else 'ns',
            })

            if ma['p'] < 0.05:
                print(f"    {ct:<18s} × {mod:<20s}: I={ma['pooled_I']:+.4f} "
                      f"[{ma['ci_lo']:+.3f},{ma['ci_hi']:+.3f}] "
                      f"p={ma['p']:.4f}{sig} I²={ma['I2']:.0f}% k={ma['k']} "
                      f"→ {direction}")
    print()

df_meta = pd.DataFrame(meta_results)
df_meta.to_csv(RESULTS_DIR / 'bivariate_meta_analysis.csv', index=False)
print(f"✓ Saved: {RESULTS_DIR / 'bivariate_meta_analysis.csv'}")
print(f"  {len(df_meta)} meta-analyses ({df_meta[df_meta['p'] < 0.05].shape[0]} significant)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 7: Stratified Meta-Analysis + Cross-Cohort Concordance
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 7: STRATIFIED META + CONCORDANCE")
print("=" * 60)
print()

strat_results = []

for x_label, I_col, SE_col in [
    ('sen_score',    'I_sen', 'SE_sen'),
    ('is_senescent', 'I_bin', 'SE_bin'),
]:
    for ct in sorted(TESTABLE_CTS.keys()):
        for mod in HALLMARK_NAMES:
            for cohort in ['Harari', 'Morabito']:
                sub = df_biv[(df_biv['cell_type'] == ct) &
                             (df_biv['module'] == mod) &
                             (df_biv['dataset'] == cohort)].dropna(subset=[I_col, SE_col])
                if len(sub) < 2:
                    continue

                ma = dersimonian_laird(sub[I_col].values, sub[SE_col].values)
                strat_results.append({
                    'x_variable': x_label, 'cohort': cohort,
                    'cell_type': ct, 'module': mod,
                    'pooled_I': ma['pooled_I'], 'pooled_se': ma['pooled_se'],
                    'ci_lo': ma['ci_lo'], 'ci_hi': ma['ci_hi'],
                    'p': ma['p'], 'I2': ma['I2'], 'k': ma['k'],
                    'direction': 'co-localize' if ma['pooled_I'] > 0 else 'segregate',
                })

df_strat = pd.DataFrame(strat_results)
df_strat.to_csv(RESULTS_DIR / 'bivariate_stratified_meta.csv', index=False)
print(f"✓ Saved: {RESULTS_DIR / 'bivariate_stratified_meta.csv'}")

# ── Concordance analysis ────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("CROSS-COHORT CONCORDANCE")
print(f"{'─'*60}")

for x_label in ['sen_score', 'is_senescent']:
    print(f"\n  ── X = {x_label} ──")
    sub_x = df_strat[df_strat['x_variable'] == x_label]

    concordant = 0
    discordant = 0
    one_only = 0

    print(f"  {'Cell Type':<18s} {'Module':<20s} {'Harari':>10s} {'Morabito':>10s} {'Match':>8s}")
    print(f"  {'─'*70}")

    for ct in sorted(TESTABLE_CTS.keys()):
        for mod in HALLMARK_NAMES:
            h = sub_x[(sub_x['cell_type'] == ct) & (sub_x['module'] == mod) &
                       (sub_x['cohort'] == 'Harari')]
            m = sub_x[(sub_x['cell_type'] == ct) & (sub_x['module'] == mod) &
                       (sub_x['cohort'] == 'Morabito')]

            if h.empty or m.empty:
                if not h.empty or not m.empty:
                    one_only += 1
                continue

            h_dir = h.iloc[0]['direction']
            m_dir = m.iloc[0]['direction']

            # Compare direction of effect (sign of pooled I), not significance
            h_sign = '+' if h.iloc[0]['pooled_I'] > 0 else '-'
            m_sign = '+' if m.iloc[0]['pooled_I'] > 0 else '-'
            match = '✓' if h_sign == m_sign else '✗ FLIP'

            if h_sign == m_sign:
                concordant += 1
            else:
                discordant += 1
                print(f"  {ct:<18s} {mod:<20s} {h_sign:>10s} {m_sign:>10s} {match:>8s}")

    total = concordant + discordant
    if total > 0:
        print(f"\n  Concordant: {concordant}/{total} ({concordant/total*100:.0f}%)")
        print(f"  Discordant: {discordant}/{total} ({discordant/total*100:.0f}%)")
        print(f"  Single-cohort only: {one_only}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 8: Heatmaps — Pooled Bivariate I (cell type × module)
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 8: HEATMAPS")
print("=" * 60)
print()

ct_order = sorted(TESTABLE_CTS.keys())
mod_order = HALLMARK_NAMES

for x_label in ['sen_score', 'is_senescent']:
    sub = df_meta[df_meta['x_variable'] == x_label]
    if sub.empty:
        continue

    # Build matrices
    mat_I = np.full((len(ct_order), len(mod_order)), np.nan)
    mat_p = np.full((len(ct_order), len(mod_order)), np.nan)

    for i, ct in enumerate(ct_order):
        for j, mod in enumerate(mod_order):
            row = sub[(sub['cell_type'] == ct) & (sub['module'] == mod)]
            if not row.empty:
                mat_I[i, j] = row.iloc[0]['pooled_I']
                mat_p[i, j] = row.iloc[0]['p']

    # Plot
    fig, ax = plt.subplots(figsize=(1.0 + 0.6 * len(mod_order),
                                     0.8 + 0.45 * len(ct_order)))

    vmax = max(np.nanmax(np.abs(mat_I[np.isfinite(mat_I)])), 0.02)
    im = ax.imshow(mat_I, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)

    for i in range(mat_I.shape[0]):
        for j in range(mat_I.shape[1]):
            val = mat_I[i, j]
            pv = mat_p[i, j]
            if np.isnan(val):
                ax.text(j, i, '—', ha='center', va='center', fontsize=5, color='#AAA')
                continue
            sig = '***' if pv < 0.001 else ('**' if pv < 0.01 else ('*' if pv < 0.05 else ''))
            tc = 'white' if abs(val) > vmax * 0.55 else 'black'
            ax.text(j, i, f'{val:+.3f}', ha='center', va='center', fontsize=5.5, color=tc)
            if sig:
                ax.text(j, i + 0.32, sig, ha='center', va='center',
                        fontsize=4.5, color=tc, fontweight='bold')

    ct_labels = [CT_ABBREV.get(ct, ct) for ct in ct_order]
    ax.set_xticks(range(len(mod_order)))
    ax.set_xticklabels(mod_order, fontsize=6, rotation=45, ha='right')
    ax.set_yticks(range(len(ct_order)))
    ax.set_yticklabels(ct_labels, fontsize=7)
    ax.set_title(f"Pooled Bivariate I — X = {x_label}\n"
                 f"(DL meta-analysis, {len(SAMPLE_REGISTRY)} controls)",
                 fontsize=8, fontweight='bold', pad=6)

    cbar = plt.colorbar(im, ax=ax, shrink=0.5, aspect=20, pad=0.04)
    cbar.ax.tick_params(labelsize=5)
    cbar.set_label("Bivariate Moran's I", fontsize=6)

    plt.tight_layout()
    for fmt in ['pdf', 'svg', 'png']:
        plt.savefig(FIGURES_DIR / f'heatmap_bivariate_meta_{x_label}.{fmt}',
                    dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  ✓ heatmap_bivariate_meta_{x_label}")

print()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 9: Forest Plots — Per Cell Type (Individual Samples + Pooled Diamond)
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 9: FOREST PLOTS — PER CELL TYPE")
print("=" * 60)
print()

def fmt_p(pv):
    if pv < 0.001: return f"{pv:.0e}"
    elif pv < 0.01: return f"{pv:.3f}"
    else: return f"{pv:.2f}"

# ── Sample colors (consistent across all plots) ─────────────────────────────
_pal = ['#D62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd',
        '#8c564b', '#e377c2', '#17becf', '#bcbd22', '#7f7f7f',
        '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
        '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5',
        '#393b79', '#637939', '#8c6d31', '#843c39', '#7b4173']

sample_order = [info['name'] for _, info in sorted(SAMPLE_REGISTRY.items(),
                key=lambda x: (0 if x[1]['dataset'] == 'Harari' else 1, x[0]))]
scol = {s: _pal[i % len(_pal)] for i, s in enumerate(sample_order)}

# ── Column layout (normalized coords) ───────────────────────────────────────
xL  = 0.01
xFL = 0.20
xFR = 0.55
xE  = 0.58
xC  = 0.67
xP  = 0.77
xI  = 0.85

# ── One figure per cell type, per X variable ────────────────────────────────
for x_label, I_col, SE_col, p_col in [
    ('sen_score',    'I_sen', 'SE_sen', 'p_sen'),
    ('is_senescent', 'I_bin', 'SE_bin', 'p_bin'),
]:
    for ct in sorted(TESTABLE_CTS.keys()):
        ct_meta = df_meta[(df_meta['cell_type'] == ct) &
                          (df_meta['x_variable'] == x_label)]
        ct_biv = df_biv[df_biv['cell_type'] == ct].dropna(subset=[I_col, SE_col])

        if ct_meta.empty or ct_biv.empty:
            continue

        ct_samps = [s for s in sample_order if s in ct_biv['sample'].values]
        if len(ct_samps) < MIN_SAMPLES:
            continue

        # Row geometry
        SR = 0.48
        MOD_GAP = 0.25
        HDR = 0.3
        FTR = 0.4
        TOP_PAD = 0.35

        n_rows_per_mod = len(ct_samps) + 1
        mod_block = (n_rows_per_mod + 1) * SR
        content_h = len(HALLMARK_NAMES) * mod_block + (len(HALLMARK_NAMES) - 1) * MOD_GAP
        total_h = TOP_PAD + HDR + content_h + FTR
        fig_w = 6.0
        fig_h = total_h * 0.20

        # Display range
        vals = []
        for _, r in ct_biv.iterrows():
            vals.extend([r[I_col] - 1.96 * r[SE_col], r[I_col] + 1.96 * r[SE_col]])
        for _, r in ct_meta.iterrows():
            vals.extend([r['ci_lo'], r['ci_hi']])
        hr = max(max(abs(v) for v in vals) * 1.1, 0.05) if vals else 0.2
        dmin, dmax = -hr, hr

        def v2x(v):
            return xFL + (max(min(v, dmax), dmin) - dmin) / (dmax - dmin) * (xFR - xFL)

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.set_xlim(0, 1)
        ax.set_ylim(-FTR, TOP_PAD + HDR + content_h)
        ax.axis('off')

        ct_abbr = CT_ABBREV.get(ct, ct)
        y_title = TOP_PAD + HDR + content_h - 0.05
        ax.text(0.5, y_title, f"{ct} — X = {x_label}",
                fontsize=7, fontweight='bold', ha='center', va='top')

        y_hdr = content_h + 0.05
        for txt, x, ha in [('Module / Sample', xL, 'left'),
                            ('Bivariate I', (xFL + xFR) / 2, 'center'),
                            ('I', xE, 'center'), ('[95% CI]', xC, 'center'),
                            ('p', xP, 'center'), ('I²', xI, 'center')]:
            ax.text(x, y_hdr, txt, fontsize=4.5, fontweight='bold', va='center', ha=ha)
        ax.plot([0, 0.92], [y_hdr - 0.1, y_hdr - 0.1], color='k', linewidth=0.5)

        ax.plot([v2x(0), v2x(0)], [-0.1, y_hdr - 0.12],
                color='#BBB', linewidth=0.4, linestyle='--', zorder=0)

        yc = content_h - SR * 0.5

        for m_idx, mod in enumerate(HALLMARK_NAMES):
            block_top = yc + SR * 0.45
            block_bot = yc - n_rows_per_mod * SR - 0.02
            if m_idx % 2 == 0:
                ax.fill_between([0, 0.92], block_bot, block_top,
                                color='#F5F5F5', zorder=0)

            ax.text(xL, yc, mod, fontsize=5, fontweight='bold', va='center')
            yc -= SR

            for sname in ct_samps:
                srow = ct_biv[(ct_biv['sample'] == sname) & (ct_biv['module'] == mod)]
                if srow.empty:
                    ax.text(xL + 0.015, yc, sname, fontsize=4, va='center', color='#AAA')
                    ax.text(xE, yc, '—', fontsize=4, va='center', ha='center', color='#AAA')
                    yc -= SR
                    continue

                sr = srow.iloc[0]
                c = scol.get(sname, '#666')
                obs = sr[I_col]
                se = sr[SE_col]
                pv = sr[p_col]
                clo = obs - 1.96 * se
                chi = obs + 1.96 * se
                sp = pv < 0.05

                ax.text(xL + 0.015, yc, sname, fontsize=4, va='center', color=c,
                        fontweight='bold' if sp else 'normal')

                xl, xh, xo = v2x(clo), v2x(chi), v2x(obs)
                ax.plot([xl, xh], [yc, yc], color=c, linewidth=0.6, zorder=2)

                wt = 1 / (se ** 2 + 1e-6)
                mx = 1 / (ct_biv[SE_col].min() ** 2 + 1e-6)
                sz = 2.0 + 3.0 * min(wt / mx, 1.0)
                ax.plot(xo, yc, marker='s', color=c, markersize=sz,
                        markeredgecolor='white', markeredgewidth=0.2, zorder=3)

                ax.text(xE, yc, f"{obs:+.3f}", fontsize=4, va='center', ha='center',
                        fontweight='bold' if sp else 'normal')
                ax.text(xC, yc, f"[{clo:+.2f},{chi:+.2f}]", fontsize=3.5,
                        va='center', ha='center', color='#555')
                ax.text(xP, yc, fmt_p(pv), fontsize=4, va='center', ha='center',
                        fontweight='bold' if sp else 'normal',
                        color='#333' if sp else '#888')
                yc -= SR

            ma_row = ct_meta[ct_meta['module'] == mod]
            if not ma_row.empty:
                ma = ma_row.iloc[0]
                sp = ma['p'] < 0.05

                ax.plot([xL, 0.88], [yc + SR * 0.42, yc + SR * 0.42],
                        color='#DDD', linewidth=0.3)

                ax.text(xL + 0.015, yc, "Pooled", fontsize=4, va='center',
                        fontweight='bold', fontstyle='italic')

                xl, xh, xp = v2x(ma['ci_lo']), v2x(ma['ci_hi']), v2x(ma['pooled_I'])
                dw = 0.004
                dh = SR * 0.32
                ax.fill([xl, xp, xh, xp], [yc, yc - dh, yc, yc + dh],
                        color='#333', zorder=4)
                ax.plot([xl, xp, xh, xp, xl], [yc, yc - dh, yc, yc + dh, yc],
                        color='k', linewidth=0.3, zorder=4)

                ax.text(xE, yc, f"{ma['pooled_I']:+.3f}", fontsize=4,
                        va='center', ha='center', fontweight='bold')
                ax.text(xC, yc, f"[{ma['ci_lo']:+.2f},{ma['ci_hi']:+.2f}]",
                        fontsize=3.5, va='center', ha='center', color='#555')
                ax.text(xP, yc, fmt_p(ma['p']), fontsize=4, va='center', ha='center',
                        fontweight='bold', color='#C44E52' if sp else '#333')
                ax.text(xI, yc, f"{ma['I2']:.0f}%", fontsize=4, va='center', ha='center',
                        color='#C44E52' if ma['I2'] > 75 else '#888')

            yc -= SR
            if m_idx < len(HALLMARK_NAMES) - 1:
                yc -= MOD_GAP

        # X-axis
        ya = -0.08
        ax.plot([xFL, xFR], [ya, ya], color='k', linewidth=0.5)
        for tv in np.linspace(dmin, dmax, 5):
            xt = v2x(tv)
            ax.plot([xt, xt], [ya, ya - 0.04], color='k', linewidth=0.4)
            ax.text(xt, ya - 0.06, f"{tv:+.2f}", fontsize=3.5, ha='center', va='top')

        # Footer legend
        yf = ya - 0.18
        ax.plot([0, 0.92], [yf + 0.06, yf + 0.06], color='#DDD', linewidth=0.3)
        lx = xL
        for sname in ct_samps:
            ax.plot(lx, yf, marker='s', color=scol.get(sname, '#666'), markersize=3,
                    markeredgecolor='white', markeredgewidth=0.2)
            ax.text(lx + 0.012, yf, sname, fontsize=3.5, va='center',
                    color=scol.get(sname, '#666'))
            lx += 0.08
            if lx > 0.85:
                lx = xL
                yf -= 0.12
        ax.plot(lx, yf, marker='D', color='#333', markersize=3, markeredgewidth=0)
        ax.text(lx + 0.012, yf, 'Pooled (DL)', fontsize=3.5, va='center', color='#333')

        fname = f'forest_bivariate_{ct_abbr}_{x_label}'
        for fmt in ['pdf', 'svg']:
            plt.savefig(FIGURES_DIR / f'{fname}.{fmt}',
                        dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        print(f"  ✓ {fname}")

print(f"\n✓ Forest plots complete")

In [ ]:
# Relabel df_biv to current registry names — no recompute needed.
# Handles both cases: df_biv keyed on current keys, or old Harari GSM keys.
sid_to_name = {sid: info['name'] for sid, info in SAMPLE_REGISTRY.items()}
gsm_to_nnd = (adata_harari.obs[['gsm_id', 'sample_name']]
              .drop_duplicates().set_index('gsm_id')['sample_name'].to_dict())

def _relabel(row):
    sid = row['sample_id']
    if sid in sid_to_name:   return sid_to_name[sid]   # current key
    if sid in gsm_to_nnd:    return gsm_to_nnd[sid]     # old Harari GSM
    return row['sample']                                # leave untouched

df_biv['sample'] = df_biv.apply(_relabel, axis=1)
print(df_biv['sample'].drop_duplicates().tolist())  # sanity check: 21 new labels

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 9: Forest Plots — Per Cell Type (Individual Samples + Pooled Diamond)
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 9: FOREST PLOTS — PER CELL TYPE")
print("=" * 60)
print()

def fmt_p(pv):
    if pv < 0.001: return f"{pv:.0e}"
    elif pv < 0.01: return f"{pv:.3f}"
    else: return f"{pv:.2f}"

# ── Sample colors (consistent across all plots) ─────────────────────────────
_pal = ['#D62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd',
        '#8c564b', '#e377c2', '#17becf', '#bcbd22', '#7f7f7f',
        '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
        '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5',
        '#393b79', '#637939', '#8c6d31', '#843c39', '#7b4173']

sample_order = [info['name'] for _, info in sorted(SAMPLE_REGISTRY.items(),
                key=lambda x: (0 if x[1]['dataset'] == 'Harari' else 1, x[0]))]
scol = {s: _pal[i % len(_pal)] for i, s in enumerate(sample_order)}

# ── Column layout (normalized coords) ───────────────────────────────────────
xL  = 0.01
xFL = 0.20
xFR = 0.55
xE  = 0.58
xC  = 0.67
xP  = 0.77
xI  = 0.85

# ── One figure per cell type, per X variable ────────────────────────────────
for x_label, I_col, SE_col, p_col in [
    ('sen_score',    'I_sen', 'SE_sen', 'p_sen'),
    ('is_senescent', 'I_bin', 'SE_bin', 'p_bin'),
]:
    for ct in sorted(TESTABLE_CTS.keys()):
        ct_meta = df_meta[(df_meta['cell_type'] == ct) &
                          (df_meta['x_variable'] == x_label)]
        ct_biv = df_biv[df_biv['cell_type'] == ct].dropna(subset=[I_col, SE_col])

        if ct_meta.empty or ct_biv.empty:
            continue

        ct_samps = [s for s in sample_order if s in ct_biv['sample'].values]
        if len(ct_samps) < MIN_SAMPLES:
            continue

        # Row geometry
        SR = 0.48
        MOD_GAP = 0.25
        HDR = 0.3
        FTR = 0.4
        TOP_PAD = 0.35

        n_rows_per_mod = len(ct_samps) + 1
        mod_block = (n_rows_per_mod + 1) * SR
        content_h = len(HALLMARK_NAMES) * mod_block + (len(HALLMARK_NAMES) - 1) * MOD_GAP
        total_h = TOP_PAD + HDR + content_h + FTR
        fig_w = 6.0
        fig_h = total_h * 0.20

        # Display range
        vals = []
        for _, r in ct_biv.iterrows():
            vals.extend([r[I_col] - 1.96 * r[SE_col], r[I_col] + 1.96 * r[SE_col]])
        for _, r in ct_meta.iterrows():
            vals.extend([r['ci_lo'], r['ci_hi']])
        hr = max(max(abs(v) for v in vals) * 1.1, 0.05) if vals else 0.2
        dmin, dmax = -hr, hr

        def v2x(v):
            return xFL + (max(min(v, dmax), dmin) - dmin) / (dmax - dmin) * (xFR - xFL)

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.set_xlim(0, 1)
        ax.set_ylim(-FTR, TOP_PAD + HDR + content_h)
        ax.axis('off')

        ct_abbr = CT_ABBREV.get(ct, ct)
        y_title = TOP_PAD + HDR + content_h - 0.05
        ax.text(0.5, y_title, f"{ct} — X = {x_label}",
                fontsize=7, fontweight='bold', ha='center', va='top')

        y_hdr = content_h + 0.05
        for txt, x, ha in [('Module / Sample', xL, 'left'),
                            ('Bivariate I', (xFL + xFR) / 2, 'center'),
                            ('I', xE, 'center'), ('[95% CI]', xC, 'center'),
                            ('p', xP, 'center'), ('I²', xI, 'center')]:
            ax.text(x, y_hdr, txt, fontsize=4.5, fontweight='bold', va='center', ha=ha)
        ax.plot([0, 0.92], [y_hdr - 0.1, y_hdr - 0.1], color='k', linewidth=0.5)

        ax.plot([v2x(0), v2x(0)], [-0.1, y_hdr - 0.12],
                color='#BBB', linewidth=0.4, linestyle='--', zorder=0)

        yc = content_h - SR * 0.5

        for m_idx, mod in enumerate(HALLMARK_NAMES):
            block_top = yc + SR * 0.45
            block_bot = yc - n_rows_per_mod * SR - 0.02
            if m_idx % 2 == 0:
                ax.fill_between([0, 0.92], block_bot, block_top,
                                color='#F5F5F5', zorder=0)

            ax.text(xL, yc, mod, fontsize=5, fontweight='bold', va='center')
            yc -= SR

            for sname in ct_samps:
                srow = ct_biv[(ct_biv['sample'] == sname) & (ct_biv['module'] == mod)]
                if srow.empty:
                    ax.text(xL + 0.015, yc, sname, fontsize=4, va='center', color='#AAA')
                    ax.text(xE, yc, '—', fontsize=4, va='center', ha='center', color='#AAA')
                    yc -= SR
                    continue

                sr = srow.iloc[0]
                c = scol.get(sname, '#666')
                obs = sr[I_col]
                se = sr[SE_col]
                pv = sr[p_col]
                clo = obs - 1.96 * se
                chi = obs + 1.96 * se
                sp = pv < 0.05

                ax.text(xL + 0.015, yc, sname, fontsize=4, va='center', color=c,
                        fontweight='bold' if sp else 'normal')

                xl, xh, xo = v2x(clo), v2x(chi), v2x(obs)
                ax.plot([xl, xh], [yc, yc], color=c, linewidth=0.6, zorder=2)

                wt = 1 / (se ** 2 + 1e-6)
                mx = 1 / (ct_biv[SE_col].min() ** 2 + 1e-6)
                sz = 2.0 + 3.0 * min(wt / mx, 1.0)
                ax.plot(xo, yc, marker='s', color=c, markersize=sz,
                        markeredgecolor='white', markeredgewidth=0.2, zorder=3)

                ax.text(xE, yc, f"{obs:+.3f}", fontsize=4, va='center', ha='center',
                        fontweight='bold' if sp else 'normal')
                ax.text(xC, yc, f"[{clo:+.2f},{chi:+.2f}]", fontsize=3.5,
                        va='center', ha='center', color='#555')
                ax.text(xP, yc, fmt_p(pv), fontsize=4, va='center', ha='center',
                        fontweight='bold' if sp else 'normal',
                        color='#333' if sp else '#888')
                yc -= SR

            ma_row = ct_meta[ct_meta['module'] == mod]
            if not ma_row.empty:
                ma = ma_row.iloc[0]
                sp = ma['p'] < 0.05

                ax.plot([xL, 0.88], [yc + SR * 0.42, yc + SR * 0.42],
                        color='#DDD', linewidth=0.3)

                ax.text(xL + 0.015, yc, "Pooled", fontsize=4, va='center',
                        fontweight='bold', fontstyle='italic')

                xl, xh, xp = v2x(ma['ci_lo']), v2x(ma['ci_hi']), v2x(ma['pooled_I'])
                dw = 0.004
                dh = SR * 0.32
                ax.fill([xl, xp, xh, xp], [yc, yc - dh, yc, yc + dh],
                        color='#333', zorder=4)
                ax.plot([xl, xp, xh, xp, xl], [yc, yc - dh, yc, yc + dh, yc],
                        color='k', linewidth=0.3, zorder=4)

                ax.text(xE, yc, f"{ma['pooled_I']:+.3f}", fontsize=4,
                        va='center', ha='center', fontweight='bold')
                ax.text(xC, yc, f"[{ma['ci_lo']:+.2f},{ma['ci_hi']:+.2f}]",
                        fontsize=3.5, va='center', ha='center', color='#555')
                ax.text(xP, yc, fmt_p(ma['p']), fontsize=4, va='center', ha='center',
                        fontweight='bold', color='#C44E52' if sp else '#333')
                ax.text(xI, yc, f"{ma['I2']:.0f}%", fontsize=4, va='center', ha='center',
                        color='#C44E52' if ma['I2'] > 75 else '#888')

            yc -= SR
            if m_idx < len(HALLMARK_NAMES) - 1:
                yc -= MOD_GAP

        # X-axis
        ya = -0.08
        ax.plot([xFL, xFR], [ya, ya], color='k', linewidth=0.5)
        for tv in np.linspace(dmin, dmax, 5):
            xt = v2x(tv)
            ax.plot([xt, xt], [ya, ya - 0.04], color='k', linewidth=0.4)
            ax.text(xt, ya - 0.06, f"{tv:+.2f}", fontsize=3.5, ha='center', va='top')

        # Footer legend
        yf = ya - 0.18
        ax.plot([0, 0.92], [yf + 0.06, yf + 0.06], color='#DDD', linewidth=0.3)
        lx = xL
        for sname in ct_samps:
            ax.plot(lx, yf, marker='s', color=scol.get(sname, '#666'), markersize=3,
                    markeredgecolor='white', markeredgewidth=0.2)
            ax.text(lx + 0.012, yf, sname, fontsize=3.5, va='center',
                    color=scol.get(sname, '#666'))
            lx += 0.08
            if lx > 0.85:
                lx = xL
                yf -= 0.12
        ax.plot(lx, yf, marker='D', color='#333', markersize=3, markeredgewidth=0)
        ax.text(lx + 0.012, yf, 'Pooled (DL)', fontsize=3.5, va='center', color='#333')

        fname = f'forest_bivariate_{ct_abbr}_{x_label}'
        for fmt in ['pdf', 'svg']:
            plt.savefig(FIGURES_DIR / f'{fname}.{fmt}',
                        dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        print(f"  ✓ {fname}")

print(f"\n✓ Forest plots complete")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 10: Summary + sen_score vs is_senescent Comparison
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("Cell 10: SUMMARY")
print("=" * 60)

# ── Key findings ────────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("KEY FINDINGS — SIGNIFICANT CO-LOCALIZATION (p < 0.05)")
print(f"{'─'*60}")

for x_label in ['sen_score', 'is_senescent']:
    sub = df_meta[(df_meta['x_variable'] == x_label) & (df_meta['p'] < 0.05)]
    print(f"\n  X = {x_label}: {len(sub)} significant results")

    if len(sub) > 0:
        co = sub[sub['direction'] == 'co-localize']
        seg = sub[sub['direction'] == 'segregate']

        if len(co) > 0:
            print(f"    CO-LOCALIZATION ({len(co)}):")
            for _, r in co.iterrows():
                print(f"      {r['cell_type']:<18s} × {r['module']:<20s}: "
                      f"I={r['pooled_I']:+.4f} p={r['p']:.4f} I²={r['I2']:.0f}%")

        if len(seg) > 0:
            print(f"    SEGREGATION ({len(seg)}):")
            for _, r in seg.iterrows():
                print(f"      {r['cell_type']:<18s} × {r['module']:<20s}: "
                      f"I={r['pooled_I']:+.4f} p={r['p']:.4f} I²={r['I2']:.0f}%")

# ── sen_score vs is_senescent agreement ─────────────────────────────────────
print(f"\n{'─'*60}")
print("SEN_SCORE vs IS_SENESCENT — DIRECTION AGREEMENT")
print(f"{'─'*60}")

meta_sen = df_meta[df_meta['x_variable'] == 'sen_score'].set_index(['cell_type', 'module'])
meta_bin = df_meta[df_meta['x_variable'] == 'is_senescent'].set_index(['cell_type', 'module'])
shared_idx = meta_sen.index.intersection(meta_bin.index)

agree = 0
disagree = 0
both_sig = 0
for idx in shared_idx:
    s = meta_sen.loc[idx]
    b = meta_bin.loc[idx]
    same_dir = (s['pooled_I'] > 0) == (b['pooled_I'] > 0)
    if same_dir:
        agree += 1
    else:
        disagree += 1
    if s['p'] < 0.05 and b['p'] < 0.05:
        both_sig += 1

total = agree + disagree
print(f"  Shared (cell_type × module) pairs: {total}")
print(f"  Same direction:     {agree}/{total} ({agree/max(total,1)*100:.0f}%)")
print(f"  Opposite direction: {disagree}/{total} ({disagree/max(total,1)*100:.0f}%)")
print(f"  Both significant:   {both_sig}/{total}")

# ── High heterogeneity ──────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("HIGH HETEROGENEITY (I² > 75%)")
print(f"{'─'*60}")

high_het = df_meta[(df_meta['I2'] > 75) & (df_meta['p'] < 0.05)]
if len(high_het) > 0:
    for _, r in high_het.iterrows():
        print(f"  {r['x_variable']:<14s} | {r['cell_type']:<18s} × {r['module']:<20s}: "
              f"I²={r['I2']:.0f}%")
else:
    print("  None with I² > 75% among significant results")

# ── Cross-cohort discordance ────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("CROSS-COHORT DIRECTION FLIPS")
print(f"{'─'*60}")

if len(df_strat) > 0:
    for x_label in ['sen_score', 'is_senescent']:
        sub_x = df_strat[df_strat['x_variable'] == x_label]
        flips = []
        for ct in sorted(TESTABLE_CTS.keys()):
            for mod in HALLMARK_NAMES:
                h = sub_x[(sub_x['cell_type'] == ct) & (sub_x['module'] == mod) &
                           (sub_x['cohort'] == 'Harari')]
                m = sub_x[(sub_x['cell_type'] == ct) & (sub_x['module'] == mod) &
                           (sub_x['cohort'] == 'Morabito')]
                if h.empty or m.empty:
                    continue
                if (h.iloc[0]['pooled_I'] > 0) != (m.iloc[0]['pooled_I'] > 0):
                    flips.append(f"    {ct:<18s} × {mod:<20s}: "
                                 f"Harari={h.iloc[0]['pooled_I']:+.3f} "
                                 f"Morabito={m.iloc[0]['pooled_I']:+.3f}")
        if flips:
            print(f"  X = {x_label}: {len(flips)} flips")
            for f in flips:
                print(f)
        else:
            print(f"  X = {x_label}: no direction flips")

# ── Circularity caveat ──────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("CIRCULARITY CAVEAT")
print(f"{'─'*60}")
print("  sen_score × SASP/SenMayo has gene overlap (both senescence gene sets).")
print("  is_senescent × SASP is cleaner (binary threshold, no shared genes).")
print("  DDR, AntiApoptosis, CellSurfaceMarkers, LysosomalContent are most")
print("  independent from the SenePy scoring genes.")

# ── Files saved ─────────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("FILES SAVED")
print(f"{'─'*60}")
for f in sorted(RESULTS_DIR.glob('bivariate_*')):
    print(f"  {f}")
for f in sorted(set(FIGURES_DIR.glob('*bivariate*')) | set(FIGURES_DIR.glob('forest_*'))):
    print(f"  {f}")
for f in sorted(FIGURES_DIR.glob('heatmap_*')):
    print(f"  {f}")

print(f"\n{'='*60}")
print("MODULE 11b COMPLETE")
print(f"{'='*60}")

In [ ]:
#!/usr/bin/env python3
# ═══════════════════════════════════════════════════════════════════════════════
# Module 11b — Senescence × SASP Forest Plots (standalone)
# ═══════════════════════════════════════════════════════════════════════════════
# Pulls only the SASP slice from Module 11b results and re-plots two flavors:
#
#   (A) Per cell type × X-variable:
#       sample-level forest + pooled diamond for SASP only
#       → forest_SASP_{ct}_{x_var}.{pdf,svg}
#
#   (B) Cross-celltype summary per X-variable:
#       pooled SASP estimate per cell type on one figure, sorted by effect
#       → forest_SASP_summary_{x_var}.{pdf,svg}
#
# Inputs (produced by Cells 5–6 of 11b_bivariate_morans_I_meta.ipynb):
#   results/10_spatial/meta_bivariate/bivariate_morans_I_all_controls.csv
#   results/10_spatial/meta_bivariate/bivariate_meta_analysis.csv
#
# Outputs:
#   figures/10_spatial/meta_bivariate/sasp_only/*.{pdf,svg}
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Paths (match notebook) ───────────────────────────────────────────────────
BASE        = Path(SEN_DATA)
RESULTS_DIR = BASE / 'results' / '10_spatial' / 'meta_bivariate'
FIGURES_DIR = BASE / 'figures'  / '10_spatial' / 'meta_bivariate' / 'sasp_only'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODULE      = 'SASP'
MIN_SAMPLES = 3

CT_ABBREV = {
    'Astrocyte': 'Astro', 'Excitatory': 'Exc', 'Inhibitory': 'Inh',
    'Oligodendrocyte': 'Oligo', 'OPC': 'OPC', 'Microglia': 'Micro',
    'Vascular': 'Vasc', 'VSMC': 'VSMC', 'Endothelial': 'Endo',
    'Pericyte': 'Peri', 'PVM': 'PVM', 'VLMC': 'VLMC',
}

print("=" * 60)
print("SENESCENCE × SASP FOREST PLOTS")
print("=" * 60)

# ── Load data ────────────────────────────────────────────────────────────────
df_biv  = pd.read_csv(RESULTS_DIR / 'bivariate_morans_I_all_controls.csv')
df_meta = pd.read_csv(RESULTS_DIR / 'bivariate_meta_analysis.csv')

# Filter to SASP only
df_biv  = df_biv[df_biv['module'] == MODULE].copy()
df_meta = df_meta[df_meta['module'] == MODULE].copy()

print(f"  Per-sample rows (SASP): {len(df_biv)}")
print(f"  Meta rows (SASP)      : {len(df_meta)}")
print(f"  Cell types in meta    : {df_meta['cell_type'].nunique()}")
print(f"  X-variables           : {sorted(df_meta['x_variable'].unique())}")
print()

# ── Sample ordering + colors (Harari first, then Morabito; stable palette) ──
_pal = ['#D62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd',
        '#8c564b', '#e377c2', '#17becf', '#bcbd22', '#7f7f7f',
        '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
        '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5',
        '#393b79', '#637939', '#8c6d31', '#843c39', '#7b4173']

sample_meta = (df_biv[['sample', 'dataset']]
               .drop_duplicates()
               .sort_values(['dataset', 'sample'],
                            key=lambda c: c.map({'Harari': 0, 'Morabito': 1})
                                           if c.name == 'dataset' else c))
sample_order = sample_meta['sample'].tolist()
scol = {s: _pal[i % len(_pal)] for i, s in enumerate(sample_order)}


def fmt_p(pv):
    if pd.isna(pv):      return '—'
    if pv < 0.001:       return f"{pv:.0e}"
    elif pv < 0.01:      return f"{pv:.3f}"
    else:                return f"{pv:.2f}"


# ═══════════════════════════════════════════════════════════════════════════════
# (A) PER CELL TYPE × X-VARIABLE — sample forest + pooled diamond, SASP only
# ═══════════════════════════════════════════════════════════════════════════════
print("─" * 60)
print("(A) PER CELL TYPE FORESTS")
print("─" * 60)

# Column x-positions (normalized)
xL, xFL, xFR = 0.01, 0.22, 0.60
xE,  xC,  xP, xI = 0.66, 0.77, 0.88, 0.95

for x_label, I_col, SE_col, p_col in [
    ('sen_score',    'I_sen', 'SE_sen', 'p_sen'),
    ('is_senescent', 'I_bin', 'SE_bin', 'p_bin'),
]:
    meta_x = df_meta[df_meta['x_variable'] == x_label]
    ct_list = sorted(meta_x['cell_type'].unique())

    for ct in ct_list:
        ct_meta = meta_x[meta_x['cell_type'] == ct]
        ct_biv  = df_biv[df_biv['cell_type'] == ct].dropna(subset=[I_col, SE_col])

        if ct_meta.empty or ct_biv.empty:
            continue

        ct_samps = [s for s in sample_order if s in ct_biv['sample'].values]
        if len(ct_samps) < MIN_SAMPLES:
            continue

        # Geometry — compact since only one module
        SR      = 0.38                              # row height
        HDR     = 0.55                              # header space
        TITLE   = 0.45                              # title space
        FTR     = 0.85                              # footer (axis + legend)
        content_h = (len(ct_samps) + 1) * SR        # samples + pooled row
        total_h = TITLE + HDR + content_h + FTR

        fig_w = 8.2
        fig_h = total_h * 0.30

        # Display range from visible CIs
        vals = []
        for _, r in ct_biv.iterrows():
            vals.extend([r[I_col] - 1.96 * r[SE_col], r[I_col] + 1.96 * r[SE_col]])
        for _, r in ct_meta.iterrows():
            vals.extend([r['ci_lo'], r['ci_hi']])
        hr = max(max(abs(v) for v in vals) * 1.1, 0.05) if vals else 0.2
        dmin, dmax = -hr, hr

        def v2x(v):
            return xFL + (max(min(v, dmax), dmin) - dmin) / (dmax - dmin) * (xFR - xFL)

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.set_xlim(0, 1)
        ax.set_ylim(-FTR, TITLE + HDR + content_h)
        ax.axis('off')

        # Title
        y_title = TITLE + HDR + content_h - 0.1
        ax.text(0.5, y_title,
                f"{ct} — Senescence × SASP  (X = {x_label})",
                fontsize=11, fontweight='bold', ha='center', va='top')

        # Header
        y_hdr = content_h + 0.18
        for txt, x, ha in [('Sample',       xL,              'left'),
                           ('Bivariate I',  (xFL + xFR)/2,   'center'),
                           ('I',            xE,              'center'),
                           ('[95% CI]',     xC,              'center'),
                           ('p',            xP,              'center'),
                           ('I²',           xI,              'center')]:
            ax.text(x, y_hdr, txt, fontsize=8, fontweight='bold',
                    va='center', ha=ha)
        ax.plot([0, 1.0], [y_hdr - 0.15, y_hdr - 0.15], color='k', linewidth=0.6)

        # Zero reference
        ax.plot([v2x(0), v2x(0)], [-0.05, y_hdr - 0.18],
                color='#BBB', linewidth=0.5, linestyle='--', zorder=0)

        yc = content_h - SR * 0.5

        # ── Sample rows ──────────────────────────────────────────────────
        for sname in ct_samps:
            srow = ct_biv[ct_biv['sample'] == sname]
            if srow.empty:
                yc -= SR; continue

            sr  = srow.iloc[0]
            c   = scol.get(sname, '#666')
            obs = sr[I_col]; se = sr[SE_col]; pv = sr[p_col]
            clo, chi = obs - 1.96 * se, obs + 1.96 * se
            sp  = pv < 0.05

            ax.text(xL, yc, sname, fontsize=7, va='center', color=c,
                    fontweight='bold' if sp else 'normal')

            xl, xh, xo = v2x(clo), v2x(chi), v2x(obs)
            ax.plot([xl, xh], [yc, yc], color=c, linewidth=1.0, zorder=2)

            # Weighted marker size (inverse-variance)
            wt = 1 / (se ** 2 + 1e-6)
            mx = 1 / (ct_biv[SE_col].min() ** 2 + 1e-6)
            sz = 4.0 + 5.0 * min(wt / mx, 1.0)
            ax.plot(xo, yc, marker='s', color=c, markersize=sz,
                    markeredgecolor='white', markeredgewidth=0.4, zorder=3)

            ax.text(xE, yc, f"{obs:+.3f}", fontsize=7, va='center', ha='center',
                    fontweight='bold' if sp else 'normal')
            ax.text(xC, yc, f"[{clo:+.2f}, {chi:+.2f}]", fontsize=6.5,
                    va='center', ha='center', color='#555')
            ax.text(xP, yc, fmt_p(pv), fontsize=7, va='center', ha='center',
                    fontweight='bold' if sp else 'normal',
                    color='#333' if sp else '#888')
            yc -= SR

        # ── Pooled row ──────────────────────────────────────────────────
        ma = ct_meta.iloc[0]
        sp = ma['p'] < 0.05
        ax.plot([0, 1.0], [yc + SR * 0.42, yc + SR * 0.42],
                color='#DDD', linewidth=0.4)

        ax.text(xL, yc, "Pooled (DL)", fontsize=7.5, va='center',
                fontweight='bold', fontstyle='italic')

        xl, xh, xp = v2x(ma['ci_lo']), v2x(ma['ci_hi']), v2x(ma['pooled_I'])
        dh = SR * 0.35
        ax.fill([xl, xp, xh, xp], [yc, yc - dh, yc, yc + dh],
                color='#222', zorder=4)
        ax.plot([xl, xp, xh, xp, xl], [yc, yc - dh, yc, yc + dh, yc],
                color='k', linewidth=0.5, zorder=4)

        ax.text(xE, yc, f"{ma['pooled_I']:+.3f}", fontsize=7.5,
                va='center', ha='center', fontweight='bold')
        ax.text(xC, yc, f"[{ma['ci_lo']:+.2f}, {ma['ci_hi']:+.2f}]",
                fontsize=6.5, va='center', ha='center', color='#555')
        ax.text(xP, yc, fmt_p(ma['p']), fontsize=7.5, va='center', ha='center',
                fontweight='bold', color='#C44E52' if sp else '#333')
        ax.text(xI, yc, f"{ma['I2']:.0f}%", fontsize=7.5, va='center', ha='center',
                color='#C44E52' if ma['I2'] > 75 else '#888')

        # X-axis
        ya = -0.22
        ax.plot([xFL, xFR], [ya, ya], color='k', linewidth=0.6)
        for tv in np.linspace(dmin, dmax, 5):
            xt = v2x(tv)
            ax.plot([xt, xt], [ya, ya - 0.07], color='k', linewidth=0.5)
            ax.text(xt, ya - 0.12, f"{tv:+.2f}", fontsize=6.5, ha='center', va='top')
        ax.text((xFL + xFR) / 2, ya - 0.28,
                "Bivariate Moran's I  (← segregate   |   co-localize →)",
                fontsize=7, ha='center', va='top', style='italic', color='#444')

        # Direction annotation
        direction = 'co-localize' if ma['pooled_I'] > 0 else 'segregate'
        dir_color = '#2E7D32' if direction == 'co-localize' else '#C44E52'
        sig_txt = ' (sig.)' if sp else ' (n.s.)'
        ax.text(0.5, ya - 0.48,
                f"Pooled direction: senescence & SASP {direction}{sig_txt}  "
                f"—  k={int(ma['k'])} samples, τ²={ma['tau2']:.4f}",
                fontsize=7.5, ha='center', va='top',
                color=dir_color, fontweight='bold')

        ct_abbr = CT_ABBREV.get(ct, ct)
        fname = f'forest_SASP_{ct_abbr}_{x_label}'
        for fmt in ['pdf', 'svg']:
            plt.savefig(FIGURES_DIR / f'{fname}.{fmt}',
                        dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        print(f"  ✓ {fname}")


# ═══════════════════════════════════════════════════════════════════════════════
# (B) CROSS-CELLTYPE SUMMARY — one figure per X-variable, SASP pooled per CT
# ═══════════════════════════════════════════════════════════════════════════════
print("─" * 60)
print("(B) CROSS-CELLTYPE SUMMARY FORESTS")
print("─" * 60)

for x_label in ['sen_score', 'is_senescent']:
    sub = (df_meta[df_meta['x_variable'] == x_label]
           .sort_values('pooled_I', ascending=False)
           .reset_index(drop=True))

    if sub.empty:
        print(f"  (no meta rows for X = {x_label})")
        continue

    n = len(sub)

    # Geometry
    SR      = 0.55
    HDR     = 0.55
    TITLE   = 0.55
    FTR     = 1.00
    content_h = n * SR
    total_h = TITLE + HDR + content_h + FTR

    fig_w = 9.0
    fig_h = total_h * 0.38

    # Column layout
    xL  = 0.01
    xFL = 0.26
    xFR = 0.62
    xE  = 0.68
    xC  = 0.80
    xP  = 0.92
    xI  = 0.97

    # Display range
    hr = max(sub[['ci_lo', 'ci_hi']].abs().max().max() * 1.1, 0.05)
    dmin, dmax = -hr, hr

    def v2x(v):
        return xFL + (max(min(v, dmax), dmin) - dmin) / (dmax - dmin) * (xFR - xFL)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.set_xlim(0, 1)
    ax.set_ylim(-FTR, TITLE + HDR + content_h)
    ax.axis('off')

    # Title
    y_title = TITLE + HDR + content_h - 0.1
    ax.text(0.5, y_title,
            f"Senescence × SASP spatial co-localization  (X = {x_label})\n"
            f"Pooled bivariate Moran's I across 21 controls (DerSimonian–Laird)",
            fontsize=11, fontweight='bold', ha='center', va='top')

    # Header
    y_hdr = content_h + 0.18
    for txt, x, ha in [('Cell type',   xL,              'left'),
                       ('Pooled I',    (xFL + xFR)/2,   'center'),
                       ('I',           xE,              'center'),
                       ('[95% CI]',    xC,              'center'),
                       ('p',           xP,              'center'),
                       ('I²',          xI,              'center')]:
        ax.text(x, y_hdr, txt, fontsize=9, fontweight='bold',
                va='center', ha=ha)
    ax.plot([0, 1.0], [y_hdr - 0.15, y_hdr - 0.15], color='k', linewidth=0.6)

    # Zero reference
    ax.plot([v2x(0), v2x(0)], [-0.10, y_hdr - 0.18],
            color='#BBB', linewidth=0.5, linestyle='--', zorder=0)

    # Alternating shading + rows
    yc = content_h - SR * 0.5
    for i, row in sub.iterrows():
        ct   = row['cell_type']
        pI   = row['pooled_I']
        clo  = row['ci_lo']; chi = row['ci_hi']
        pv   = row['p']; I2 = row['I2']; k = int(row['k'])
        sp   = pv < 0.05

        # Zebra
        if i % 2 == 0:
            ax.fill_between([0, 1.0], yc - SR * 0.48, yc + SR * 0.48,
                            color='#F7F7F7', zorder=0)

        # Color diamond by direction + significance
        if sp:
            color = '#2E7D32' if pI > 0 else '#C44E52'
        else:
            color = '#555'

        # CT label
        ax.text(xL, yc, ct, fontsize=8.5, va='center',
                fontweight='bold' if sp else 'normal',
                color=color if sp else '#222')
        ax.text(xL + 0.12, yc, f"(k={k})", fontsize=6.5, va='center',
                color='#888', style='italic')

        # Diamond
        xl, xh, xp = v2x(clo), v2x(chi), v2x(pI)
        dh = SR * 0.30
        ax.fill([xl, xp, xh, xp], [yc, yc - dh, yc, yc + dh],
                color=color, zorder=4)
        ax.plot([xl, xp, xh, xp, xl], [yc, yc - dh, yc, yc + dh, yc],
                color='k', linewidth=0.4, zorder=4)

        # Stats
        ax.text(xE, yc, f"{pI:+.3f}", fontsize=8,
                va='center', ha='center',
                fontweight='bold' if sp else 'normal')
        ax.text(xC, yc, f"[{clo:+.2f}, {chi:+.2f}]",
                fontsize=7, va='center', ha='center', color='#555')
        ax.text(xP, yc, fmt_p(pv), fontsize=8, va='center', ha='center',
                fontweight='bold' if sp else 'normal',
                color=color if sp else '#888')
        ax.text(xI, yc, f"{I2:.0f}%", fontsize=8, va='center', ha='center',
                color='#C44E52' if I2 > 75 else '#666')

        yc -= SR

    # X-axis
    ya = -0.30
    ax.plot([xFL, xFR], [ya, ya], color='k', linewidth=0.6)
    for tv in np.linspace(dmin, dmax, 5):
        xt = v2x(tv)
        ax.plot([xt, xt], [ya, ya - 0.08], color='k', linewidth=0.5)
        ax.text(xt, ya - 0.15, f"{tv:+.2f}", fontsize=7, ha='center', va='top')

    # Axis label with direction arrows
    ax.text((xFL + xFR) / 2, ya - 0.34,
            "Bivariate Moran's I  (← segregate   |   co-localize →)",
            fontsize=8, ha='center', va='top', style='italic', color='#444')

    # Legend
    yf = ya - 0.62
    legend_items = [
        ('Sig. co-localization', '#2E7D32'),
        ('Sig. segregation',     '#C44E52'),
        ('n.s.',                 '#555'),
    ]
    lx = 0.25
    for txt, col in legend_items:
        ax.fill([lx, lx + 0.008, lx + 0.016, lx + 0.008],
                [yf, yf - 0.05, yf, yf + 0.05], color=col)
        ax.text(lx + 0.022, yf, txt, fontsize=7, va='center', color='#333')
        lx += 0.18

    fname = f'forest_SASP_summary_{x_label}'
    for fmt in ['pdf', 'svg']:
        plt.savefig(FIGURES_DIR / f'{fname}.{fmt}',
                    dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  ✓ {fname}")

print()
print("=" * 60)
print(f"✓ Done. Figures saved to: {FIGURES_DIR}")
print("=" * 60)

---
## What this module can and cannot establish

**Can.** That the senescence score is spatially autocorrelated in control brain,
that it replicates across two independent cohorts, and — if section 08 holds —
that the autocorrelation is not fully explained by cell-type architecture.

**Cannot.** Anything cell-type-specific at spot resolution. Each spot is a
mixture, and microglia in particular are too sparse in these sections to support
a microglia-specific spatial claim. A statement about microglial senescence foci
does not follow from these data.

**Cannot.** Direction. Bivariate Moran's I in section 18 is an association
between a variable here and a variable next door. Reading it as propagation
requires an assumption the statistic does not test.